<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/07_DP_CTGAN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 07 — DP-CTGAN BASELINE
# Research Framework: SPP-GAN
# Version: 1.0
# Purpose:
#   Differentially Private CTGAN baseline using:
#       • Notebook 02 authoritative training data
#       • CTGAN data transformation and conditional sampling
#       • CTGAN Generator / Discriminator architecture
#       • DP-SGD on discriminator
#       • Per-sample gradient clipping
#       • Gaussian noise
#       • Poisson sampling
#       • RDP privacy accounting
#
# IMPORTANT:
#   This notebook is a BASELINE.
#   No SPP-GAN statistical guidance is used.
#   No SPP-GAN adaptive mechanism is used.
#   No SPP-GAN privacy layer is used beyond the explicit DP-SGD baseline.
#   No validation/test data are used for training.
# ==================================================================================================

NOTEBOOK_ID = "07"
NOTEBOOK_NAME = "DP-CTGAN Baseline"
NOTEBOOK_VERSION = "1.0"

print("=" * 100)
print("Notebook 07 — DP-CTGAN Baseline")
print("=" * 100)
print(f"Notebook ID      : {NOTEBOOK_ID}")
print(f"Notebook version : {NOTEBOOK_VERSION}")
print(f"Notebook name    : {NOTEBOOK_NAME}")
print("=" * 100)

Notebook 07 — DP-CTGAN Baseline
Notebook ID      : 07
Notebook version : 1.0
Notebook name    : DP-CTGAN Baseline


In [2]:
# ==================================================================================================
# 1. HEADER & SCOPE
# ==================================================================================================

from pathlib import Path

# --------------------------------------------------------------------------------------------------
# Notebook identity
# --------------------------------------------------------------------------------------------------

NOTEBOOK_ID = "07"
NOTEBOOK_NAME = "DP-CTGAN Baseline"
NOTEBOOK_VERSION = "1.0"

# --------------------------------------------------------------------------------------------------
# Canonical project root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

print("=" * 100)
print("SECTION 1 — HEADER & SCOPE")
print("=" * 100)

print(f"Notebook ID       : {NOTEBOOK_ID}")
print(f"Notebook Name     : {NOTEBOOK_NAME}")
print(f"Notebook Version  : {NOTEBOOK_VERSION}")
print(f"Project Root      : {PROJECT_ROOT}")

# --------------------------------------------------------------------------------------------------
# Scope
# --------------------------------------------------------------------------------------------------

print("\nScope:")
print("✓ Train-only")
print("✓ Notebook 02 authoritative preprocessing")
print("✓ CTGAN conditional tabular generation")
print("✓ DP-SGD discriminator")
print("✓ Per-sample gradient clipping")
print("✓ Gaussian noise")
print("✓ Poisson sampling")
print("✓ RDP accounting")
print("✓ No statistical guidance")
print("✓ No SPP-GAN components")
print("✓ No validation/test training")

# --------------------------------------------------------------------------------------------------
# Final status
# --------------------------------------------------------------------------------------------------

print("\n✓ SECTION 1 — PASS")

SECTION 1 — HEADER & SCOPE
Notebook ID       : 07
Notebook Name     : DP-CTGAN Baseline
Notebook Version  : 1.0
Project Root      : /content/drive/MyDrive/SPP_GAN_Research

Scope:
✓ Train-only
✓ Notebook 02 authoritative preprocessing
✓ CTGAN conditional tabular generation
✓ DP-SGD discriminator
✓ Per-sample gradient clipping
✓ Gaussian noise
✓ Poisson sampling
✓ RDP accounting
✓ No statistical guidance
✓ No SPP-GAN components
✓ No validation/test training

✓ SECTION 1 — PASS


In [3]:
# ==================================================================================================
# 2. LOAD CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("SECTION 2 — LOAD CONFIGURATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 2.1 — Standard library imports
# --------------------------------------------------------------------------------------------------

from pathlib import Path
import os
import sys
import json
import time
import math
import gc
import random
import hashlib
import subprocess
import inspect
import warnings
from datetime import datetime, timezone

# --------------------------------------------------------------------------------------------------
# 2.2 — Google Drive
# --------------------------------------------------------------------------------------------------

from google.colab import drive

MYDRIVE_ROOT = Path("/content/drive/MyDrive")

if not MYDRIVE_ROOT.exists():
    drive.mount("/content/drive")
else:
    print("✓ Google Drive already mounted")

assert MYDRIVE_ROOT.exists(), (
    "Google Drive MyDrive is not available."
)

# --------------------------------------------------------------------------------------------------
# 2.3 — Canonical project root
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

assert PROJECT_ROOT.exists(), (
    f"Project root not found: {PROJECT_ROOT}"
)

print(f"✓ Project root      : {PROJECT_ROOT}")

# --------------------------------------------------------------------------------------------------
# 2.4 — Required Python packages
#
# IMPORTANT:
#   SDV 1.38.3 was frozen/validated in Notebook 05 and Notebook 06.
#   Opacus 1.6.0 is the required DP dependency for Notebook 07.
#
#   Install only when the required version is missing.
#   This avoids unnecessary package reinstallations in Colab.
# --------------------------------------------------------------------------------------------------

REQUIRED_SDV_VERSION = "1.38.3"
REQUIRED_OPACUS_VERSION = "1.6.0"


def ensure_package(
    package_name,
    required_version,
    import_name=None,
):
    """
    Ensure an exact package version is available.

    Returns:
        imported module
    """

    if import_name is None:
        import_name = package_name

    try:
        module = __import__(import_name)
        installed_version = getattr(
            module,
            "__version__",
            None,
        )

    except ModuleNotFoundError:

        installed_version = None

    # ----------------------------------------------------------------------------------------------
    # Install only if missing or incorrect.
    # ----------------------------------------------------------------------------------------------

    if installed_version != required_version:

        print(
            f"Installing {package_name}=={required_version} ..."
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--disable-pip-version-check",
                f"{package_name}=={required_version}",
            ],
            check=True,
        )

        # ------------------------------------------------------------------------------------------
        # Import after installation.
        # ------------------------------------------------------------------------------------------

        module = __import__(
            import_name
        )

        installed_version = getattr(
            module,
            "__version__",
            None,
        )

    assert installed_version == required_version, (
        f"{package_name} version mismatch. "
        f"Expected {required_version}; "
        f"found {installed_version}."
    )

    return module


# --------------------------------------------------------------------------------------------------
# 2.5 — Ensure SDV and Opacus
# --------------------------------------------------------------------------------------------------

sdv = ensure_package(
    package_name="sdv",
    required_version=REQUIRED_SDV_VERSION,
    import_name="sdv",
)

opacus = ensure_package(
    package_name="opacus",
    required_version=REQUIRED_OPACUS_VERSION,
    import_name="opacus",
)

print(f"✓ SDV version       : {sdv.__version__}")
print(f"✓ Opacus version    : {opacus.__version__}")

# --------------------------------------------------------------------------------------------------
# 2.6 — Scientific / ML imports
# --------------------------------------------------------------------------------------------------

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader,
    TensorDataset,
)

# --------------------------------------------------------------------------------------------------
# 2.7 — Opacus imports
# --------------------------------------------------------------------------------------------------

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

# --------------------------------------------------------------------------------------------------
# 2.8 — CTGAN imports
#
# These are the CTGAN components used by the DP-CTGAN baseline.
# --------------------------------------------------------------------------------------------------

from ctgan.data_transformer import DataTransformer
from ctgan.data_sampler import DataSampler
from ctgan.synthesizers.ctgan import (
    Generator,
    Discriminator,
)

# --------------------------------------------------------------------------------------------------
# 2.9 — Runtime verification
# --------------------------------------------------------------------------------------------------

print(f"✓ PyTorch version   : {torch.__version__}")
print(f"✓ CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(
        f"✓ GPU               : "
        f"{torch.cuda.get_device_name(0)}"
    )

else:

    print("⚠ GPU               : CPU")

# --------------------------------------------------------------------------------------------------
# 2.10 — Canonical Notebook 02 / 06 / 07 paths
# --------------------------------------------------------------------------------------------------

NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB06_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_06"
)

NB07_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_07"
)

NB07_CONFIG_ROOT = (
    NB07_ROOT / "config"
)

NB07_HISTORY_ROOT = (
    NB07_ROOT / "history"
)

NB07_MODEL_ROOT = (
    NB07_ROOT / "models"
)

NB07_CHECKPOINT_ROOT = (
    NB07_ROOT / "checkpoints"
)

NB07_SYNTHETIC_ROOT = (
    NB07_ROOT / "synthetic"
)

NB07_PRIVACY_ROOT = (
    NB07_ROOT / "privacy"
)

NB07_MANIFEST_ROOT = (
    NB07_ROOT / "manifest"
)

NB07_VALIDATION_ROOT = (
    NB07_ROOT / "validation"
)

# --------------------------------------------------------------------------------------------------
# 2.11 — Create Notebook 07 output directories
# --------------------------------------------------------------------------------------------------

NB07_DIRECTORIES = {
    "root": NB07_ROOT,
    "config": NB07_CONFIG_ROOT,
    "history": NB07_HISTORY_ROOT,
    "models": NB07_MODEL_ROOT,
    "checkpoints": NB07_CHECKPOINT_ROOT,
    "synthetic": NB07_SYNTHETIC_ROOT,
    "privacy": NB07_PRIVACY_ROOT,
    "manifest": NB07_MANIFEST_ROOT,
    "validation": NB07_VALIDATION_ROOT,
}

for name, path in NB07_DIRECTORIES.items():

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    assert path.exists()

print("\n✓ Notebook 07 directories verified.")

# --------------------------------------------------------------------------------------------------
# 2.12 — Dataset registry
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

PROVENANCE_COLUMN = (
    "__original_row_id__"
)

# --------------------------------------------------------------------------------------------------
# 2.13 — Dataset registry validation
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}

assert set(DATASET_IDS) == EXPECTED_DATASETS

assert set(
    TARGET_COLUMNS.keys()
) == EXPECTED_DATASETS

assert set(
    IDENTIFIER_COLUMNS.keys()
) == EXPECTED_DATASETS

assert len(DATASET_IDS) == len(
    set(DATASET_IDS)
)

print(
    f"✓ Datasets registered : {len(DATASET_IDS)}"
)

# --------------------------------------------------------------------------------------------------
# 2.14 — Reproducibility / seed policy
#
# Identical to the validated Notebook 00 / Notebook 06 policy.
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

PRIMARY_REPETITION = (
    "rep_01"
)

REPETITION_SEEDS = {
    "rep_01": 3026,
    "rep_02": 3027,
    "rep_03": 3028,
    "rep_04": 3029,
    "rep_05": 3030,
}

DATASET_SEEDS = {
    "adult_income": 3126,
    "bank_marketing": 3226,
    "diabetes_130us": 3326,
}

assert (
    REPETITION_SEEDS[
        PRIMARY_REPETITION
    ] == 3026
)

assert DATASET_SEEDS[
    "adult_income"
] == 3126

assert DATASET_SEEDS[
    "bank_marketing"
] == 3226

assert DATASET_SEEDS[
    "diabetes_130us"
] == 3326

# --------------------------------------------------------------------------------------------------
# 2.15 — DP configuration
# --------------------------------------------------------------------------------------------------

DP_CONFIG = {
    "enabled": True,
    "algorithm": "DP-SGD",
    "privacy_accountant": "rdp",
    "target_epsilon": 5.0,
    "delta_rule": "min(1e-5, 1/N)",
    "max_grad_norm": 1.0,
    "clipping": "flat",
    "poisson_sampling": True,
    "grad_sample_mode": "hooks",
    "secure_mode": False,
}

# --------------------------------------------------------------------------------------------------
# 2.16 — DP-CTGAN model configuration
# --------------------------------------------------------------------------------------------------

DP_CTGAN_CONFIG = {
    "embedding_dim": 128,
    "generator_dim": (256, 256),
    "discriminator_dim": (256, 256),

    "generator_lr": 2e-4,
    "generator_decay": 1e-6,

    "discriminator_lr": 2e-4,
    "discriminator_decay": 1e-6,

    "batch_size": 128,
    "discriminator_steps": 1,

    "log_frequency": True,
    "epochs": 300,

    # ----------------------------------------------------------------------------------------------
    # DP baseline policy:
    # PAC=1 ensures the discriminator operates on individual
    # records for record-level DP accounting.
    # ----------------------------------------------------------------------------------------------

    "pac": 1,

    "enable_gpu": True,

    "train_only": True,
    "validation_training": False,
    "test_training": False,

    "statistical_guidance": False,
    "spp_gan_components": False,
}

# --------------------------------------------------------------------------------------------------
# 2.17 — Configuration validation
# --------------------------------------------------------------------------------------------------

assert DP_CONFIG["enabled"] is True

assert (
    DP_CONFIG["algorithm"]
    == "DP-SGD"
)

assert (
    DP_CONFIG["privacy_accountant"]
    == "rdp"
)

assert (
    DP_CONFIG["target_epsilon"]
    > 0
)

assert (
    DP_CONFIG["max_grad_norm"]
    > 0
)

assert (
    DP_CONFIG["poisson_sampling"]
    is True
)

assert (
    DP_CTGAN_CONFIG["pac"]
    == 1
)

assert (
    DP_CTGAN_CONFIG["epochs"]
    > 0
)

assert (
    DP_CTGAN_CONFIG["batch_size"]
    > 0
)

assert (
    DP_CTGAN_CONFIG["train_only"]
    is True
)

assert (
    DP_CTGAN_CONFIG["validation_training"]
    is False
)

assert (
    DP_CTGAN_CONFIG["test_training"]
    is False
)

assert (
    DP_CTGAN_CONFIG["statistical_guidance"]
    is False
)

assert (
    DP_CTGAN_CONFIG["spp_gan_components"]
    is False
)

# --------------------------------------------------------------------------------------------------
# 2.18 — Frozen Notebook 06 dependency
# --------------------------------------------------------------------------------------------------

NB06_COMPLETION_REPORT = (
    NB06_ROOT
    / "validation"
    / "ctgan_completion_report.json"
)

assert NB06_COMPLETION_REPORT.exists(), (
    "Notebook 06 completion report not found. "
    "DP-CTGAN cannot proceed."
)

with open(
    NB06_COMPLETION_REPORT,
    "r",
    encoding="utf-8",
) as f:

    NB06_REPORT = json.load(f)

assert isinstance(
    NB06_REPORT,
    dict,
)

assert (
    NB06_REPORT.get(
        "final_status"
    )
    == "PASS"
), (
    "Notebook 06 completion report does not "
    "contain final_status='PASS'."
)

print(
    "\n✓ Notebook 06 dependency : PASS"
)

# --------------------------------------------------------------------------------------------------
# 2.19 — Persist Notebook 07 configuration
# --------------------------------------------------------------------------------------------------

CONFIG_RECORD = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,

    "project_root": str(
        PROJECT_ROOT
    ),

    "datasets": DATASET_IDS,

    "targets": TARGET_COLUMNS,

    "identifiers": IDENTIFIER_COLUMNS,

    "provenance_column": PROVENANCE_COLUMN,

    "seeds": {
        "master_seed": MASTER_SEED,
        "primary_repetition": PRIMARY_REPETITION,
        "repetition_seeds": REPETITION_SEEDS,
        "dataset_seeds": DATASET_SEEDS,
    },

    "dependencies": {
        "python": sys.version,
        "pytorch": torch.__version__,
        "sdv": sdv.__version__,
        "opacus": opacus.__version__,
        "cuda_available": bool(
            torch.cuda.is_available()
        ),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        ),
    },

    "dp_config": DP_CONFIG,

    "dp_ctgan_config": DP_CTGAN_CONFIG,

    "research_policy": {
        "train_only": True,
        "validation_training": False,
        "test_training": False,
        "statistical_guidance": False,
        "spp_gan_components": False,
        "notebook_02_preprocessing_reused": True,
        "notebook_06_dependency_required": True,
    },

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

CONFIG_PATH = (
    NB07_CONFIG_ROOT
    / "dp_ctgan_experiment_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        CONFIG_RECORD,
        f,
        indent=2,
        default=str,
    )

assert CONFIG_PATH.exists()
assert CONFIG_PATH.stat().st_size > 0

# --------------------------------------------------------------------------------------------------
# 2.20 — Reload persisted configuration
# --------------------------------------------------------------------------------------------------

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    RELOADED_CONFIG = json.load(f)

assert (
    RELOADED_CONFIG[
        "notebook_id"
    ]
    == NOTEBOOK_ID
)

assert (
    RELOADED_CONFIG[
        "notebook_name"
    ]
    == NOTEBOOK_NAME
)

assert (
    RELOADED_CONFIG[
        "notebook_version"
    ]
    == NOTEBOOK_VERSION
)

assert (
    RELOADED_CONFIG[
        "dependencies"
    ]["sdv"]
    == REQUIRED_SDV_VERSION
)

assert (
    RELOADED_CONFIG[
        "dependencies"
    ]["opacus"]
    == REQUIRED_OPACUS_VERSION
)

print("\n" + "-" * 100)
print("CONFIGURATION SUMMARY")
print("-" * 100)

print(
    f"SDV version             : "
    f"{sdv.__version__}"
)

print(
    f"Opacus version          : "
    f"{opacus.__version__}"
)

print(
    f"PyTorch version         : "
    f"{torch.__version__}"
)

print(
    f"CUDA available          : "
    f"{torch.cuda.is_available()}"
)

print(
    f"DP target epsilon       : "
    f"{DP_CONFIG['target_epsilon']}"
)

print(
    f"DP accountant            : "
    f"{DP_CONFIG['privacy_accountant']}"
)

print(
    f"DP clipping norm         : "
    f"{DP_CONFIG['max_grad_norm']}"
)

print(
    f"DP Poisson sampling     : "
    f"{DP_CONFIG['poisson_sampling']}"
)

print(
    f"DP-CTGAN epochs         : "
    f"{DP_CTGAN_CONFIG['epochs']}"
)

print(
    f"DP-CTGAN batch size     : "
    f"{DP_CTGAN_CONFIG['batch_size']}"
)

print(
    f"DP-CTGAN PAC            : "
    f"{DP_CTGAN_CONFIG['pac']}"
)

print(
    f"Configuration artifact  : "
    f"{CONFIG_PATH}"
)

print("\n✓ SECTION 2 — PASS")

SECTION 2 — LOAD CONFIGURATION
Mounted at /content/drive
✓ Project root      : /content/drive/MyDrive/SPP_GAN_Research
Installing sdv==1.38.3 ...
Installing opacus==1.6.0 ...
✓ SDV version       : 1.38.3
✓ Opacus version    : 1.6.0
✓ PyTorch version   : 2.11.0+cu128
✓ CUDA available    : True
✓ GPU               : Tesla T4

✓ Notebook 07 directories verified.
✓ Datasets registered : 3

✓ Notebook 06 dependency : PASS

----------------------------------------------------------------------------------------------------
CONFIGURATION SUMMARY
----------------------------------------------------------------------------------------------------
SDV version             : 1.38.3
Opacus version          : 1.6.0
PyTorch version         : 2.11.0+cu128
CUDA available          : True
DP target epsilon       : 5.0
DP accountant            : rdp
DP clipping norm         : 1.0
DP Poisson sampling     : True
DP-CTGAN epochs         : 300
DP-CTGAN batch size     : 128
DP-CTGAN PAC            : 1
Configur

In [4]:
# ==================================================================================================
# 3. LOAD TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("3. LOAD TRAINING DATA")
print("=" * 100)

from pathlib import Path
import gc

# --------------------------------------------------------------------------------------------------
# 1. Verify Notebook 02 root and native directory
# --------------------------------------------------------------------------------------------------

assert NB02_ROOT.exists(), (
    f"Notebook 02 root not found: {NB02_ROOT}"
)

NATIVE_ROOT = NB02_ROOT / "native"

assert NATIVE_ROOT.exists(), (
    f"Notebook 02 native directory not found: {NATIVE_ROOT}"
)

print(f"✓ Notebook 02 root : {NB02_ROOT}")
print(f"✓ Native directory : {NATIVE_ROOT}")

# --------------------------------------------------------------------------------------------------
# 2. Initialize training-data registries
# --------------------------------------------------------------------------------------------------

TRAINING_DATA = {}
TRAINING_SCHEMA = {}
TRAINING_SOURCE_PATHS = {}
TRAINING_MISSINGNESS = {}
TRAINING_IDENTIFIER_STATUS = {}

# --------------------------------------------------------------------------------------------------
# 3. Load authoritative Notebook 02 TRAIN split
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_id}")
    print("-" * 100)

    dataset_native_root = NATIVE_ROOT / dataset_id
    train_path = dataset_native_root / "train.csv"

    # ----------------------------------------------------------------------------------------------
    # Authoritative path validation
    # ----------------------------------------------------------------------------------------------

    assert dataset_native_root.exists(), (
        f"Notebook 02 native dataset directory not found for "
        f"{dataset_id}: {dataset_native_root}"
    )

    assert dataset_native_root.is_dir(), (
        f"Expected directory for {dataset_id}: "
        f"{dataset_native_root}"
    )

    assert train_path.exists(), (
        f"Notebook 02 native training split not found for "
        f"{dataset_id}: {train_path}"
    )

    assert train_path.is_file(), (
        f"Notebook 02 native training artifact is not a file: "
        f"{train_path}"
    )

    print(f"✓ Dataset directory : {dataset_native_root}")
    print(f"✓ Training artifact : {train_path}")

    # ----------------------------------------------------------------------------------------------
    # Load TRAINING split only
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        train_path,
        low_memory=False
    )

    print(
        f"✓ Native train shape : "
        f"{df.shape[0]:,} rows × {df.shape[1]:,} columns"
    )

    # ----------------------------------------------------------------------------------------------
    # Dataset configuration
    # ----------------------------------------------------------------------------------------------

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Required provenance validation
    # ----------------------------------------------------------------------------------------------

    assert PROVENANCE_COLUMN in df.columns, (
        f"Provenance column '{PROVENANCE_COLUMN}' missing "
        f"from Notebook 02 training data for {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Target validation
    # ----------------------------------------------------------------------------------------------

    assert target in df.columns, (
        f"Target column '{target}' missing "
        f"from Notebook 02 training data for {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Duplicate-column validation
    # ----------------------------------------------------------------------------------------------

    assert not df.columns.duplicated().any(), (
        f"Duplicate columns detected in Notebook 02 training data "
        f"for {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Provenance validation
    # ----------------------------------------------------------------------------------------------

    assert df[PROVENANCE_COLUMN].notna().all(), (
        f"Null provenance values detected in {dataset_id}"
    )

    assert df[PROVENANCE_COLUMN].is_unique, (
        f"Duplicate provenance values detected in {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Identifier status
    #
    # Notebook 02 may already have removed identifiers from the native modeling split.
    #
    # Therefore:
    #   • Present identifier  → exclude it here.
    #   • Absent identifier   → accept as already excluded upstream.
    #
    # We never reconstruct or re-add identifiers.
    # ----------------------------------------------------------------------------------------------

    present_identifiers = [
        identifier
        for identifier in identifiers
        if identifier in df.columns
    ]

    absent_identifiers = [
        identifier
        for identifier in identifiers
        if identifier not in df.columns
    ]

    TRAINING_IDENTIFIER_STATUS[dataset_id] = {
        "configured_identifiers": list(identifiers),
        "present_in_native_train": list(present_identifiers),
        "already_excluded_upstream": list(absent_identifiers),
    }

    if present_identifiers:
        print(
            f"✓ Identifiers present : "
            f"{present_identifiers} → excluded"
        )

    if absent_identifiers:
        print(
            f"✓ Identifiers already excluded upstream : "
            f"{absent_identifiers}"
        )

    # ----------------------------------------------------------------------------------------------
    # Construct generative schema
    #
    # Exclude:
    #   • provenance
    #   • any identifiers still present
    #
    # Retain:
    #   • modeling features
    #   • target
    # ----------------------------------------------------------------------------------------------

    excluded_columns = {
        PROVENANCE_COLUMN,
        *present_identifiers,
    }

    generative_columns = [
        column
        for column in df.columns
        if column not in excluded_columns
    ]

    assert target in generative_columns, (
        f"Target '{target}' not retained in generative schema "
        f"for {dataset_id}"
    )

    train_df = df[generative_columns].copy()

    # ----------------------------------------------------------------------------------------------
    # Validate generative schema
    # ----------------------------------------------------------------------------------------------

    assert len(train_df.columns) > 0, (
        f"No generative columns retained for {dataset_id}"
    )

    assert not train_df.columns.duplicated().any(), (
        f"Duplicate generative columns detected for {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Missingness assessment
    #
    # Missing values are intentionally preserved.
    # No imputation or row deletion is performed in Notebook 07.
    # ----------------------------------------------------------------------------------------------

    missing_by_column = train_df.isna().sum()

    missing_cells = int(
        missing_by_column.sum()
    )

    missing_rows = int(
        train_df.isna().any(axis=1).sum()
    )

    total_cells = int(
        train_df.shape[0] * train_df.shape[1]
    )

    missing_rate = (
        missing_cells / total_cells
        if total_cells > 0
        else 0.0
    )

    columns_with_missingness = int(
        (missing_by_column > 0).sum()
    )

    # ----------------------------------------------------------------------------------------------
    # Memory reduction
    # ----------------------------------------------------------------------------------------------

    float64_columns = train_df.select_dtypes(
        include=["float64"]
    ).columns.tolist()

    for column in float64_columns:
        train_df[column] = train_df[column].astype(
            "float32"
        )

    # ----------------------------------------------------------------------------------------------
    # Basic integrity
    # ----------------------------------------------------------------------------------------------

    assert len(train_df) > 0, (
        f"Empty training dataset: {dataset_id}"
    )

    assert target in train_df.columns, (
        f"Target '{target}' missing from final training dataframe "
        f"for {dataset_id}"
    )

    # ----------------------------------------------------------------------------------------------
    # Persist registries
    # ----------------------------------------------------------------------------------------------

    TRAINING_DATA[dataset_id] = train_df

    TRAINING_SCHEMA[dataset_id] = list(
        train_df.columns
    )

    TRAINING_SOURCE_PATHS[dataset_id] = str(
        train_path
    )

    TRAINING_MISSINGNESS[dataset_id] = {
        "rows": int(train_df.shape[0]),
        "columns": int(train_df.shape[1]),
        "missing_cells": missing_cells,
        "missing_rows": missing_rows,
        "missing_rate": float(missing_rate),
        "columns_with_missingness": columns_with_missingness,
        "missing_by_column": {
            column: int(value)
            for column, value in missing_by_column.items()
            if value > 0
        },
    }

    # ----------------------------------------------------------------------------------------------
    # Dataset-level report
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Training rows       : {len(train_df):,}"
    )

    print(
        f"✓ Generative columns  : {len(train_df.columns):,}"
    )

    print(
        f"✓ Target              : {target}"
    )

    print(
        f"✓ Identifiers excluded: "
        f"{len(identifiers)}"
    )

    print(
        f"✓ Provenance          : excluded"
    )

    print(
        f"✓ Missing cells       : {missing_cells:,}"
    )

    print(
        f"✓ Rows with missing   : {missing_rows:,}"
    )

    print(
        f"✓ Missingness rate    : {missing_rate:.6%}"
    )

    print(
        f"✓ Columns with missing: "
        f"{columns_with_missingness}"
    )

    print(
        f"✓ float64 → float32   : "
        f"{len(float64_columns)} columns"
    )

    # ----------------------------------------------------------------------------------------------
    # Release native dataframe
    # ----------------------------------------------------------------------------------------------

    del df
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 4. Cross-dataset registry validation
# --------------------------------------------------------------------------------------------------

assert set(TRAINING_DATA.keys()) == set(DATASET_IDS), (
    "TRAINING_DATA does not contain exactly the registered datasets."
)

assert set(TRAINING_SCHEMA.keys()) == set(DATASET_IDS), (
    "TRAINING_SCHEMA does not contain exactly the registered datasets."
)

assert set(TRAINING_SOURCE_PATHS.keys()) == set(DATASET_IDS), (
    "TRAINING_SOURCE_PATHS does not contain exactly the registered datasets."
)

assert set(TRAINING_MISSINGNESS.keys()) == set(DATASET_IDS), (
    "TRAINING_MISSINGNESS does not contain exactly the registered datasets."
)

assert set(TRAINING_IDENTIFIER_STATUS.keys()) == set(DATASET_IDS), (
    "TRAINING_IDENTIFIER_STATUS does not contain exactly the registered datasets."
)

# --------------------------------------------------------------------------------------------------
# 5. Final summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 3 — TRAINING DATA SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[dataset_id]
    missing_info = TRAINING_MISSINGNESS[dataset_id]

    print(
        f"{dataset_id:<18} | "
        f"Rows: {len(train_df):>8,} | "
        f"Columns: {len(train_df.columns):>3} | "
        f"Missing: {missing_info['missing_cells']:>8,} | "
        f"Missing Rate: {missing_info['missing_rate']:.4%} | "
        f"Target: {TARGET_COLUMNS[dataset_id]}"
    )

print("\n✓ Notebook 02 TRAIN splits loaded.")
print("✓ Training-only data used.")
print("✓ Provenance excluded.")
print("✓ Explicit identifiers excluded.")
print("✓ Upstream identifier removal accepted.")
print("✓ Only generative columns retained.")
print("✓ Target variables retained.")
print("✓ Missingness measured and preserved.")
print("✓ No imputation performed.")
print("✓ No rows dropped.")
print("✓ Schema integrity verified.")
print("✓ Memory optimization applied.")
print("✓ SECTION 3 — PASS")

3. LOAD TRAINING DATA
✓ Notebook 02 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native directory : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native

----------------------------------------------------------------------------------------------------
Dataset : adult_income
----------------------------------------------------------------------------------------------------
✓ Dataset directory : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/adult_income
✓ Training artifact : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/adult_income/train.csv
✓ Native train shape : 34,189 rows × 16 columns
✓ Training rows       : 34,189
✓ Generative columns  : 15
✓ Target              : income
✓ Identifiers excluded: 0
✓ Provenance          : excluded
✓ Missing cells       : 4,599
✓ Rows with missing   : 2,575
✓ Missingness rate    : 0.896780%
✓ Columns with missing: 3
✓ float64 → float32   : 0

In [5]:
# ==================================================================================================
# 4. VALIDATE SCHEMA
# ==================================================================================================

print("=" * 100)
print("4. VALIDATE SCHEMA")
print("=" * 100)

SCHEMA_VALIDATION = []

for dataset_id in DATASET_IDS:

    df = TRAINING_DATA[dataset_id]

    target = TARGET_COLUMNS[dataset_id]
    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Structural validation
    # ----------------------------------------------------------------------------------------------

    duplicate_columns = bool(
        df.columns.duplicated().any()
    )

    target_present = target in df.columns

    identifiers_absent = all(
        identifier not in df.columns
        for identifier in identifiers
    )

    provenance_absent = (
        PROVENANCE_COLUMN not in df.columns
    )

    # ----------------------------------------------------------------------------------------------
    # Missingness assessment
    #
    # Missing values are permitted and preserved.
    # They are NOT treated as schema failures.
    # ----------------------------------------------------------------------------------------------

    missing_cells = int(
        df.isna().sum().sum()
    )

    missing_rows = int(
        df.isna().any(axis=1).sum()
    )

    total_cells = int(
        df.shape[0] * df.shape[1]
    )

    missing_rate = (
        missing_cells / total_cells
        if total_cells > 0
        else 0.0
    )

    columns_with_missingness = int(
        df.isna().sum().gt(0).sum()
    )

    # ----------------------------------------------------------------------------------------------
    # Schema status
    #
    # IMPORTANT:
    # Missing values do not cause FAIL because Section 3
    # intentionally preserves the authoritative Notebook 02
    # training data without imputation.
    # ----------------------------------------------------------------------------------------------

    status = (
        "PASS"
        if (
            not duplicate_columns
            and target_present
            and identifiers_absent
            and provenance_absent
            and len(df) > 0
            and len(df.columns) > 0
        )
        else "FAIL"
    )

    SCHEMA_VALIDATION.append({
        "dataset_id": dataset_id,
        "rows": len(df),
        "columns": len(df.columns),
        "target": target,
        "duplicate_columns": duplicate_columns,
        "target_present": target_present,
        "identifiers_absent": identifiers_absent,
        "provenance_absent": provenance_absent,
        "missing_cells": missing_cells,
        "missing_rows": missing_rows,
        "missing_rate": missing_rate,
        "columns_with_missingness": columns_with_missingness,
        "status": status,
    })


# --------------------------------------------------------------------------------------------------
# 1. Create validation dataframe
# --------------------------------------------------------------------------------------------------

SCHEMA_VALIDATION_DF = pd.DataFrame(
    SCHEMA_VALIDATION
)

display(
    SCHEMA_VALIDATION_DF
)

# --------------------------------------------------------------------------------------------------
# 2. Assert structural schema validity
# --------------------------------------------------------------------------------------------------

assert (
    SCHEMA_VALIDATION_DF["status"] == "PASS"
).all(), (
    "One or more datasets failed structural schema validation."
)

# --------------------------------------------------------------------------------------------------
# 3. Final validation report
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SCHEMA VALIDATION SUMMARY")
print("-" * 100)

for dataset_id in DATASET_IDS:

    row = SCHEMA_VALIDATION_DF[
        SCHEMA_VALIDATION_DF["dataset_id"] == dataset_id
    ].iloc[0]

    print(
        f"{dataset_id:<18} | "
        f"Rows: {int(row['rows']):>8,} | "
        f"Columns: {int(row['columns']):>3} | "
        f"Target: {row['target']:<12} | "
        f"Missing: {int(row['missing_cells']):>8,} | "
        f"Missing Rate: {row['missing_rate']:.4%} | "
        f"Status: {row['status']}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Explicit methodology confirmation
# --------------------------------------------------------------------------------------------------

print("\n✓ Duplicate columns validated.")
print("✓ Target variables validated.")
print("✓ Explicit identifiers absent from modeling data.")
print("✓ Provenance column absent from modeling data.")
print("✓ Missingness measured but not treated as schema failure.")
print("✓ Missing values preserved from Notebook 02.")
print("✓ No imputation performed.")
print("✓ No rows dropped.")
print("✓ All datasets passed structural schema validation.")

print("\n✓ SECTION 4 — PASS")

4. VALIDATE SCHEMA


,dataset_id,rows,columns,target,duplicate_columns,target_present,identifiers_absent,provenance_absent,missing_cells,missing_rows,missing_rate,columns_with_missingness,status
0,adult_income,34189,15,income,False,True,True,True,4599,2575,0.008968,3,PASS
1,bank_marketing,31647,17,y,False,True,True,True,0,0,0.000000,0,PASS
2,diabetes_130us,71236,48,readmitted,False,True,True,True,261953,71236,0.076609,9,PASS



----------------------------------------------------------------------------------------------------
SCHEMA VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------
adult_income       | Rows:   34,189 | Columns:  15 | Target: income       | Missing:    4,599 | Missing Rate: 0.8968% | Status: PASS
bank_marketing     | Rows:   31,647 | Columns:  17 | Target: y            | Missing:        0 | Missing Rate: 0.0000% | Status: PASS
diabetes_130us     | Rows:   71,236 | Columns:  48 | Target: readmitted   | Missing:  261,953 | Missing Rate: 7.6609% | Status: PASS

✓ Duplicate columns validated.
✓ Target variables validated.
✓ Explicit identifiers absent from modeling data.
✓ Provenance column absent from modeling data.
✓ Missingness measured but not treated as schema failure.
✓ Missing values preserved from Notebook 02.
✓ No imputation performed.
✓ No rows dropped.
✓ All datasets passed structural schema validation.

✓ SECTION 4

In [6]:
# ==================================================================================================
# 5. CONFIGURE DP-CTGAN
# ==================================================================================================

print("=" * 100)
print("5. CONFIGURE DP-CTGAN")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Architecture policy
# --------------------------------------------------------------------------------------------------

assert DP_CTGAN_CONFIG["pac"] == 1, (
    "DP-CTGAN baseline must use PAC=1 for record-level DP accounting."
)

assert DP_CTGAN_CONFIG["batch_size"] % 2 == 0
assert DP_CTGAN_CONFIG["epochs"] > 0
assert DP_CTGAN_CONFIG["embedding_dim"] > 0

print("CTGAN architecture:")
for key, value in DP_CTGAN_CONFIG.items():
    print(f"{key:<25}: {value}")

print("\nDP adaptation:")
print("✓ PAC = 1")
print("✓ Standard CTGAN gradient penalty disabled")
print("✓ WGAN discriminator objective retained")
print("✓ DP-SGD applied to discriminator")
print("✓ Generator is trained through the privatized discriminator")

print("\n✓ SECTION 5 — PASS")

5. CONFIGURE DP-CTGAN
CTGAN architecture:
embedding_dim            : 128
generator_dim            : (256, 256)
discriminator_dim        : (256, 256)
generator_lr             : 0.0002
generator_decay          : 1e-06
discriminator_lr         : 0.0002
discriminator_decay      : 1e-06
batch_size               : 128
discriminator_steps      : 1
log_frequency            : True
epochs                   : 300
pac                      : 1
enable_gpu               : True
train_only               : True
validation_training      : False
test_training            : False
statistical_guidance     : False
spp_gan_components       : False

DP adaptation:
✓ PAC = 1
✓ Standard CTGAN gradient penalty disabled
✓ WGAN discriminator objective retained
✓ DP-SGD applied to discriminator
✓ Generator is trained through the privatized discriminator

✓ SECTION 5 — PASS


In [7]:
# ==================================================================================================
# 6. CONFIGURE PRIVACY PARAMETERS
# ==================================================================================================

print("=" * 100)
print("6. CONFIGURE PRIVACY PARAMETERS")
print("=" * 100)

PRIVACY_PARAMETERS = {}

for dataset_id in DATASET_IDS:

    n_rows = len(TRAINING_DATA[dataset_id])

    target_delta = min(
        1e-5,
        1.0 / float(n_rows)
    )

    assert 0 < target_delta < 1

    PRIVACY_PARAMETERS[dataset_id] = {
        "target_epsilon": float(DP_CONFIG["target_epsilon"]),
        "target_delta": float(target_delta),
        "max_grad_norm": float(DP_CONFIG["max_grad_norm"]),
        "accountant": DP_CONFIG["privacy_accountant"],
        "clipping": DP_CONFIG["clipping"],
        "poisson_sampling": bool(DP_CONFIG["poisson_sampling"]),
        "delta_rule": DP_CONFIG["delta_rule"],
    }

    print(
        f"{dataset_id:<18} | "
        f"ε target: {target_epsilon if False else PRIVACY_PARAMETERS[dataset_id]['target_epsilon']:.4f} | "
        f"δ: {target_delta:.8f}"
    )

assert all(
    params["target_epsilon"] > 0
    for params in PRIVACY_PARAMETERS.values()
)

print("\n✓ SECTION 6 — PASS")

6. CONFIGURE PRIVACY PARAMETERS
adult_income       | ε target: 5.0000 | δ: 0.00001000
bank_marketing     | ε target: 5.0000 | δ: 0.00001000
diabetes_130us     | ε target: 5.0000 | δ: 0.00001000

✓ SECTION 6 — PASS


In [8]:
# ==================================================================================================
# 7. CONFIGURE SAMPLING
# ==================================================================================================

print("=" * 100)
print("7. CONFIGURE SAMPLING")
print("=" * 100)

SAMPLING_CONFIG = {
    "batch_size": DP_CTGAN_CONFIG["batch_size"],
    "poisson_sampling": True,
    "shuffle": True,
    "drop_last": False,
    "num_workers": 0,
    "pin_memory": bool(torch.cuda.is_available()),
}

assert SAMPLING_CONFIG["poisson_sampling"] is True
assert SAMPLING_CONFIG["num_workers"] == 0

print(json.dumps(SAMPLING_CONFIG, indent=2))

print("\n✓ Poisson sampling required for the selected DP accounting.")
print("✓ num_workers = 0 for Colab stability and lower RAM overhead.")

print("\n✓ SECTION 7 — PASS")

7. CONFIGURE SAMPLING
{
  "batch_size": 128,
  "poisson_sampling": true,
  "shuffle": true,
  "drop_last": false,
  "num_workers": 0,
  "pin_memory": true
}

✓ Poisson sampling required for the selected DP accounting.
✓ num_workers = 0 for Colab stability and lower RAM overhead.

✓ SECTION 7 — PASS


In [9]:
# ==================================================================================================
# 8. CONFIGURE GRADIENT CLIPPING
# ==================================================================================================

print("=" * 100)
print("8. CONFIGURE GRADIENT CLIPPING")
print("=" * 100)

CLIPPING_CONFIG = {
    "method": "flat",
    "max_grad_norm": float(DP_CONFIG["max_grad_norm"]),
    "per_sample": True,
}

assert CLIPPING_CONFIG["max_grad_norm"] > 0
assert CLIPPING_CONFIG["per_sample"] is True

print(json.dumps(CLIPPING_CONFIG, indent=2))

print("\n✓ Per-sample gradient clipping enabled.")
print("✓ Flat clipping selected.")
print("✓ Clipping threshold C =", CLIPPING_CONFIG["max_grad_norm"])

print("\n✓ SECTION 8 — PASS")

8. CONFIGURE GRADIENT CLIPPING
{
  "method": "flat",
  "max_grad_norm": 1.0,
  "per_sample": true
}

✓ Per-sample gradient clipping enabled.
✓ Flat clipping selected.
✓ Clipping threshold C = 1.0

✓ SECTION 8 — PASS


In [13]:
# ==================================================================================================
# 9. CONFIGURE GAUSSIAN NOISE
# ==================================================================================================

print("=" * 100)
print("9. CONFIGURE GAUSSIAN NOISE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 9.1 — Initialize noise configuration
# --------------------------------------------------------------------------------------------------

NOISE_CONFIG = {}

for dataset_id in DATASET_IDS:

    # ----------------------------------------------------------------------------------------------
    # Dataset-specific training population
    # ----------------------------------------------------------------------------------------------

    n_rows = len(TRAINING_DATA[dataset_id])

    assert n_rows > 0, (
        f"Training population must be positive for {dataset_id}."
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy parameters
    #
    # Noise multiplier is intentionally NOT hard-coded.
    # It will be calibrated against:
    #   • target epsilon
    #   • target delta
    #   • sampling rate
    #   • number of training steps / epochs
    #   • RDP accountant
    #
    # The calibrated value will be established before DP-SGD training.
    # ----------------------------------------------------------------------------------------------

    target_epsilon = float(
        PRIVACY_PARAMETERS[dataset_id]["target_epsilon"]
    )

    target_delta = float(
        PRIVACY_PARAMETERS[dataset_id]["target_delta"]
    )

    max_grad_norm = float(
        PRIVACY_PARAMETERS[dataset_id]["max_grad_norm"]
    )

    accountant = str(
        PRIVACY_PARAMETERS[dataset_id]["accountant"]
    )

    batch_size = int(
        SAMPLING_CONFIG["batch_size"]
    )

    # ----------------------------------------------------------------------------------------------
    # Validate privacy configuration
    # ----------------------------------------------------------------------------------------------

    assert target_epsilon > 0, (
        f"Target epsilon must be positive for {dataset_id}."
    )

    assert 0 < target_delta < 1, (
        f"Target delta must lie in (0, 1) for {dataset_id}."
    )

    assert max_grad_norm > 0, (
        f"Maximum gradient norm must be positive for {dataset_id}."
    )

    assert batch_size > 0, (
        f"Batch size must be positive for {dataset_id}."
    )

    assert accountant == "rdp", (
        f"Unexpected accountant for {dataset_id}: {accountant}"
    )

    # ----------------------------------------------------------------------------------------------
    # Sampling-rate diagnostic
    #
    # For DP-SGD with Poisson sampling, the nominal sampling probability is:
    #
    #       q = batch_size / N
    #
    # The actual PrivacyEngine/accountant implementation remains authoritative.
    # ----------------------------------------------------------------------------------------------

    sampling_rate = min(
        1.0,
        float(batch_size) / float(n_rows)
    )

    assert 0 < sampling_rate <= 1.0, (
        f"Invalid sampling rate for {dataset_id}: {sampling_rate}"
    )

    # ----------------------------------------------------------------------------------------------
    # Noise multiplier placeholder
    #
    # IMPORTANT:
    # None is intentional here.
    # It prevents an uncalibrated noise value from being used as an
    # achieved privacy guarantee.
    # ----------------------------------------------------------------------------------------------

    NOISE_CONFIG[dataset_id] = {
        "mechanism": "Gaussian",
        "noise_multiplier": None,
        "noise_multiplier_status": "PENDING_CALIBRATION",
        "target_epsilon": target_epsilon,
        "target_delta": target_delta,
        "max_grad_norm": max_grad_norm,
        "batch_size": batch_size,
        "population_size": n_rows,
        "nominal_sampling_rate": sampling_rate,
        "poisson_sampling": True,
        "accountant": accountant,
        "application": "discriminator_gradients",
        "generator_noise": False,
        "calibration_required": True,
        "achieved_epsilon": None,
        "achieved_delta": target_delta,
    }

    # ----------------------------------------------------------------------------------------------
    # Dataset-level report
    # ----------------------------------------------------------------------------------------------

    print(
        f"{dataset_id:<18} | "
        f"N: {n_rows:>8,} | "
        f"Batch: {batch_size:>3} | "
        f"q: {sampling_rate:.8f} | "
        f"ε target: {target_epsilon:.4f} | "
        f"δ: {target_delta:.8f}"
    )

# --------------------------------------------------------------------------------------------------
# 9.2 — Global validation
# --------------------------------------------------------------------------------------------------

assert set(NOISE_CONFIG.keys()) == set(DATASET_IDS)

for dataset_id, config in NOISE_CONFIG.items():

    assert config["mechanism"] == "Gaussian"

    assert config["noise_multiplier"] is None

    assert (
        config["noise_multiplier_status"]
        == "PENDING_CALIBRATION"
    )

    assert config["poisson_sampling"] is True

    assert config["application"] == "discriminator_gradients"

    assert config["generator_noise"] is False

    assert config["calibration_required"] is True

    assert config["achieved_epsilon"] is None

# --------------------------------------------------------------------------------------------------
# 9.3 — Methodology confirmation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("GAUSSIAN NOISE CONFIGURATION SUMMARY")
print("-" * 100)

print("✓ Gaussian mechanism configured.")
print("✓ Noise is applied to privatized discriminator gradients.")
print("✓ Per-sample clipping threshold is inherited from Section 8.")
print("✓ Poisson sampling is configured for DP accounting.")
print("✓ RDP accountant selected.")
print("✓ Noise multiplier calibration deferred to the DP calibration step.")
print("✓ Achieved epsilon is NOT claimed at this stage.")
print("✓ Generator gradients are not independently noised in this baseline.")
print("✓ Dataset-specific privacy parameters retained.")

print("\n✓ SECTION 9 — PASS")

9. CONFIGURE GAUSSIAN NOISE
adult_income       | N:   34,189 | Batch: 128 | q: 0.00374389 | ε target: 5.0000 | δ: 0.00001000
bank_marketing     | N:   31,647 | Batch: 128 | q: 0.00404462 | ε target: 5.0000 | δ: 0.00001000
diabetes_130us     | N:   71,236 | Batch: 128 | q: 0.00179684 | ε target: 5.0000 | δ: 0.00001000

----------------------------------------------------------------------------------------------------
GAUSSIAN NOISE CONFIGURATION SUMMARY
----------------------------------------------------------------------------------------------------
✓ Gaussian mechanism configured.
✓ Noise is applied to privatized discriminator gradients.
✓ Per-sample clipping threshold is inherited from Section 8.
✓ Poisson sampling is configured for DP accounting.
✓ RDP accountant selected.
✓ Noise multiplier calibration deferred to the DP calibration step.
✓ Achieved epsilon is NOT claimed at this stage.
✓ Generator gradients are not independently noised in this baseline.
✓ Dataset-specific priva

In [10]:
# ==================================================================================================
# 10. SET SEEDS
# ==================================================================================================

print("=" * 100)
print("10. SET SEEDS")
print("=" * 100)

def seed_everything(seed: int):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


assert REPETITION_SEEDS[PRIMARY_REPETITION] == 3026

assert DATASET_SEEDS["adult_income"] == 3126
assert DATASET_SEEDS["bank_marketing"] == 3226
assert DATASET_SEEDS["diabetes_130us"] == 3326

seed_everything(MASTER_SEED)

SEED_POLICY = {
    "master_seed": MASTER_SEED,
    "primary_repetition": PRIMARY_REPETITION,
    "repetition_seeds": REPETITION_SEEDS,
    "dataset_seeds": DATASET_SEEDS,
    "deterministic": True,
}

with open(
    NB07_CONFIG_ROOT / "dp_ctgan_seed_policy.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(SEED_POLICY, f, indent=2)

print(json.dumps(SEED_POLICY, indent=2))

print("\n✓ SECTION 10 — PASS")

10. SET SEEDS
{
  "master_seed": 2025,
  "primary_repetition": "rep_01",
  "repetition_seeds": {
    "rep_01": 3026,
    "rep_02": 3027,
    "rep_03": 3028,
    "rep_04": 3029,
    "rep_05": 3030
  },
  "dataset_seeds": {
    "adult_income": 3126,
    "bank_marketing": 3226,
    "diabetes_130us": 3326
  },
  "deterministic": true
}

✓ SECTION 10 — PASS


In [11]:
# ==================================================================================================
# 11. INITIALIZE DP-CTGAN
# ==================================================================================================

print("=" * 100)
print("11. INITIALIZE DP-CTGAN")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Required imports
# --------------------------------------------------------------------------------------------------

import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from ctgan.data_transformer import DataTransformer
from ctgan.data_sampler import DataSampler
from ctgan.synthesizers.ctgan import Generator, Discriminator

from opacus.validators import ModuleValidator

# --------------------------------------------------------------------------------------------------
# 2. DP-CTGAN model container
# --------------------------------------------------------------------------------------------------

class DPCTGANModel:

    def __init__(
        self,
        embedding_dim=128,
        generator_dim=(256, 256),
        discriminator_dim=(256, 256),
        generator_lr=2e-4,
        generator_decay=1e-6,
        discriminator_lr=2e-4,
        discriminator_decay=1e-6,
        batch_size=128,
        discriminator_steps=1,
        log_frequency=True,
        pac=1,
        device=None,
    ):

        # ------------------------------------------------------------------------------------------
        # Configuration
        # ------------------------------------------------------------------------------------------

        self.embedding_dim = int(embedding_dim)

        self.generator_dim = tuple(
            generator_dim
        )

        self.discriminator_dim = tuple(
            discriminator_dim
        )

        self.generator_lr = float(
            generator_lr
        )

        self.generator_decay = float(
            generator_decay
        )

        self.discriminator_lr = float(
            discriminator_lr
        )

        self.discriminator_decay = float(
            discriminator_decay
        )

        self.batch_size = int(
            batch_size
        )

        self.discriminator_steps = int(
            discriminator_steps
        )

        self.log_frequency = bool(
            log_frequency
        )

        self.pac = int(
            pac
        )

        # ------------------------------------------------------------------------------------------
        # Configuration assertions
        # ------------------------------------------------------------------------------------------

        assert self.embedding_dim > 0
        assert self.batch_size > 0
        assert self.discriminator_steps > 0
        assert self.pac == 1, (
            "DP-CTGAN requires PAC=1 for the selected "
            "record-level DP accounting design."
        )

        # ------------------------------------------------------------------------------------------
        # Device
        # ------------------------------------------------------------------------------------------

        self.device = (
            torch.device(device)
            if device is not None
            else torch.device(
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
        )

        # ------------------------------------------------------------------------------------------
        # Model components
        # ------------------------------------------------------------------------------------------

        self.transformer = None
        self.data_sampler = None

        self.generator = None
        self.discriminator = None

        # ------------------------------------------------------------------------------------------
        # Training and privacy history
        # ------------------------------------------------------------------------------------------

        self.loss_history = []
        self.privacy_history = []

        # ------------------------------------------------------------------------------------------
        # CTGAN metadata
        # ------------------------------------------------------------------------------------------

        self.discrete_columns = []
        self.output_dimensions = None

        self.generator_input_dim = None
        self.discriminator_input_dim = None

    # ==============================================================================================
    # FIT TRANSFORMER AND INITIALIZE NETWORKS
    # ==============================================================================================

    def fit_transformer(
        self,
        train_df,
        discrete_columns,
    ):

        assert isinstance(
            train_df,
            pd.DataFrame
        )

        assert len(train_df) > 0

        self.discrete_columns = list(
            discrete_columns
        )

        # ------------------------------------------------------------------------------------------
        # CTGAN data transformation
        # ------------------------------------------------------------------------------------------

        self.transformer = DataTransformer()

        self.transformer.fit(
            train_df,
            self.discrete_columns,
        )

        transformed = self.transformer.transform(
            train_df
        ).astype(
            "float32"
        )

        assert transformed.ndim == 2
        assert len(transformed) == len(train_df)
        assert not np.isnan(transformed).any(), (
            "CTGAN transformation produced NaN values."
        )

        # ------------------------------------------------------------------------------------------
        # Conditional sampler
        # ------------------------------------------------------------------------------------------

        self.data_sampler = DataSampler(
            transformed,
            self.transformer.output_info_list,
            self.log_frequency,
        )

        self.output_dimensions = int(
            self.transformer.output_dimensions
        )

        # ------------------------------------------------------------------------------------------
        # Generator
        # ------------------------------------------------------------------------------------------

        cond_dim = int(
            self.data_sampler.dim_cond_vec()
        )

        self.generator_input_dim = (
            self.embedding_dim
            + cond_dim
        )

        self.generator = Generator(
            self.generator_input_dim,
            self.generator_dim,
            self.output_dimensions,
        ).to(
            self.device
        )

        # ------------------------------------------------------------------------------------------
        # Discriminator
        #
        # PAC=1 is mandatory for the selected DP accounting design.
        # ------------------------------------------------------------------------------------------

        self.discriminator_input_dim = (
            self.output_dimensions
            + cond_dim
        )

        self.discriminator = Discriminator(
            self.discriminator_input_dim,
            self.discriminator_dim,
            pac=self.pac,
        ).to(
            self.device
        )

        # ------------------------------------------------------------------------------------------
        # Opacus compatibility validation
        # ------------------------------------------------------------------------------------------

        validation_errors = ModuleValidator.validate(
            self.discriminator,
            strict=False,
        )

        assert len(validation_errors) == 0, (
            "DP discriminator failed Opacus module validation:\n"
            + "\n".join(
                str(error)
                for error in validation_errors
            )
        )

        return transformed

    # ==============================================================================================
    # GENERATE CONDITIONAL VECTOR
    # ==============================================================================================

    def generate_conditional_vector(
        self,
        batch_size,
        condition_column=None,
        condition_value=None,
    ):

        assert self.data_sampler is not None

        batch_size = int(
            batch_size
        )

        assert batch_size > 0

        # ------------------------------------------------------------------------------------------
        # Explicit condition
        # ------------------------------------------------------------------------------------------

        if (
            condition_column is not None
            and condition_value is not None
        ):

            condition_info = (
                self.transformer
                .convert_column_name_value_to_id(
                    condition_column,
                    condition_value,
                )
            )

            condvec = (
                self.data_sampler
                .generate_cond_from_condition_column_info(
                    condition_info,
                    batch_size,
                )
            )

            return condvec

        # ------------------------------------------------------------------------------------------
        # Standard CTGAN conditional sampling
        # ------------------------------------------------------------------------------------------

        return (
            self.data_sampler
            .sample_original_condvec(
                batch_size
            )
        )

    # ==============================================================================================
    # GENERATE SYNTHETIC DATA
    # ==============================================================================================

    @torch.no_grad()
    def sample(
        self,
        n,
        condition_column=None,
        condition_value=None,
    ):

        assert self.generator is not None
        assert self.transformer is not None
        assert self.data_sampler is not None

        n = int(n)

        assert n > 0

        self.generator.eval()

        generated_batches = []

        remaining = n

        while remaining > 0:

            current_batch = min(
                self.batch_size,
                remaining,
            )

            # --------------------------------------------------------------------------------------
            # Noise
            # --------------------------------------------------------------------------------------

            z = torch.randn(
                current_batch,
                self.embedding_dim,
                device=self.device,
            )

            # --------------------------------------------------------------------------------------
            # Conditional vector
            # --------------------------------------------------------------------------------------

            condvec = (
                self.generate_conditional_vector(
                    current_batch,
                    condition_column=condition_column,
                    condition_value=condition_value,
                )
            )

            if condvec is not None:

                c = torch.from_numpy(
                    condvec
                ).to(
                    self.device
                )

                z = torch.cat(
                    [z, c],
                    dim=1,
                )

            # --------------------------------------------------------------------------------------
            # Generator forward pass
            # --------------------------------------------------------------------------------------

            fake = self.generator(
                z
            )

            # --------------------------------------------------------------------------------------
            # CTGAN activation
            # --------------------------------------------------------------------------------------

            fake_activated = self.apply_activate(
                fake
            )

            generated_batches.append(
                fake_activated.cpu().numpy()
            )

            remaining -= current_batch

        synthetic_array = np.concatenate(
            generated_batches,
            axis=0,
        )

        assert len(synthetic_array) == n

        return self.transformer.inverse_transform(
            synthetic_array
        )

    # ==============================================================================================
    # CTGAN OUTPUT ACTIVATION
    # ==============================================================================================

    def apply_activate(
        self,
        data,
    ):

        assert self.transformer is not None

        activated = []

        start = 0

        for column_info in (
            self.transformer.output_info_list
        ):

            for span_info in column_info:

                end = (
                    start
                    + span_info.dim
                )

                if span_info.activation_fn == "tanh":

                    activated.append(
                        torch.tanh(
                            data[:, start:end]
                        )
                    )

                elif span_info.activation_fn == "softmax":

                    activated.append(
                        torch.nn.functional.gumbel_softmax(
                            data[:, start:end],
                            tau=0.2,
                            hard=False,
                            dim=1,
                        )
                    )

                else:

                    raise ValueError(
                        "Unexpected CTGAN activation "
                        f"function: {span_info.activation_fn}"
                    )

                start = end

        return torch.cat(
            activated,
            dim=1,
        )


# --------------------------------------------------------------------------------------------------
# 3. Initialize model registries
# --------------------------------------------------------------------------------------------------

DPCTGAN_MODELS = {}
DPCTGAN_TRAINING_OBJECTS = {}

# --------------------------------------------------------------------------------------------------
# 4. Final initialization checks
# --------------------------------------------------------------------------------------------------

assert DP_CTGAN_CONFIG["pac"] == 1
assert DP_CTGAN_CONFIG["embedding_dim"] > 0
assert DP_CTGAN_CONFIG["batch_size"] > 0

print("✓ DPCTGANModel class defined.")
print("✓ PAC=1 enforced.")
print("✓ CTGAN transformer/sampler integration defined.")
print("✓ Generator initialization defined.")
print("✓ Discriminator initialization defined.")
print("✓ Opacus discriminator validation defined.")
print("✓ DP mechanism deferred to training stage.")
print("✓ Model registries initialized.")

print("\n✓ SECTION 11 — PASS")

11. INITIALIZE DP-CTGAN
✓ DPCTGANModel class defined.
✓ PAC=1 enforced.
✓ CTGAN transformer/sampler integration defined.
✓ Generator initialization defined.
✓ Discriminator initialization defined.
✓ Opacus discriminator validation defined.
✓ DP mechanism deferred to training stage.
✓ Model registries initialized.

✓ SECTION 11 — PASS


In [12]:
# ==================================================================================================
# 12. TRAIN WITH DP — CHECKPOINTED / RESUMABLE / CRASH-SAFE / PUBLICATION VERSION
# ==================================================================================================

print("=" * 100)
print("12. TRAIN WITH DP")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# PURPOSE
# --------------------------------------------------------------------------------------------------
#
# This section:
#
#   1. Trains DP-CTGAN for all registered datasets.
#   2. Uses Opacus DP-SGD on the discriminator.
#   3. Uses Poisson sampling.
#   4. Uses per-sample gradient clipping.
#   5. Uses Gaussian noise.
#   6. Uses RDP privacy accounting.
#   7. Uses an explicit expanded RDP alpha grid.
#   8. Calibrates the noise multiplier using the same alpha grid used for final accounting.
#   9. Preserves the validated non-wrapping Opacus strategy.
#  10. Removes Opacus hooks during the generator update.
#  11. Reinstalls Opacus hooks before the next discriminator update.
#  12. Saves resumable checkpoints after completed epochs.
#  13. Uses atomic checkpoint replacement.
#  14. Uses byte-identical copying for the latest checkpoint pointer.
#  15. Verifies checkpoint file integrity using SHA-256.
#  16. Preserves optimizer state.
#  17. Preserves privacy-accountant state.
#  18. Preserves Python / NumPy / PyTorch / CUDA RNG state.
#  19. Saves training and privacy histories.
#  20. Never records an incomplete dataset as PASS.
#
# IMPORTANT:
#   - Section 11 remains unchanged.
#   - A checkpoint is written only after a COMPLETED epoch.
#   - An incomplete epoch is never represented as completed.
#   - Existing valid checkpoints may be resumed when FORCE_RESTART=False.
#   - For the first clean publication run after this update, use FORCE_RESTART=True.
#
# ==================================================================================================


# --------------------------------------------------------------------------------------------------
# 0. Required imports
# --------------------------------------------------------------------------------------------------

import os
import gc
import json
import time
import math
import random
import shutil
import hashlib
import traceback

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.accountants.utils import get_noise_multiplier


# ==================================================================================================
# 1. Runtime / reproducibility configuration
# ==================================================================================================

CHECKPOINT_INTERVAL = 10

# -----------------------------------------------------------------------------------------------
# IMPORTANT:
#
# Set True for the FIRST CLEAN PUBLICATION RUN after this code update.
#
# True:
#     Ignore old checkpoints and train from epoch 1.
#
# False:
#     Resume from the latest validated checkpoint.
# -----------------------------------------------------------------------------------------------

FORCE_RESTART = False

# Keep the latest N periodic checkpoints.
MAX_PERIODIC_CHECKPOINTS = 5

# -----------------------------------------------------------------------------------------------
# Expanded RDP alpha grid
#
# Opacus recommends:
#     [1 + x / 10.0 for x in range(1, 100)] + list(range(12, 64))
#
# We extend the upper range further so that the accountant is less likely
# to terminate at the largest available alpha.
# -----------------------------------------------------------------------------------------------

RDP_ALPHAS = sorted(
    set(
        [1.0 + x / 10.0 for x in range(1, 100)]
        + list(range(12, 129))
        + [160, 192, 256]
    )
)

RDP_EPSILON_TOLERANCE = 0.001

assert len(RDP_ALPHAS) > 100
assert min(RDP_ALPHAS) > 1.0
assert RDP_ALPHAS == sorted(RDP_ALPHAS)


# ==================================================================================================
# 2. Verify checkpoint directories
# ==================================================================================================

NB07_CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints"
)

NB07_HISTORY_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/history"
)

NB07_PRIVACY_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/privacy"
)

NB07_TRAINING_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/training"
)


NB07_CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB07_HISTORY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB07_PRIVACY_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB07_TRAINING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"✓ Checkpoint root : {NB07_CHECKPOINT_ROOT}")
print(f"✓ History root    : {NB07_HISTORY_ROOT}")
print(f"✓ Privacy root    : {NB07_PRIVACY_ROOT}")
print(f"✓ Training root   : {NB07_TRAINING_ROOT}")
print(f"✓ Checkpoint freq : every {CHECKPOINT_INTERVAL} epochs")
print(f"✓ Force restart   : {FORCE_RESTART}")
print(f"✓ RDP alpha count : {len(RDP_ALPHAS)}")
print(f"✓ RDP alpha range : {min(RDP_ALPHAS):.1f} → {max(RDP_ALPHAS):.1f}")
print(f"✓ RDP tolerance   : {RDP_EPSILON_TOLERANCE}")


# ==================================================================================================
# 3. Reset registries
# ==================================================================================================

TRAINING_RESULTS = []

PRIVACY_ENGINES = {}
DP_OPTIMIZERS = {}
DP_MODULES = {}
DP_LOADERS = {}
DP_HOOKS = {}

DPCTGAN_MODELS = {}
DPCTGAN_TRAINING_OBJECTS = {}


# ==================================================================================================
# 4. Local CTGAN conditional loss
# ==================================================================================================

def ctgan_conditional_loss(
    model,
    data,
    condvec,
    mask,
):
    """
    CTGAN conditional cross-entropy loss.

    Kept local to Section 12 so that the frozen Section 11
    model definition remains unchanged.
    """

    if condvec is None or mask is None:
        return torch.tensor(
            0.0,
            device=data.device,
        )

    losses = []

    start_data = 0
    start_cond = 0

    for column_info in model.transformer.output_info_list:

        for span_info in column_info:

            if (
                len(column_info) != 1
                or span_info.activation_fn != "softmax"
            ):
                start_data += span_info.dim

            else:

                end_data = (
                    start_data
                    + span_info.dim
                )

                end_cond = (
                    start_cond
                    + span_info.dim
                )

                logits = data[
                    :,
                    start_data:end_data,
                ]

                targets = torch.argmax(
                    condvec[
                        :,
                        start_cond:end_cond,
                    ],
                    dim=1,
                )

                loss = nn.functional.cross_entropy(
                    logits,
                    targets,
                    reduction="none",
                )

                losses.append(loss)

                start_data = end_data
                start_cond = end_cond

    if not losses:
        return torch.tensor(
            0.0,
            device=data.device,
        )

    conditional_losses = torch.stack(
        losses,
        dim=1,
    )

    mask = mask[:, :conditional_losses.shape[1]]

    return (
        conditional_losses * mask
    ).sum() / data.size(0)


# ==================================================================================================
# 5. Opacus hook helpers
# ==================================================================================================

def remove_dp_hooks(
    dp_hooks,
):
    """
    Remove Opacus hooks before generator optimization.
    """

    if dp_hooks is not None:
        dp_hooks.remove_hooks()


def reinstall_dp_hooks(
    model,
    dp_hooks,
):
    """
    Reinstall Opacus hooks after generator optimization.
    """

    if dp_hooks is not None:

        dp_hooks.add_hooks(
            loss_reduction="mean",
            batch_first=True,
            force_functorch=False,
        )

    assert isinstance(
        model.discriminator,
        nn.Module,
    ), (
        "Discriminator is no longer a valid PyTorch module "
        "after Opacus hook restoration."
    )


# ==================================================================================================
# 6. RNG state helpers
# ==================================================================================================

def capture_rng_state():
    """
    Capture Python / NumPy / PyTorch / CUDA RNG states.
    """

    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }

    if torch.cuda.is_available():

        state["cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    else:

        state["cuda"] = None

    return state


def restore_rng_state(
    state,
):
    """
    Restore previously captured RNG states.
    """

    if state is None:
        return

    if "python" in state:
        random.setstate(
            state["python"]
        )

    if "numpy" in state:
        np.random.set_state(
            state["numpy"]
        )

    if "torch" in state:
        torch.set_rng_state(
            state["torch"]
        )

    if (
        torch.cuda.is_available()
        and state.get("cuda") is not None
    ):
        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


# ==================================================================================================
# 7. RDP privacy helpers
# ==================================================================================================

def get_rdp_epsilon(
    privacy_engine,
    target_delta,
):
    """
    Compute epsilon using the explicitly configured RDP alpha grid.
    """

    accountant = privacy_engine.accountant

    return float(
        accountant.get_epsilon(
            delta=float(target_delta),
            alphas=RDP_ALPHAS,
        )
    )


def get_rdp_privacy_spent(
    privacy_engine,
    target_delta,
):
    """
    Return epsilon and the optimal RDP alpha.
    """

    accountant = privacy_engine.accountant

    epsilon, optimal_alpha = (
        accountant.get_privacy_spent(
            delta=float(target_delta),
            alphas=RDP_ALPHAS,
        )
    )

    return (
        float(epsilon),
        float(optimal_alpha),
    )


def validate_rdp_boundary(
    privacy_engine,
    target_delta,
):
    """
    Ensure the optimum is not sitting on the boundary of the
    configured alpha grid.

    If the optimum is at a boundary, the run is stopped rather
    than reporting an unnecessarily loose privacy estimate.
    """

    epsilon, optimal_alpha = (
        get_rdp_privacy_spent(
            privacy_engine,
            target_delta,
        )
    )

    boundary_hit = (
        np.isclose(
            optimal_alpha,
            min(RDP_ALPHAS),
        )
        or
        np.isclose(
            optimal_alpha,
            max(RDP_ALPHAS),
        )
    )

    if boundary_hit:

        raise RuntimeError(
            "RDP optimal alpha reached the configured boundary. "
            f"epsilon={epsilon:.8f}, "
            f"optimal_alpha={optimal_alpha}, "
            f"alpha_range=({min(RDP_ALPHAS)}, {max(RDP_ALPHAS)}). "
            "Expand RDP_ALPHAS before accepting the final experiment."
        )

    return epsilon, optimal_alpha


# ==================================================================================================
# 8. Checkpoint file hashing
# ==================================================================================================

def calculate_file_sha256(
    path,
    chunk_size=1024 * 1024,
):
    """
    Calculate SHA-256 hash of a file.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint file does not exist: {path}"
        )

    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


# ==================================================================================================
# 9. Checkpoint path helpers
# ==================================================================================================

def get_dataset_checkpoint_dir(
    dataset_id,
):
    path = (
        NB07_CHECKPOINT_ROOT
        / dataset_id
    )

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    return path


def get_periodic_checkpoint_path(
    dataset_id,
    epoch,
):
    return (
        get_dataset_checkpoint_dir(
            dataset_id
        )
        / f"dp_ctgan_epoch_{epoch:04d}.pt"
    )


def get_latest_checkpoint_path(
    dataset_id,
):
    return (
        get_dataset_checkpoint_dir(
            dataset_id
        )
        / "latest_checkpoint.pt"
    )


def get_final_checkpoint_path(
    dataset_id,
):
    return (
        get_dataset_checkpoint_dir(
            dataset_id
        )
        / "dp_ctgan_final.pt"
    )


def get_checkpoint_manifest_path(
    dataset_id,
):
    return (
        get_dataset_checkpoint_dir(
            dataset_id
        )
        / "checkpoint_manifest.json"
    )


# ==================================================================================================
# 10. Atomic file writers
# ==================================================================================================

def atomic_torch_save(
    payload,
    destination,
):
    """
    Atomically write a PyTorch checkpoint.
    """

    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = destination.with_name(
        destination.name + ".tmp"
    )

    if temp_path.exists():
        temp_path.unlink()

    torch.save(
        payload,
        temp_path,
    )

    assert temp_path.exists()
    assert temp_path.stat().st_size > 0

    with open(
        temp_path,
        "rb",
    ) as handle:
        os.fsync(
            handle.fileno()
        )

    os.replace(
        temp_path,
        destination,
    )

    assert destination.exists()
    assert destination.stat().st_size > 0


def atomic_copy_file(
    source,
    destination,
):
    """
    Atomically copy an already serialized checkpoint.

    This is intentionally used for latest_checkpoint.pt so that
    the latest checkpoint is byte-for-byte identical to the
    corresponding periodic checkpoint.
    """

    source = Path(source)
    destination = Path(destination)

    if not source.exists():
        raise FileNotFoundError(
            f"Source checkpoint does not exist: {source}"
        )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = destination.with_name(
        destination.name + ".tmp"
    )

    if temp_path.exists():
        temp_path.unlink()

    shutil.copyfile(
        source,
        temp_path,
    )

    assert temp_path.exists()
    assert temp_path.stat().st_size == source.stat().st_size

    with open(
        temp_path,
        "rb",
    ) as handle:
        os.fsync(
            handle.fileno()
        )

    os.replace(
        temp_path,
        destination,
    )

    assert destination.exists()
    assert destination.stat().st_size == source.stat().st_size


# ==================================================================================================
# 11. Build checkpoint payload
# ==================================================================================================

def build_checkpoint_payload(
    dataset_id,
    epoch,
    model,
    optimizer_g,
    optimizer_d,
    privacy_engine,
    dp_hooks,
    dp_loader,
    seed,
    target_epsilon,
    target_delta,
    noise_multiplier,
    sample_rate,
    max_grad_norm,
    runtime_seconds,
    train_rows,
    train_columns,
    transformed_dimension,
    discrete_columns,
):
    """
    Build a complete resumable checkpoint.
    """

    actual_epsilon, optimal_alpha = (
        get_rdp_privacy_spent(
            privacy_engine,
            target_delta,
        )
    )

    accountant_state = (
        privacy_engine.accountant.state_dict()
    )

    payload = {

        # ------------------------------------------------------------------------------------------
        # Metadata
        # ------------------------------------------------------------------------------------------

        "checkpoint_version": "12.2",

        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "dataset_id": dataset_id,

        "epoch": int(epoch),

        "seed": int(seed),

        "train_rows": int(train_rows),

        "train_columns": int(train_columns),

        "transformed_dimension": int(
            transformed_dimension
        ),

        "discrete_columns": list(
            discrete_columns
        ),

        # ------------------------------------------------------------------------------------------
        # Configuration
        # ------------------------------------------------------------------------------------------

        "dp_ctgan_config": dict(
            DP_CTGAN_CONFIG
        ),

        "dp_config": dict(
            DP_CONFIG
        ),

        "privacy_parameters": dict(
            PRIVACY_PARAMETERS[dataset_id]
        ),

        "rdp_accounting": {
            "accountant": "rdp",
            "alphas": list(
                RDP_ALPHAS
            ),
            "epsilon_tolerance": float(
                RDP_EPSILON_TOLERANCE
            ),
            "optimal_alpha": float(
                optimal_alpha
            ),
        },

        # ------------------------------------------------------------------------------------------
        # Model state
        # ------------------------------------------------------------------------------------------

        "generator_state_dict": {
            key: value.detach().cpu()
            for key, value
            in model.generator.state_dict().items()
        },

        "discriminator_state_dict": {
            key: value.detach().cpu()
            for key, value
            in model.discriminator.state_dict().items()
        },

        # ------------------------------------------------------------------------------------------
        # Optimizer state
        # ------------------------------------------------------------------------------------------

        "generator_optimizer_state_dict": (
            optimizer_g.state_dict()
        ),

        "discriminator_optimizer_state_dict": (
            optimizer_d.state_dict()
        ),

        # ------------------------------------------------------------------------------------------
        # Privacy accountant
        # ------------------------------------------------------------------------------------------

        "privacy_accountant_state": (
            accountant_state
        ),

        "noise_multiplier": float(
            noise_multiplier
        ),

        "sample_rate": float(
            sample_rate
        ),

        "max_grad_norm": float(
            max_grad_norm
        ),

        "target_epsilon": float(
            target_epsilon
        ),

        "target_delta": float(
            target_delta
        ),

        "actual_epsilon": float(
            actual_epsilon
        ),

        "optimal_alpha": float(
            optimal_alpha
        ),

        # ------------------------------------------------------------------------------------------
        # Histories
        # ------------------------------------------------------------------------------------------

        "loss_history": list(
            model.loss_history
        ),

        "privacy_history": list(
            model.privacy_history
        ),

        # ------------------------------------------------------------------------------------------
        # Runtime
        # ------------------------------------------------------------------------------------------

        "runtime_seconds": float(
            runtime_seconds
        ),

        # ------------------------------------------------------------------------------------------
        # RNG
        # ------------------------------------------------------------------------------------------

        "rng_state": capture_rng_state(),
    }

    return payload


# ==================================================================================================
# 12. Checkpoint content validation
# ==================================================================================================

def validate_checkpoint_contents(
    checkpoint,
    dataset_id,
):
    """
    Validate the structural contents of a checkpoint.
    """

    required_keys = {
        "checkpoint_version",
        "dataset_id",
        "epoch",
        "seed",
        "train_rows",
        "train_columns",
        "transformed_dimension",
        "discrete_columns",
        "generator_state_dict",
        "discriminator_state_dict",
        "generator_optimizer_state_dict",
        "discriminator_optimizer_state_dict",
        "privacy_accountant_state",
        "noise_multiplier",
        "sample_rate",
        "max_grad_norm",
        "target_epsilon",
        "target_delta",
        "actual_epsilon",
        "loss_history",
        "privacy_history",
        "rng_state",
        "rdp_accounting",
    }

    missing = (
        required_keys
        - set(checkpoint.keys())
    )

    if missing:
        raise ValueError(
            "Missing checkpoint keys: "
            + str(sorted(missing))
        )

    if checkpoint["dataset_id"] != dataset_id:
        raise ValueError(
            f"Dataset mismatch: "
            f"{checkpoint['dataset_id']} != {dataset_id}"
        )

    epoch = int(
        checkpoint["epoch"]
    )

    if epoch < 1:
        raise ValueError(
            f"Invalid checkpoint epoch: {epoch}"
        )

    rdp_info = checkpoint[
        "rdp_accounting"
    ]

    checkpoint_alphas = [
        float(alpha)
        for alpha
        in rdp_info["alphas"]
    ]

    if checkpoint_alphas != [
        float(alpha)
        for alpha
        in RDP_ALPHAS
    ]:
        raise ValueError(
            "Checkpoint RDP alpha grid does not match "
            "the current configured grid."
        )

    return True


# ==================================================================================================
# 13. Save checkpoint
# ==================================================================================================

def save_training_checkpoint(
    dataset_id,
    epoch,
    model,
    optimizer_g,
    optimizer_d,
    privacy_engine,
    dp_hooks,
    dp_loader,
    seed,
    target_epsilon,
    target_delta,
    noise_multiplier,
    sample_rate,
    max_grad_norm,
    runtime_seconds,
    train_rows,
    train_columns,
    transformed_dimension,
    discrete_columns,
):
    """
    Save periodic checkpoint and byte-identical latest checkpoint.

    The previous implementation independently serialized the same
    payload twice and then compared the two file hashes.

    This implementation serializes ONCE and copies the resulting
    bytes to latest_checkpoint.pt.
    """

    payload = build_checkpoint_payload(
        dataset_id=dataset_id,
        epoch=epoch,
        model=model,
        optimizer_g=optimizer_g,
        optimizer_d=optimizer_d,
        privacy_engine=privacy_engine,
        dp_hooks=dp_hooks,
        dp_loader=dp_loader,
        seed=seed,
        target_epsilon=target_epsilon,
        target_delta=target_delta,
        noise_multiplier=noise_multiplier,
        sample_rate=sample_rate,
        max_grad_norm=max_grad_norm,
        runtime_seconds=runtime_seconds,
        train_rows=train_rows,
        train_columns=train_columns,
        transformed_dimension=transformed_dimension,
        discrete_columns=discrete_columns,
    )

    checkpoint_path = get_periodic_checkpoint_path(
        dataset_id,
        epoch,
    )

    latest_path = get_latest_checkpoint_path(
        dataset_id
    )

    # ----------------------------------------------------------------------------------------------
    # Save periodic checkpoint ONCE
    # ----------------------------------------------------------------------------------------------

    atomic_torch_save(
        payload,
        checkpoint_path,
    )

    # ----------------------------------------------------------------------------------------------
    # Verify periodic checkpoint
    # ----------------------------------------------------------------------------------------------

    periodic_hash = calculate_file_sha256(
        checkpoint_path
    )

    # ----------------------------------------------------------------------------------------------
    # Load immediately after save
    # ----------------------------------------------------------------------------------------------

    reloaded_payload = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_contents(
        reloaded_payload,
        dataset_id,
    )

    assert int(
        reloaded_payload["epoch"]
    ) == int(epoch)

    # ----------------------------------------------------------------------------------------------
    # Copy EXACT periodic bytes to latest checkpoint
    # ----------------------------------------------------------------------------------------------

    atomic_copy_file(
        checkpoint_path,
        latest_path,
    )

    latest_hash = calculate_file_sha256(
        latest_path
    )

    # This equality is now valid because latest is an exact byte copy.
    assert periodic_hash == latest_hash, (
        "Latest checkpoint does not match "
        "the periodic checkpoint byte-for-byte."
    )

    # ----------------------------------------------------------------------------------------------
    # Re-load latest checkpoint
    # ----------------------------------------------------------------------------------------------

    latest_payload = torch.load(
        latest_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_contents(
        latest_payload,
        dataset_id,
    )

    assert int(
        latest_payload["epoch"]
    ) == int(epoch)

    # ----------------------------------------------------------------------------------------------
    # Update checkpoint manifest
    # ----------------------------------------------------------------------------------------------

    manifest_path = get_checkpoint_manifest_path(
        dataset_id
    )

    manifest = {
        "manifest_version": "12.2",
        "dataset_id": dataset_id,
        "latest_epoch": int(epoch),
        "latest_checkpoint": str(
            latest_path
        ),
        "latest_checkpoint_sha256": latest_hash,
        "periodic_checkpoint": str(
            checkpoint_path
        ),
        "periodic_checkpoint_sha256": periodic_hash,
        "checkpoint_version": str(
            payload["checkpoint_version"]
        ),
        "actual_epsilon": float(
            payload["actual_epsilon"]
        ),
        "optimal_alpha": float(
            payload["optimal_alpha"]
        ),
        "rdp_alpha_min": float(
            min(RDP_ALPHAS)
        ),
        "rdp_alpha_max": float(
            max(RDP_ALPHAS)
        ),
        "updated_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    manifest_tmp = manifest_path.with_name(
        manifest_path.name + ".tmp"
    )

    with open(
        manifest_tmp,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2,
            sort_keys=True,
        )

        handle.flush()
        os.fsync(
            handle.fileno()
        )

    os.replace(
        manifest_tmp,
        manifest_path,
    )

    # ----------------------------------------------------------------------------------------------
    # Retain recent periodic checkpoints
    # ----------------------------------------------------------------------------------------------

    periodic_paths = sorted(
        get_dataset_checkpoint_dir(
            dataset_id
        ).glob(
            "dp_ctgan_epoch_*.pt"
        )
    )

    if len(periodic_paths) > MAX_PERIODIC_CHECKPOINTS:

        paths_to_remove = periodic_paths[
            :-MAX_PERIODIC_CHECKPOINTS
        ]

        for old_path in paths_to_remove:

            try:

                old_path.unlink()

            except Exception as cleanup_error:

                print(
                    f"⚠ Could not remove old checkpoint "
                    f"{old_path}: {cleanup_error}"
                )

    print(
        f"✓ Checkpoint saved : epoch {epoch}"
    )

    print(
        f"  Periodic path    : {checkpoint_path}"
    )

    print(
        f"  Latest path      : {latest_path}"
    )

    print(
        f"  SHA256           : {periodic_hash}"
    )

    print(
        f"  ε                : "
        f"{payload['actual_epsilon']:.6f}"
    )

    print(
        f"  α*               : "
        f"{payload['optimal_alpha']:.4f}"
    )

    return checkpoint_path


# ==================================================================================================
# 14. Locate latest valid checkpoint
# ==================================================================================================

def find_latest_valid_checkpoint(
    dataset_id,
):
    """
    Locate the newest structurally valid checkpoint.

    Preference:
        1. latest_checkpoint.pt
        2. newest periodic checkpoint
    """

    checkpoint_dir = get_dataset_checkpoint_dir(
        dataset_id
    )

    candidates = []

    latest_path = get_latest_checkpoint_path(
        dataset_id
    )

    if latest_path.exists():
        candidates.append(
            latest_path
        )

    periodic_paths = sorted(
        checkpoint_dir.glob(
            "dp_ctgan_epoch_*.pt"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    candidates.extend(
        periodic_paths
    )

    seen = set()

    for path in candidates:

        path = path.resolve()

        if path in seen:
            continue

        seen.add(path)

        try:

            if not path.exists():
                continue

            if path.stat().st_size <= 0:
                raise ValueError(
                    "Checkpoint file is empty."
                )

            checkpoint_hash = (
                calculate_file_sha256(
                    path
                )
            )

            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=False,
            )

            validate_checkpoint_contents(
                checkpoint,
                dataset_id,
            )

            # --------------------------------------------------------------------------------------
            # If this is latest, verify its manifest hash when possible.
            # --------------------------------------------------------------------------------------

            manifest_path = (
                get_checkpoint_manifest_path(
                    dataset_id
                )
            )

            if (
                path.name
                == "latest_checkpoint.pt"
                and manifest_path.exists()
            ):

                with open(
                    manifest_path,
                    "r",
                    encoding="utf-8",
                ) as handle:

                    manifest = json.load(
                        handle
                    )

                expected_hash = (
                    manifest.get(
                        "latest_checkpoint_sha256"
                    )
                )

                if (
                    expected_hash is not None
                    and expected_hash != checkpoint_hash
                ):
                    raise ValueError(
                        "Latest checkpoint SHA-256 does not "
                        "match checkpoint manifest."
                    )

            print(
                f"✓ Valid checkpoint found : {path}"
            )

            print(
                f"  Completed epoch        : "
                f"{checkpoint['epoch']}"
            )

            print(
                f"  Stored epsilon         : "
                f"{checkpoint.get('actual_epsilon', np.nan):.6f}"
            )

            print(
                f"  Optimal alpha          : "
                f"{checkpoint.get('optimal_alpha', np.nan):.4f}"
            )

            print(
                f"  SHA256                 : "
                f"{checkpoint_hash}"
            )

            return path, checkpoint

        except Exception as checkpoint_error:

            print(
                f"⚠ Invalid checkpoint ignored: {path}"
            )

            print(
                f"  Reason: {checkpoint_error}"
            )

    return None, None


# ==================================================================================================
# 15. Restore model / optimizer / accountant
# ==================================================================================================

def restore_checkpoint_state(
    checkpoint,
    model,
    optimizer_g,
    optimizer_d,
    privacy_engine,
    target_delta,
):
    """
    Restore model, optimizers, privacy accountant,
    histories and RNG state.
    """

    # ----------------------------------------------------------------------------------------------
    # Model
    # ----------------------------------------------------------------------------------------------

    model.generator.load_state_dict(
        checkpoint[
            "generator_state_dict"
        ]
    )

    model.discriminator.load_state_dict(
        checkpoint[
            "discriminator_state_dict"
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Optimizers
    # ----------------------------------------------------------------------------------------------

    optimizer_g.load_state_dict(
        checkpoint[
            "generator_optimizer_state_dict"
        ]
    )

    optimizer_d.load_state_dict(
        checkpoint[
            "discriminator_optimizer_state_dict"
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy accountant
    # ----------------------------------------------------------------------------------------------

    privacy_engine.accountant.load_state_dict(
        checkpoint[
            "privacy_accountant_state"
        ]
    )

    # ----------------------------------------------------------------------------------------------
    # Histories
    # ----------------------------------------------------------------------------------------------

    model.loss_history = list(
        checkpoint.get(
            "loss_history",
            [],
        )
    )

    model.privacy_history = list(
        checkpoint.get(
            "privacy_history",
            [],
        )
    )

    # ----------------------------------------------------------------------------------------------
    # RNG
    # ----------------------------------------------------------------------------------------------

    restore_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )

    restored_epoch = int(
        checkpoint["epoch"]
    )

    restored_epsilon, restored_alpha = (
        get_rdp_privacy_spent(
            privacy_engine,
            target_delta,
        )
    )

    print(
        "✓ Checkpoint state restored."
    )

    print(
        f"  Resume epoch      : "
        f"{restored_epoch + 1}"
    )

    print(
        f"  Restored epsilon  : "
        f"{restored_epsilon:.6f}"
    )

    print(
        f"  Optimal alpha     : "
        f"{restored_alpha:.4f}"
    )

    return restored_epoch


# ==================================================================================================
# 16. Dataset-level training
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(
        f"Training DP-CTGAN : {dataset_id}"
    )
    print("-" * 100)

    # ==============================================================================================
    # 16.1 Reproducibility
    # ==============================================================================================

    seed = DATASET_SEEDS[
        dataset_id
    ]

    seed_everything(
        seed
    )

    # ==============================================================================================
    # 16.2 Load training data
    # ==============================================================================================

    train_df = TRAINING_DATA[
        dataset_id
    ]

    target = TARGET_COLUMNS[
        dataset_id
    ]

    assert isinstance(
        train_df,
        pd.DataFrame,
    )

    assert len(
        train_df
    ) > 0

    assert target in train_df.columns

    # ==============================================================================================
    # 16.3 Identify discrete columns
    # ==============================================================================================

    discrete_columns = list(
        train_df.select_dtypes(
            include=[
                "object",
                "category",
                "bool",
            ]
        ).columns
    )

    if target not in discrete_columns:

        discrete_columns.append(
            target
        )

    discrete_columns = list(
        dict.fromkeys(
            discrete_columns
        )
    )

    assert target in discrete_columns

    # ==============================================================================================
    # 16.4 Initialize DP-CTGAN
    # ==============================================================================================

    model = DPCTGANModel(
        embedding_dim=DP_CTGAN_CONFIG[
            "embedding_dim"
        ],
        generator_dim=DP_CTGAN_CONFIG[
            "generator_dim"
        ],
        discriminator_dim=DP_CTGAN_CONFIG[
            "discriminator_dim"
        ],
        generator_lr=DP_CTGAN_CONFIG[
            "generator_lr"
        ],
        generator_decay=DP_CTGAN_CONFIG[
            "generator_decay"
        ],
        discriminator_lr=DP_CTGAN_CONFIG[
            "discriminator_lr"
        ],
        discriminator_decay=DP_CTGAN_CONFIG[
            "discriminator_decay"
        ],
        batch_size=DP_CTGAN_CONFIG[
            "batch_size"
        ],
        discriminator_steps=DP_CTGAN_CONFIG[
            "discriminator_steps"
        ],
        log_frequency=DP_CTGAN_CONFIG[
            "log_frequency"
        ],
        pac=DP_CTGAN_CONFIG[
            "pac"
        ],
        device=(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),
    )

    # ==============================================================================================
    # 16.5 Fit transformer
    # ==============================================================================================

    transformed = model.fit_transformer(
        train_df,
        discrete_columns,
    )

    assert isinstance(
        transformed,
        np.ndarray,
    )

    assert len(
        transformed
    ) == len(train_df)

    assert np.isfinite(
        transformed
    ).all()

    print(
        f"Rows                 : "
        f"{len(train_df):,}"
    )

    print(
        f"Generative columns   : "
        f"{len(train_df.columns)}"
    )

    print(
        f"Discrete columns     : "
        f"{len(discrete_columns)}"
    )

    print(
        f"Transformed dimension: "
        f"{transformed.shape[1]}"
    )

    print(
        f"Device               : "
        f"{model.device}"
    )

    # ==============================================================================================
    # 16.6 Validate discriminator before Opacus
    # ==============================================================================================

    validation_errors = (
        ModuleValidator.validate(
            model.discriminator,
            strict=False,
        )
    )

    if validation_errors:

        raise RuntimeError(
            "DP discriminator is not Opacus-compatible:\n"
            + "\n".join(
                str(error)
                for error in validation_errors
            )
        )

    assert isinstance(
        model.discriminator,
        nn.Module,
    )

    print(
        "✓ Discriminator is Opacus-compatible."
    )

    # ==============================================================================================
    # 16.7 Tensor dataset
    # ==============================================================================================

    transformed_tensor = torch.from_numpy(
        transformed.astype(
            np.float32,
            copy=False,
        )
    )

    dp_dataset = TensorDataset(
        transformed_tensor
    )

    # ==============================================================================================
    # 16.8 Base DataLoader
    # ==============================================================================================

    base_loader = DataLoader(
        dp_dataset,
        batch_size=DP_CTGAN_CONFIG[
            "batch_size"
        ],
        shuffle=True,
        drop_last=False,
        num_workers=SAMPLING_CONFIG[
            "num_workers"
        ],
        pin_memory=SAMPLING_CONFIG[
            "pin_memory"
        ],
    )

    assert len(
        base_loader
    ) > 0

    # ==============================================================================================
    # 16.9 Generator optimizer
    # ==============================================================================================

    optimizer_g = torch.optim.Adam(
        model.generator.parameters(),
        lr=model.generator_lr,
        betas=(0.5, 0.9),
        weight_decay=model.generator_decay,
    )

    # ==============================================================================================
    # 16.10 Base discriminator optimizer
    # ==============================================================================================

    optimizer_d_base = torch.optim.Adam(
        model.discriminator.parameters(),
        lr=model.discriminator_lr,
        betas=(0.5, 0.9),
        weight_decay=model.discriminator_decay,
    )

    # ==============================================================================================
    # 16.11 Privacy configuration
    # ==============================================================================================

    privacy_params = PRIVACY_PARAMETERS[
        dataset_id
    ]

    target_epsilon = float(
        privacy_params[
            "target_epsilon"
        ]
    )

    target_delta = float(
        privacy_params[
            "target_delta"
        ]
    )

    max_grad_norm = float(
        privacy_params[
            "max_grad_norm"
        ]
    )

    assert target_epsilon > 0
    assert 0 < target_delta < 1
    assert max_grad_norm > 0

    # ==============================================================================================
    # 16.12 Privacy engine
    # ==============================================================================================

    privacy_engine = PrivacyEngine(
        accountant="rdp",
        secure_mode=DP_CONFIG[
            "secure_mode"
        ],
    )

    # ==============================================================================================
    # 16.13 Calculate sample rate explicitly
    # ==============================================================================================

    sample_rate = (
        1.0 / len(base_loader)
    )

    assert 0 < sample_rate <= 1

    # ==============================================================================================
    # 16.14 Calibrate noise multiplier using the SAME RDP alpha grid
    # ==============================================================================================

    noise_multiplier = get_noise_multiplier(
        target_epsilon=target_epsilon,
        target_delta=target_delta,
        sample_rate=sample_rate,
        epochs=int(
            DP_CTGAN_CONFIG[
                "epochs"
            ]
        ),
        accountant="rdp",
        epsilon_tolerance=RDP_EPSILON_TOLERANCE,
        alphas=RDP_ALPHAS,
    )

    noise_multiplier = float(
        noise_multiplier
    )

    assert np.isfinite(
        noise_multiplier
    )

    assert noise_multiplier > 0

    print(
        f"RDP noise multiplier : "
        f"{noise_multiplier:.6f}"
    )

    print(
        f"RDP sample rate      : "
        f"{sample_rate:.8f}"
    )

    print(
        f"RDP max grad norm    : "
        f"{max_grad_norm:.6f}"
    )

    print(
        f"Target epsilon       : "
        f"{target_epsilon:.6f}"
    )

    print(
        f"Target delta         : "
        f"{target_delta:.8f}"
    )

    print(
        f"RDP alpha count      : "
        f"{len(RDP_ALPHAS)}"
    )

    print(
        f"RDP alpha range      : "
        f"{min(RDP_ALPHAS):.1f} → "
        f"{max(RDP_ALPHAS):.1f}"
    )

    # ==============================================================================================
    # 16.15 Attach Opacus using explicit calibrated noise
    # ==============================================================================================

    dp_hooks, optimizer_d, dp_loader = (
        privacy_engine.make_private(
            module=model.discriminator,
            optimizer=optimizer_d_base,
            data_loader=base_loader,
            noise_multiplier=noise_multiplier,
            max_grad_norm=max_grad_norm,
            poisson_sampling=True,
            clipping=privacy_params[
                "clipping"
            ],
            grad_sample_mode=DP_CONFIG[
                "grad_sample_mode"
            ],
            wrap_model=False,
        )
    )

    assert dp_hooks is not None
    assert optimizer_d is not None
    assert dp_loader is not None

    assert isinstance(
        model.discriminator,
        nn.Module,
    )

    print(
        "✓ Opacus gradient-sampling hooks attached."
    )

    print(
        "✓ Original discriminator remains active."
    )

    # ==============================================================================================
    # 16.16 Validate discriminator after Opacus attachment
    # ==============================================================================================

    validation_errors_after = (
        ModuleValidator.validate(
            model.discriminator,
            strict=False,
        )
    )

    if validation_errors_after:

        raise RuntimeError(
            "Discriminator failed validation after "
            "Opacus attachment:\n"
            + "\n".join(
                str(error)
                for error in validation_errors_after
            )
        )

    print(
        "✓ Discriminator remains Opacus-compatible."
    )

    # ==============================================================================================
    # 16.17 Validate DP optimizer
    # ==============================================================================================

    assert hasattr(
        optimizer_d,
        "noise_multiplier",
    )

    actual_noise_multiplier = float(
        optimizer_d.noise_multiplier
    )

    assert np.isclose(
        actual_noise_multiplier,
        noise_multiplier,
        rtol=1e-6,
        atol=1e-8,
    )

    assert hasattr(
        optimizer_d,
        "max_grad_norm",
    )

    actual_max_grad_norm = float(
        optimizer_d.max_grad_norm
    )

    assert np.isclose(
        actual_max_grad_norm,
        max_grad_norm,
    )

    # ==============================================================================================
    # 16.18 Validate DP loader
    # ==============================================================================================

    assert hasattr(
        dp_loader,
        "sample_rate",
    )

    actual_sample_rate = float(
        dp_loader.sample_rate
    )

    assert np.isclose(
        actual_sample_rate,
        sample_rate,
        rtol=1e-6,
        atol=1e-8,
    )

    print(
        "✓ DP optimizer verified."
    )

    print(
        "✓ Poisson DP loader verified."
    )

    # ==============================================================================================
    # 16.19 Initial privacy calculation
    # ==============================================================================================

    initial_epsilon = get_rdp_epsilon(
        privacy_engine,
        target_delta,
    )

    print(
        f"✓ Initial epsilon     : "
        f"{initial_epsilon:.8f}"
    )

    # ==============================================================================================
    # 16.20 Training state
    # ==============================================================================================

    model.discriminator.train()
    model.generator.train()

    start_time = time.time()

    completed_epoch = 0

    # ==============================================================================================
    # 16.21 Checkpoint discovery / resume
    # ==============================================================================================

    checkpoint_path = None
    checkpoint = None

    if not FORCE_RESTART:

        checkpoint_path, checkpoint = (
            find_latest_valid_checkpoint(
                dataset_id
            )
        )

    if checkpoint is not None:

        # ------------------------------------------------------------------------------------------
        # Validate checkpoint against current experiment
        # ------------------------------------------------------------------------------------------

        checkpoint_rows = int(
            checkpoint.get(
                "train_rows",
                -1,
            )
        )

        checkpoint_columns = int(
            checkpoint.get(
                "train_columns",
                -1,
            )
        )

        checkpoint_dimension = int(
            checkpoint.get(
                "transformed_dimension",
                -1,
            )
        )

        assert checkpoint_rows == len(
            train_df
        ), (
            f"Checkpoint row mismatch for "
            f"{dataset_id}: "
            f"{checkpoint_rows} != "
            f"{len(train_df)}"
        )

        assert checkpoint_columns == len(
            train_df.columns
        ), (
            f"Checkpoint column mismatch for "
            f"{dataset_id}: "
            f"{checkpoint_columns} != "
            f"{len(train_df.columns)}"
        )

        assert checkpoint_dimension == (
            transformed.shape[1]
        ), (
            f"Checkpoint transformed-dimension "
            f"mismatch for {dataset_id}: "
            f"{checkpoint_dimension} != "
            f"{transformed.shape[1]}"
        )

        assert np.isclose(
            float(
                checkpoint[
                    "target_epsilon"
                ]
            ),
            target_epsilon,
        )

        assert np.isclose(
            float(
                checkpoint[
                    "target_delta"
                ]
            ),
            target_delta,
        )

        assert np.isclose(
            float(
                checkpoint[
                    "noise_multiplier"
                ]
            ),
            actual_noise_multiplier,
            rtol=1e-5,
            atol=1e-7,
        ), (
            "Noise multiplier mismatch. "
            "The checkpoint cannot safely be resumed."
        )

        assert np.isclose(
            float(
                checkpoint[
                    "sample_rate"
                ]
            ),
            actual_sample_rate,
            rtol=1e-5,
            atol=1e-7,
        ), (
            "Sample-rate mismatch. "
            "The checkpoint cannot safely be resumed."
        )

        completed_epoch = (
            restore_checkpoint_state(
                checkpoint=checkpoint,
                model=model,
                optimizer_g=optimizer_g,
                optimizer_d=optimizer_d,
                privacy_engine=privacy_engine,
                target_delta=target_delta,
            )
        )

        print(
            "\n" + "=" * 100
        )

        print(
            f"RESUMING DP-CTGAN : "
            f"{dataset_id}"
        )

        print(
            "=" * 100
        )

        print(
            f"Checkpoint         : "
            f"{checkpoint_path}"
        )

        print(
            f"Completed epoch    : "
            f"{completed_epoch}"
        )

        print(
            f"Next epoch         : "
            f"{completed_epoch + 1}"
        )

        print(
            f"Target epochs      : "
            f"{DP_CTGAN_CONFIG['epochs']}"
        )

        resumed_epsilon = get_rdp_epsilon(
            privacy_engine,
            target_delta,
        )

        print(
            f"Restored epsilon   : "
            f"{resumed_epsilon:.6f}"
        )

    else:

        print(
            f"✓ No valid checkpoint found "
            f"for {dataset_id}."
        )

        print(
            "✓ Starting training from epoch 1."
        )

    # ==============================================================================================
    # 16.22 Training loop
    # ==============================================================================================

    total_epochs = int(
        DP_CTGAN_CONFIG[
            "epochs"
        ]
    )

    if completed_epoch >= total_epochs:

        print(
            f"✓ Dataset {dataset_id} already has "
            f"{completed_epoch} completed epochs."
        )

        print(
            "✓ Skipping training and proceeding "
            "to finalization."
        )

    else:

        try:

            for epoch in range(
                completed_epoch + 1,
                total_epochs + 1,
            ):

                epoch_g_losses = []
                epoch_d_losses = []

                epoch_start_time = time.time()

                # ==================================================================================
                # 16.22.1 Poisson DP batches
                # ==================================================================================

                for batch in dp_loader:

                    real = batch[0].to(
                        model.device,
                        non_blocking=True,
                    )

                    current_batch = real.size(
                        0
                    )

                    if current_batch == 0:
                        continue

                    # ==============================================================================
                    # 16.22.2 DISCRIMINATOR UPDATE — PRIVATE
                    # ==============================================================================

                    for _ in range(
                        DP_CTGAN_CONFIG[
                            "discriminator_steps"
                        ]
                    ):

                        optimizer_d.zero_grad(
                            set_to_none=True
                        )

                        z = torch.randn(
                            current_batch,
                            model.embedding_dim,
                            device=model.device,
                        )

                        condvec = (
                            model.data_sampler.sample_condvec(
                                current_batch
                            )
                        )

                        if condvec is None:

                            c1 = None
                            m1 = None

                            fakez = z
                            real_cat = real

                        else:

                            c1_np, m1_np, _, _ = (
                                condvec
                            )

                            c1 = torch.from_numpy(
                                c1_np
                            ).to(
                                model.device,
                                dtype=torch.float32,
                            )

                            m1 = torch.from_numpy(
                                m1_np
                            ).to(
                                model.device,
                                dtype=torch.float32,
                            )

                            fakez = torch.cat(
                                [
                                    z,
                                    c1,
                                ],
                                dim=1,
                            )

                            perm = (
                                np.random.permutation(
                                    current_batch
                                )
                            )

                            perm_tensor = (
                                torch.from_numpy(
                                    perm
                                ).to(
                                    model.device
                                )
                            )

                            c2 = c1[
                                perm_tensor
                            ]

                            real_cat = torch.cat(
                                [
                                    real,
                                    c2,
                                ],
                                dim=1,
                            )

                        fake = model.generator(
                            fakez
                        )

                        fakeact = (
                            model.apply_activate(
                                fake
                            )
                        )

                        if c1 is None:

                            fake_cat = fakeact

                        else:

                            fake_cat = torch.cat(
                                [
                                    fakeact,
                                    c1,
                                ],
                                dim=1,
                            )

                        fake_cat_for_d = (
                            fake_cat.detach()
                        )

                        y_real = (
                            model.discriminator(
                                real_cat
                            )
                        )

                        y_fake = (
                            model.discriminator(
                                fake_cat_for_d
                            )
                        )

                        # WGAN-style scalar critic loss.
                        loss_d = (
                            y_fake.mean()
                            - y_real.mean()
                        )

                        loss_d.backward()

                        optimizer_d.step()

                        epoch_d_losses.append(
                            float(
                                loss_d.detach().cpu()
                            )
                        )

                    # ==============================================================================
                    # 16.22.3 GENERATOR UPDATE — NON-DP PATH
                    # ==============================================================================

                    optimizer_g.zero_grad(
                        set_to_none=True
                    )

                    remove_dp_hooks(
                        dp_hooks
                    )

                    discriminator_requires_grad = [
                        parameter.requires_grad
                        for parameter
                        in model.discriminator.parameters()
                    ]

                    for parameter in (
                        model.discriminator.parameters()
                    ):

                        parameter.requires_grad_(
                            False
                        )

                    try:

                        z = torch.randn(
                            current_batch,
                            model.embedding_dim,
                            device=model.device,
                        )

                        condvec = (
                            model.data_sampler.sample_condvec(
                                current_batch
                            )
                        )

                        if condvec is None:

                            c1 = None
                            m1 = None
                            fakez = z

                        else:

                            c1_np, m1_np, _, _ = (
                                condvec
                            )

                            c1 = torch.from_numpy(
                                c1_np
                            ).to(
                                model.device,
                                dtype=torch.float32,
                            )

                            m1 = torch.from_numpy(
                                m1_np
                            ).to(
                                model.device,
                                dtype=torch.float32,
                            )

                            fakez = torch.cat(
                                [
                                    z,
                                    c1,
                                ],
                                dim=1,
                            )

                        fake = model.generator(
                            fakez
                        )

                        fakeact = (
                            model.apply_activate(
                                fake
                            )
                        )

                        if c1 is None:

                            y_fake = (
                                model.discriminator(
                                    fakeact
                                )
                            )

                        else:

                            fake_cat_for_g = (
                                torch.cat(
                                    [
                                        fakeact,
                                        c1,
                                    ],
                                    dim=1,
                                )
                            )

                            y_fake = (
                                model.discriminator(
                                    fake_cat_for_g
                                )
                            )

                        cross_entropy = (
                            ctgan_conditional_loss(
                                model=model,
                                data=fake,
                                condvec=c1,
                                mask=m1,
                            )
                        )

                        loss_g = (
                            -y_fake.mean()
                            + cross_entropy
                        )

                        loss_g.backward()

                        optimizer_g.step()

                    finally:

                        for (
                            parameter,
                            previous_state
                        ) in zip(
                            model.discriminator.parameters(),
                            discriminator_requires_grad,
                        ):

                            parameter.requires_grad_(
                                previous_state
                            )

                        reinstall_dp_hooks(
                            model=model,
                            dp_hooks=dp_hooks,
                        )

                        assert getattr(
                            dp_hooks,
                            "hooks_enabled",
                            False,
                        ), (
                            "Opacus hooks were not "
                            "re-enabled."
                        )

                    epoch_g_losses.append(
                        float(
                            loss_g.detach().cpu()
                        )
                    )

                # ==================================================================================
                # 16.22.4 Privacy accounting
                # ==================================================================================

                current_epsilon, optimal_alpha = (
                    get_rdp_privacy_spent(
                        privacy_engine,
                        target_delta,
                    )
                )

                # Do not silently accept a boundary optimum.
                if (
                    np.isclose(
                        optimal_alpha,
                        min(RDP_ALPHAS),
                    )
                    or
                    np.isclose(
                        optimal_alpha,
                        max(RDP_ALPHAS),
                    )
                ):

                    raise RuntimeError(
                        "RDP optimal alpha reached the "
                        "configured boundary at epoch "
                        f"{epoch}: "
                        f"alpha={optimal_alpha}. "
                        "Expand RDP_ALPHAS before continuing."
                    )

                # ==================================================================================
                # 16.22.5 Epoch losses
                # ==================================================================================

                mean_g = (
                    float(
                        np.mean(
                            epoch_g_losses
                        )
                    )
                    if epoch_g_losses
                    else np.nan
                )

                mean_d = (
                    float(
                        np.mean(
                            epoch_d_losses
                        )
                    )
                    if epoch_d_losses
                    else np.nan
                )

                epoch_runtime = (
                    time.time()
                    - epoch_start_time
                )

                # ==================================================================================
                # 16.22.6 Store histories
                # ==================================================================================

                model.loss_history.append(
                    {
                        "epoch": int(epoch),
                        "generator_loss": mean_g,
                        "discriminator_loss": mean_d,
                        "epoch_runtime_seconds": float(
                            epoch_runtime
                        ),
                    }
                )

                model.privacy_history.append(
                    {
                        "epoch": int(epoch),
                        "epsilon": float(
                            current_epsilon
                        ),
                        "delta": float(
                            target_delta
                        ),
                        "noise_multiplier": float(
                            actual_noise_multiplier
                        ),
                        "sample_rate": float(
                            actual_sample_rate
                        ),
                        "max_grad_norm": float(
                            actual_max_grad_norm
                        ),
                        "optimal_alpha": float(
                            optimal_alpha
                        ),
                    }
                )

                completed_epoch = epoch

                # ==================================================================================
                # 16.22.7 Progress
                # ==================================================================================

                print(
                    f"Epoch {epoch:>3}/{total_epochs} | "
                    f"G={mean_g:.5f} | "
                    f"D={mean_d:.5f} | "
                    f"ε={current_epsilon:.5f} | "
                    f"α={optimal_alpha:.2f} | "
                    f"time={epoch_runtime:.2f}s"
                )

                # ==================================================================================
                # 16.22.8 Save histories every epoch
                # ==================================================================================

                history_dir = (
                    NB07_HISTORY_ROOT
                    / dataset_id
                )

                privacy_dir = (
                    NB07_PRIVACY_ROOT
                    / dataset_id
                )

                history_dir.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                privacy_dir.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                history_df = pd.DataFrame(
                    model.loss_history
                )

                privacy_history_df = pd.DataFrame(
                    model.privacy_history
                )

                history_df.to_csv(
                    history_dir
                    / "dp_ctgan_training_history.csv",
                    index=False,
                )

                privacy_history_df.to_csv(
                    privacy_dir
                    / "dp_ctgan_privacy_history.csv",
                    index=False,
                )

                # ==================================================================================
                # 16.22.9 Checkpoint condition
                # ==================================================================================

                should_checkpoint = (
                    epoch % CHECKPOINT_INTERVAL == 0
                    or epoch == total_epochs
                )

                if should_checkpoint:

                    elapsed_runtime = (
                        time.time()
                        - start_time
                    )

                    save_training_checkpoint(
                        dataset_id=dataset_id,
                        epoch=epoch,
                        model=model,
                        optimizer_g=optimizer_g,
                        optimizer_d=optimizer_d,
                        privacy_engine=privacy_engine,
                        dp_hooks=dp_hooks,
                        dp_loader=dp_loader,
                        seed=seed,
                        target_epsilon=target_epsilon,
                        target_delta=target_delta,
                        noise_multiplier=actual_noise_multiplier,
                        sample_rate=actual_sample_rate,
                        max_grad_norm=actual_max_grad_norm,
                        runtime_seconds=elapsed_runtime,
                        train_rows=len(train_df),
                        train_columns=len(train_df.columns),
                        transformed_dimension=transformed.shape[1],
                        discrete_columns=discrete_columns,
                    )

        except KeyboardInterrupt:

            print(
                "\n" + "=" * 100
            )

            print(
                f"TRAINING INTERRUPTED : "
                f"{dataset_id}"
            )

            print(
                "=" * 100
            )

            print(
                f"Last completed epoch : "
                f"{completed_epoch}"
            )

            print(
                "Only the last completed checkpoint "
                "is considered resumable."
            )

            print(
                "This dataset will NOT be marked PASS."
            )

            raise

        except Exception as training_error:

            print(
                "\n" + "=" * 100
            )

            print(
                f"TRAINING FAILED : "
                f"{dataset_id}"
            )

            print(
                "=" * 100
            )

            print(
                f"Last completed epoch : "
                f"{completed_epoch}"
            )

            print(
                f"Error                : "
                f"{training_error}"
            )

            print(
                "\nFull traceback:"
            )

            traceback.print_exc()

            print(
                "\nThe last validated checkpoint "
                "remains available."
            )

            print(
                "This dataset will NOT be marked PASS."
            )

            raise

    # ==============================================================================================
    # 16.23 Final runtime
    # ==============================================================================================

    runtime_seconds = (
        time.time()
        - start_time
    )

    # ==============================================================================================
    # 16.24 Final privacy accounting
    # ==============================================================================================

    actual_epsilon, optimal_alpha = (
        get_rdp_privacy_spent(
            privacy_engine,
            target_delta,
        )
    )

    # ==============================================================================================
    # 16.25 Final RDP boundary validation
    # ==============================================================================================

    if (
        np.isclose(
            optimal_alpha,
            min(RDP_ALPHAS),
        )
        or
        np.isclose(
            optimal_alpha,
            max(RDP_ALPHAS),
        )
    ):

        raise RuntimeError(
            "Final RDP optimum is at an alpha boundary. "
            f"optimal_alpha={optimal_alpha}. "
            "Expand RDP_ALPHAS before accepting this experiment."
        )

    # ==============================================================================================
    # 16.26 Final assertions
    # ==============================================================================================

    assert completed_epoch >= total_epochs, (
        f"{dataset_id} has only "
        f"{completed_epoch}/{total_epochs} "
        f"completed epochs."
    )

    assert actual_epsilon <= (
        target_epsilon + 1e-6
    ), (
        f"Privacy budget exceeded for "
        f"{dataset_id}: "
        f"actual epsilon={actual_epsilon:.8f}, "
        f"target epsilon={target_epsilon:.8f}"
    )

    assert len(
        model.loss_history
    ) >= total_epochs

    assert len(
        model.privacy_history
    ) >= total_epochs

    # ==============================================================================================
    # 16.27 Store model
    # ==============================================================================================

    DPCTGAN_MODELS[
        dataset_id
    ] = model

    # ==============================================================================================
    # 16.28 Store training objects
    # ==============================================================================================

    DPCTGAN_TRAINING_OBJECTS[
        dataset_id
    ] = {

        "generator_optimizer":
            optimizer_g,

        "discriminator_optimizer":
            optimizer_d,

        "privacy_engine":
            privacy_engine,

        "dp_hooks":
            dp_hooks,

        "dp_loader":
            dp_loader,

        "noise_multiplier":
            actual_noise_multiplier,

        "sample_rate":
            actual_sample_rate,

        "max_grad_norm":
            actual_max_grad_norm,

        "actual_epsilon":
            actual_epsilon,

        "optimal_alpha":
            optimal_alpha,

        "target_epsilon":
            target_epsilon,

        "target_delta":
            target_delta,

        "runtime_seconds":
            runtime_seconds,

        "completed_epoch":
            completed_epoch,

        "checkpoint_interval":
            CHECKPOINT_INTERVAL,

        "rdp_alphas":
            list(RDP_ALPHAS),
    }

    # ==============================================================================================
    # 16.29 Global registries
    # ==============================================================================================

    PRIVACY_ENGINES[
        dataset_id
    ] = privacy_engine

    DP_OPTIMIZERS[
        dataset_id
    ] = optimizer_d

    DP_MODULES[
        dataset_id
    ] = model.discriminator

    DP_LOADERS[
        dataset_id
    ] = dp_loader

    DP_HOOKS[
        dataset_id
    ] = dp_hooks

    # ==============================================================================================
    # 16.30 Save final histories
    # ==============================================================================================

    history_dir = (
        NB07_HISTORY_ROOT
        / dataset_id
    )

    privacy_dir = (
        NB07_PRIVACY_ROOT
        / dataset_id
    )

    history_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    privacy_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    history_df = pd.DataFrame(
        model.loss_history
    )

    privacy_history_df = pd.DataFrame(
        model.privacy_history
    )

    history_path = (
        history_dir
        / "dp_ctgan_training_history.csv"
    )

    privacy_history_path = (
        privacy_dir
        / "dp_ctgan_privacy_history.csv"
    )

    history_df.to_csv(
        history_path,
        index=False,
    )

    privacy_history_df.to_csv(
        privacy_history_path,
        index=False,
    )

    # ==============================================================================================
    # 16.31 Save FINAL checkpoint
    # ==============================================================================================

    final_checkpoint_path = (
        get_final_checkpoint_path(
            dataset_id
        )
    )

    final_payload = build_checkpoint_payload(
        dataset_id=dataset_id,
        epoch=completed_epoch,
        model=model,
        optimizer_g=optimizer_g,
        optimizer_d=optimizer_d,
        privacy_engine=privacy_engine,
        dp_hooks=dp_hooks,
        dp_loader=dp_loader,
        seed=seed,
        target_epsilon=target_epsilon,
        target_delta=target_delta,
        noise_multiplier=actual_noise_multiplier,
        sample_rate=actual_sample_rate,
        max_grad_norm=actual_max_grad_norm,
        runtime_seconds=runtime_seconds,
        train_rows=len(train_df),
        train_columns=len(train_df.columns),
        transformed_dimension=transformed.shape[1],
        discrete_columns=discrete_columns,
    )

    atomic_torch_save(
        final_payload,
        final_checkpoint_path,
    )

    final_checkpoint_hash = (
        calculate_file_sha256(
            final_checkpoint_path
        )
    )

    final_reload = torch.load(
        final_checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_contents(
        final_reload,
        dataset_id,
    )

    assert int(
        final_reload["epoch"]
    ) == total_epochs

    assert np.isclose(
        float(
            final_reload["actual_epsilon"]
        ),
        actual_epsilon,
        rtol=1e-6,
        atol=1e-8,
    )

    print(
        f"✓ Final checkpoint saved : "
        f"{final_checkpoint_path}"
    )

    print(
        f"✓ Final checkpoint SHA256 : "
        f"{final_checkpoint_hash}"
    )

    # ==============================================================================================
    # 16.32 Record result
    # ==============================================================================================

    TRAINING_RESULTS.append(
        {
            "dataset_id":
                dataset_id,

            "rows":
                len(train_df),

            "columns":
                len(train_df.columns),

            "epochs":
                total_epochs,

            "completed_epochs":
                completed_epoch,

            "batch_size":
                DP_CTGAN_CONFIG[
                    "batch_size"
                ],

            "pac":
                DP_CTGAN_CONFIG[
                    "pac"
                ],

            "seed":
                seed,

            "runtime_seconds":
                runtime_seconds,

            "noise_multiplier":
                actual_noise_multiplier,

            "sample_rate":
                actual_sample_rate,

            "max_grad_norm":
                actual_max_grad_norm,

            "target_epsilon":
                target_epsilon,

            "actual_epsilon":
                actual_epsilon,

            "optimal_alpha":
                optimal_alpha,

            "target_delta":
                target_delta,

            "rdp_alpha_count":
                len(RDP_ALPHAS),

            "rdp_alpha_min":
                min(RDP_ALPHAS),

            "rdp_alpha_max":
                max(RDP_ALPHAS),

            "checkpoint_interval":
                CHECKPOINT_INTERVAL,

            "final_checkpoint":
                str(
                    final_checkpoint_path
                ),

            "final_checkpoint_sha256":
                final_checkpoint_hash,

            "status":
                "PASS",
        }
    )

    # ==============================================================================================
    # 16.33 Dataset completion
    # ==============================================================================================

    print(
        f"\n✓ Training completed : "
        f"{runtime_seconds:.2f} seconds"
    )

    print(
        f"✓ Completed epochs   : "
        f"{completed_epoch}/{total_epochs}"
    )

    print(
        f"✓ Final epsilon      : "
        f"{actual_epsilon:.5f}"
    )

    print(
        f"✓ Optimal alpha      : "
        f"{optimal_alpha:.4f}"
    )

    print(
        f"✓ Final delta        : "
        f"{target_delta:.8f}"
    )

    print(
        f"✓ Final checkpoint   : "
        f"{final_checkpoint_path}"
    )

    # ==============================================================================================
    # 16.34 Cleanup temporary data
    # ==============================================================================================

    del transformed
    del transformed_tensor
    del base_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ==================================================================================================
# 17. Final results
# ==================================================================================================

TRAINING_RESULTS_DF = pd.DataFrame(
    TRAINING_RESULTS
)

display(
    TRAINING_RESULTS_DF
)


# ==================================================================================================
# 18. Save final training results
# ==================================================================================================

NB07_TRAINING_RESULTS_PATH = (
    NB07_TRAINING_ROOT
    / "dp_ctgan_training_results.csv"
)

TRAINING_RESULTS_DF.to_csv(
    NB07_TRAINING_RESULTS_PATH,
    index=False,
)

print(
    f"✓ Training results saved : "
    f"{NB07_TRAINING_RESULTS_PATH}"
)


# ==================================================================================================
# 19. Final assertions
# ==================================================================================================

assert len(
    TRAINING_RESULTS_DF
) == len(
    DATASET_IDS
), (
    "Training results do not contain "
    "all registered datasets."
)

assert set(
    TRAINING_RESULTS_DF[
        "dataset_id"
    ]
) == set(
    DATASET_IDS
), (
    "Training results do not contain "
    "exactly the registered datasets."
)

assert (
    TRAINING_RESULTS_DF[
        "status"
    ] == "PASS"
).all(), (
    "One or more DP-CTGAN training runs failed."
)

assert (
    TRAINING_RESULTS_DF[
        "completed_epochs"
    ]
    >=
    TRAINING_RESULTS_DF[
        "epochs"
    ]
).all(), (
    "One or more datasets did not complete "
    "all training epochs."
)

assert (
    TRAINING_RESULTS_DF[
        "actual_epsilon"
    ]
    <=
    TRAINING_RESULTS_DF[
        "target_epsilon"
    ] + 1e-6
).all(), (
    "One or more datasets exceeded "
    "the configured privacy budget."
)

assert (
    TRAINING_RESULTS_DF[
        "optimal_alpha"
    ]
    >
    TRAINING_RESULTS_DF[
        "rdp_alpha_min"
    ]
).all()

assert (
    TRAINING_RESULTS_DF[
        "optimal_alpha"
    ]
    <
    TRAINING_RESULTS_DF[
        "rdp_alpha_max"
    ]
).all()

assert len(
    DPCTGAN_MODELS
) == len(
    DATASET_IDS
)

assert len(
    PRIVACY_ENGINES
) == len(
    DATASET_IDS
)

assert len(
    DP_OPTIMIZERS
) == len(
    DATASET_IDS
)

assert len(
    DP_MODULES
) == len(
    DATASET_IDS
)

assert len(
    DP_LOADERS
) == len(
    DATASET_IDS
)

assert len(
    DP_HOOKS
) == len(
    DATASET_IDS
)


# ==================================================================================================
# 20. Final checkpoint verification
# ==================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "FINAL CHECKPOINT VERIFICATION"
)

print(
    "=" * 100
)


for dataset_id in DATASET_IDS:

    final_path = (
        get_final_checkpoint_path(
            dataset_id
        )
    )

    assert final_path.exists(), (
        f"Missing final checkpoint "
        f"for {dataset_id}"
    )

    assert final_path.stat().st_size > 0, (
        f"Final checkpoint is empty "
        f"for {dataset_id}"
    )

    final_hash = (
        calculate_file_sha256(
            final_path
        )
    )

    final_checkpoint = torch.load(
        final_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_contents(
        final_checkpoint,
        dataset_id,
    )

    assert (
        final_checkpoint[
            "dataset_id"
        ]
        == dataset_id
    )

    assert int(
        final_checkpoint[
            "epoch"
        ]
    ) == int(
        DP_CTGAN_CONFIG[
            "epochs"
        ]
    )

    assert (
        "privacy_accountant_state"
        in final_checkpoint
    )

    assert (
        "generator_state_dict"
        in final_checkpoint
    )

    assert (
        "discriminator_state_dict"
        in final_checkpoint
    )

    assert (
        "generator_optimizer_state_dict"
        in final_checkpoint
    )

    assert (
        "discriminator_optimizer_state_dict"
        in final_checkpoint
    )

    assert (
        "rng_state"
        in final_checkpoint
    )

    assert (
        "rdp_accounting"
        in final_checkpoint
    )

    assert (
        final_checkpoint[
            "rdp_accounting"
        ]["alphas"]
        ==
        list(RDP_ALPHAS)
    )

    # ----------------------------------------------------------------------------------------------
    # Verify latest checkpoint against periodic/final state
    # ----------------------------------------------------------------------------------------------

    latest_path = (
        get_latest_checkpoint_path(
            dataset_id
        )
    )

    assert latest_path.exists(), (
        f"Missing latest checkpoint "
        f"for {dataset_id}"
    )

    latest_hash = (
        calculate_file_sha256(
            latest_path
        )
    )

    latest_checkpoint = torch.load(
        latest_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_contents(
        latest_checkpoint,
        dataset_id,
    )

    assert int(
        latest_checkpoint[
            "epoch"
        ]
    ) == int(
        DP_CTGAN_CONFIG[
            "epochs"
        ]
    )

    print(
        f"✓ {dataset_id:<20} "
        f"epoch={final_checkpoint['epoch']:>3} "
        f"ε={final_checkpoint['actual_epsilon']:.5f} "
        f"α={final_checkpoint['optimal_alpha']:.2f}"
    )

    print(
        f"  Final SHA256  : {final_hash}"
    )

    print(
        f"  Latest SHA256 : {latest_hash}"
    )


# ==================================================================================================
# 21. Completion
# ==================================================================================================

print(
    "\n" + "=" * 100
)

print(
    "SECTION 12 VERIFICATION"
)

print(
    "=" * 100
)

print(
    "✓ All DP-CTGAN training runs completed."
)

print(
    "✓ All datasets completed the configured "
    "number of epochs."
)

print(
    "✓ DP-SGD discriminator training executed."
)

print(
    "✓ Opacus gradient-sampling hooks managed explicitly."
)

print(
    "✓ Generator update executed without active "
    "Opacus hooks."
)

print(
    "✓ Poisson sampling verified."
)

print(
    "✓ Per-sample gradient clipping configured."
)

print(
    "✓ Gaussian noise configured."
)

print(
    "✓ RDP privacy accounting executed."
)

print(
    "✓ Expanded RDP alpha grid used."
)

print(
    "✓ Noise multiplier calibrated using the "
    "same RDP alpha grid."
)

print(
    "✓ RDP optimal alpha verified away from "
    "the configured boundary."
)

print(
    "✓ Privacy-accountant state persisted "
    "in checkpoints."
)

print(
    "✓ Generator optimizer state persisted."
)

print(
    "✓ Discriminator optimizer state persisted."
)

print(
    "✓ Python / NumPy / PyTorch / CUDA "
    "RNG states persisted."
)

print(
    "✓ Automatic checkpoint resume enabled."
)

print(
    "✓ Checkpoint serialization performed once "
    "per periodic checkpoint."
)

print(
    "✓ Latest checkpoint created by exact "
    "byte-copy from periodic checkpoint."
)

print(
    "✓ Checkpoint SHA-256 integrity verified."
)

print(
    "✓ Training histories persisted."
)

print(
    "✓ Privacy histories persisted."
)

print(
    "✓ Final checkpoints reloaded and verified."
)

print(
    "✓ Final training results verified."
)

print(
    "\n✓ SECTION 12 — PASS"
)

12. TRAIN WITH DP
✓ Checkpoint root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints
✓ History root    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/history
✓ Privacy root    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/privacy
✓ Training root   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/training
✓ Checkpoint freq : every 10 epochs
✓ Force restart   : False
✓ RDP alpha count : 219
✓ RDP alpha range : 1.1 → 256.0
✓ RDP tolerance   : 0.001

----------------------------------------------------------------------------------------------------
Training DP-CTGAN : adult_income
----------------------------------------------------------------------------------------------------
Rows                 : 34,189
Generative columns   : 15
Discrete columns     : 9
Transformed dimension: 158
Device               : cuda
✓ Discriminator is Opacus-compatible.


/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


RDP noise multiplier : 1.214905
RDP sample rate      : 0.00373134
RDP max grad norm    : 1.000000
Target epsilon       : 5.000000
Target delta         : 0.00001000
RDP alpha count      : 219
RDP alpha range      : 1.1 → 256.0
✓ Opacus gradient-sampling hooks attached.
✓ Original discriminator remains active.
✓ Discriminator remains Opacus-compatible.
✓ DP optimizer verified.
✓ Poisson DP loader verified.
✓ Initial epsilon     : 0.00000000
✓ Valid checkpoint found : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints/adult_income/latest_checkpoint.pt
  Completed epoch        : 300
  Stored epsilon         : 4.999028
  Optimal alpha          : 5.2000
  SHA256                 : 167f6b50f6190c7039fb412a26f10a4f4667160dba652a696c5ed67da560c342
✓ Checkpoint state restored.
  Resume epoch      : 301
  Restored epsilon  : 4.999028
  Optimal alpha     : 5.2000

RESUMING DP-CTGAN : adult_income
Checkpoint         : /content/drive/MyDrive/SPP_GAN_Research/results/not

/usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


RDP noise multiplier : 1.249695
RDP sample rate      : 0.00403226
RDP max grad norm    : 1.000000
Target epsilon       : 5.000000
Target delta         : 0.00001000
RDP alpha count      : 219
RDP alpha range      : 1.1 → 256.0
✓ Opacus gradient-sampling hooks attached.
✓ Original discriminator remains active.
✓ Discriminator remains Opacus-compatible.
✓ DP optimizer verified.
✓ Poisson DP loader verified.
✓ Initial epsilon     : 0.00000000
✓ Valid checkpoint found : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints/bank_marketing/latest_checkpoint.pt
  Completed epoch        : 300
  Stored epsilon         : 4.999538
  Optimal alpha          : 5.2000
  SHA256                 : af0b29e93cc0e28de98b98db4cb8dc5b5ea360d2ceac7802eeaab4a8e85b2eb4
✓ Checkpoint state restored.
  Resume epoch      : 301
  Restored epsilon  : 4.999538
  Optimal alpha     : 5.2000

RESUMING DP-CTGAN : bank_marketing
Checkpoint         : /content/drive/MyDrive/SPP_GAN_Research/results

...

RDP noise multiplier : 0.953217
RDP sample rate      : 0.00179533
RDP max grad norm    : 1.000000
Target epsilon       : 5.000000
Target delta         : 0.00001000
RDP alpha count      : 219
RDP alpha range      : 1.1 → 256.0
✓ Opacus gradient-sampling hooks attached.
✓ Original discriminator remains active.
✓ Discriminator remains Opacus-compatible.
✓ DP optimizer verified.
✓ Poisson DP loader verified.
✓ Initial epsilon     : 0.00000000
✓ Valid checkpoint found : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints/diabetes_130us/latest_checkpoint.pt
  Completed epoch        : 150
  Stored epsilon         : 3.388043
  Optimal alpha          : 6.7000
  SHA256                 : 6558e647989420d6fbaa988373cdfb9bb14b6c2809dbb6e68a116694371db4d0
✓ Checkpoint state restored.
  Resume epoch      : 151
  Restored epsilon  : 3.388043
  Optimal alpha     : 6.7000

RESUMING DP-CTGAN : diabetes_130us
Checkpoint         : /content/drive/MyDrive/SPP_GAN_Research/results

<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Epoch 151/300 | G=-84.77010 | D=-1.63021 | ε=3.40047 | α=6.70 | time=41.88s
Epoch 152/300 | G=-87.49317 | D=-1.23752 | ε=3.41289 | α=6.70 | time=40.46s
Epoch 153/300 | G=-87.27256 | D=-1.69357 | ε=3.42531 | α=6.70 | time=41.34s
Epoch 154/300 | G=-89.18555 | D=-1.00632 | ε=3.43774 | α=6.70 | time=42.21s
Epoch 155/300 | G=-91.67114 | D=-1.25485 | ε=3.45016 | α=6.70 | time=41.42s
Epoch 156/300 | G=-95.49928 | D=0.21762 | ε=3.46252 | α=6.60 | time=41.41s
Epoch 157/300 | G=-92.97533 | D=-0.57241 | ε=3.47475 | α=6.60 | time=41.12s
Epoch 158/300 | G=-94.40988 | D=-1.30259 | ε=3.48698 | α=6.60 | time=41.68s
Epoch 159/300 | G=-95.36205 | D=-1.11451 | ε=3.49921 | α=6.60 | time=41.25s
Epoch 160/300 | G=-98.05795 | D=-0.89436 | ε=3.51144 | α=6.60 | time=41.10s
✓ Checkpoint saved : epoch 160
  Periodic path    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints/diabetes_130us/dp_ctgan_epoch_0160.pt
  Latest path      : /content/drive/MyDrive/SPP_GAN_Research/results/

,dataset_id,rows,columns,epochs,completed_epochs,batch_size,pac,seed,runtime_seconds,noise_multiplier,...,actual_epsilon,optimal_alpha,target_delta,rdp_alpha_count,rdp_alpha_min,rdp_alpha_max,checkpoint_interval,final_checkpoint,final_checkpoint_sha256,status
0,adult_income,34189,15,300,300,128,1,3126,1.922281,1.214905,...,4.999028,5.2,0.00001,219,1.1,256,10,/content/drive/MyDrive/SPP_GAN_Research/result...,7cd26ea622bc1f1d74a82a374dbcc18df9f723005519df...,PASS
1,bank_marketing,31647,17,300,300,128,1,3226,1.894360,1.249695,...,4.999538,5.2,0.00001,219,1.1,256,10,/content/drive/MyDrive/SPP_GAN_Research/result...,7e45410ba7c1a1aa615bb40d54e8a933ad54c4e6c5beef...,PASS
2,diabetes_130us,71236,48,300,300,128,1,3326,6217.139438,0.953217,...,4.999684,5.2,0.00001,219,1.1,256,10,/content/drive/MyDrive/SPP_GAN_Research/result...,5f615b268251963ad73898fa3e32ad1e2636442b95d0bd...,PASS


✓ Training results saved : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/training/dp_ctgan_training_results.csv

FINAL CHECKPOINT VERIFICATION
✓ adult_income         epoch=300 ε=4.99903 α=5.20
  Final SHA256  : 7cd26ea622bc1f1d74a82a374dbcc18df9f723005519df2797c0b6b1fe50d730
  Latest SHA256 : 167f6b50f6190c7039fb412a26f10a4f4667160dba652a696c5ed67da560c342
✓ bank_marketing       epoch=300 ε=4.99954 α=5.20
  Final SHA256  : 7e45410ba7c1a1aa615bb40d54e8a933ad54c4e6c5beef8e545dc687e5f6143f
  Latest SHA256 : af0b29e93cc0e28de98b98db4cb8dc5b5ea360d2ceac7802eeaab4a8e85b2eb4
✓ diabetes_130us       epoch=300 ε=4.99968 α=5.20
  Final SHA256  : 5f615b268251963ad73898fa3e32ad1e2636442b95d0bd1dd07207394f7652a8
  Latest SHA256 : d3be4a9beab51da8ea19b2c36dc9ade074369068cc137961a068cca145eb9d85

SECTION 12 VERIFICATION
✓ All DP-CTGAN training runs completed.
✓ All datasets completed the configured number of epochs.
✓ DP-SGD discriminator training executed.
✓ Opacus gradient-sa

In [15]:
# ==================================================================================================
# 13. TRACK PRIVACY PARAMETERS
# ==================================================================================================

print("=" * 100)
print("13. TRACK PRIVACY PARAMETERS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Purpose
# --------------------------------------------------------------------------------------------------
# This section creates the canonical privacy registry for the DP-CTGAN baseline.
#
# The registry records:
#   1. Target privacy budget
#   2. Achieved privacy budget
#   3. Delta
#   4. Gradient clipping
#   5. Gaussian noise multiplier
#   6. Sampling rate
#   7. Poisson sampling configuration
#   8. Training configuration
#   9. Secure RNG configuration
#  10. Explicit DP protection boundary
#  11. End-to-end DP status
#
# IMPORTANT:
# DP-SGD is applied to discriminator training.
#
# DataTransformer fitting and DataSampler construction are data-dependent
# operations performed outside the DP-SGD mechanism. Therefore, this registry
# does NOT claim end-to-end record-level DP for the complete pipeline unless
# those additional operations are separately protected/accounted for.
#
# A run with secure_mode=False is retained as a diagnostic/reproducibility
# record but is NOT classified as publication-ready.
# --------------------------------------------------------------------------------------------------


# --------------------------------------------------------------------------------------------------
# 1. Reset Privacy Registry
# --------------------------------------------------------------------------------------------------

PRIVACY_TRACKING = []


# --------------------------------------------------------------------------------------------------
# 2. Validate Required Objects
# --------------------------------------------------------------------------------------------------

assert "DPCTGAN_TRAINING_OBJECTS" in globals(), (
    "DPCTGAN_TRAINING_OBJECTS is not available. "
    "Run the DP-CTGAN training section first."
)

assert "PRIVACY_PARAMETERS" in globals(), (
    "PRIVACY_PARAMETERS is not available. "
    "Run the privacy configuration section first."
)

assert "DP_CTGAN_CONFIG" in globals(), (
    "DP_CTGAN_CONFIG is not available."
)

assert "DP_CONFIG" in globals(), (
    "DP_CONFIG is not available."
)

assert "DATASET_IDS" in globals(), (
    "DATASET_IDS is not available."
)

assert len(DPCTGAN_TRAINING_OBJECTS) == len(DATASET_IDS), (
    "Training-object count does not match DATASET_IDS."
)


# --------------------------------------------------------------------------------------------------
# 3. Build Privacy Registry
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    assert dataset_id in DPCTGAN_TRAINING_OBJECTS, (
        f"Missing DP-CTGAN training object for dataset: {dataset_id}"
    )

    assert dataset_id in PRIVACY_PARAMETERS, (
        f"Missing privacy parameters for dataset: {dataset_id}"
    )

    obj = DPCTGAN_TRAINING_OBJECTS[dataset_id]
    params = PRIVACY_PARAMETERS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Retrieve achieved privacy values
    # ----------------------------------------------------------------------------------------------

    actual_epsilon = obj.get("actual_epsilon")
    noise_multiplier = obj.get("noise_multiplier")
    sample_rate = obj.get("sample_rate")

    # ----------------------------------------------------------------------------------------------
    # Validate essential privacy values
    # ----------------------------------------------------------------------------------------------

    assert actual_epsilon is not None, (
        f"{dataset_id}: achieved epsilon is missing."
    )

    assert np.isfinite(float(actual_epsilon)), (
        f"{dataset_id}: achieved epsilon is not finite."
    )

    assert noise_multiplier is not None, (
        f"{dataset_id}: noise multiplier is missing."
    )

    assert np.isfinite(float(noise_multiplier)), (
        f"{dataset_id}: noise multiplier is not finite."
    )

    assert sample_rate is not None, (
        f"{dataset_id}: sample rate is missing."
    )

    assert 0.0 < float(sample_rate) <= 1.0, (
        f"{dataset_id}: invalid sample rate: {sample_rate}"
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy-budget validation
    # ----------------------------------------------------------------------------------------------

    target_epsilon = float(params["target_epsilon"])
    target_delta = float(params["target_delta"])
    actual_epsilon = float(actual_epsilon)

    epsilon_within_target = (
        actual_epsilon <= target_epsilon + 0.05
    )

    # ----------------------------------------------------------------------------------------------
    # Secure RNG status
    # ----------------------------------------------------------------------------------------------

    secure_mode = bool(DP_CONFIG["secure_mode"])

    # ----------------------------------------------------------------------------------------------
    # Explicit privacy boundary
    # ----------------------------------------------------------------------------------------------
    #
    # DP-SGD protects discriminator training.
    #
    # The data-dependent transformer and conditional sampler are not treated
    # as independently DP mechanisms in this baseline.
    # ----------------------------------------------------------------------------------------------

    privacy_boundary = "DP-SGD discriminator training"

    end_to_end_dp = False

    formal_dp_status = (
        "DISCRIMINATOR_TRAINING_ONLY"
    )

    # ----------------------------------------------------------------------------------------------
    # Publication-readiness status
    # ----------------------------------------------------------------------------------------------
    #
    # A final PASS requires:
    #   - achieved epsilon within target
    #   - secure RNG enabled
    #   - complete training
    #
    # The current baseline privacy boundary remains explicitly limited to
    # discriminator DP-SGD.
    # ----------------------------------------------------------------------------------------------

    training_epochs_completed = obj.get(
        "completed_epochs",
        obj.get("epochs_completed", DP_CTGAN_CONFIG["epochs"])
    )

    target_epochs = int(DP_CTGAN_CONFIG["epochs"])

    training_complete = (
        training_epochs_completed is not None
        and int(training_epochs_completed) >= target_epochs
    )

    if (
        epsilon_within_target
        and secure_mode
        and training_complete
    ):
        status = "PASS"
    else:
        status = "DIAGNOSTIC_ONLY"

    # ----------------------------------------------------------------------------------------------
    # Registry record
    # ----------------------------------------------------------------------------------------------

    record = {
        # Dataset / algorithm
        "dataset_id": dataset_id,
        "algorithm": "DP-CTGAN",
        "dp_algorithm": "DP-SGD",

        # Privacy accounting
        "accountant": params["accountant"],
        "target_epsilon": target_epsilon,
        "actual_epsilon": actual_epsilon,
        "target_delta": target_delta,

        # Gradient privacy
        "max_grad_norm": float(params["max_grad_norm"]),
        "clipping": params["clipping"],
        "noise_multiplier": float(noise_multiplier),

        # Sampling
        "sample_rate": float(sample_rate),
        "poisson_sampling": bool(params["poisson_sampling"]),

        # Training configuration
        "epochs": target_epochs,
        "completed_epochs": int(training_epochs_completed),
        "batch_size": int(DP_CTGAN_CONFIG["batch_size"]),
        "pac": int(DP_CTGAN_CONFIG["pac"]),

        # Secure execution
        "secure_mode": secure_mode,

        # Explicit privacy boundary
        "privacy_boundary": privacy_boundary,
        "end_to_end_dp": end_to_end_dp,
        "formal_dp_status": formal_dp_status,

        # Validation flags
        "epsilon_within_target": epsilon_within_target,
        "training_complete": training_complete,

        # Final registry status
        "status": status,
    }

    PRIVACY_TRACKING.append(record)


# --------------------------------------------------------------------------------------------------
# 4. Create Privacy DataFrame
# --------------------------------------------------------------------------------------------------

PRIVACY_TRACKING_DF = pd.DataFrame(
    PRIVACY_TRACKING
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Registry Structure
# --------------------------------------------------------------------------------------------------

required_columns = [
    "dataset_id",
    "algorithm",
    "dp_algorithm",
    "accountant",
    "target_epsilon",
    "actual_epsilon",
    "target_delta",
    "max_grad_norm",
    "clipping",
    "noise_multiplier",
    "sample_rate",
    "poisson_sampling",
    "epochs",
    "completed_epochs",
    "batch_size",
    "pac",
    "secure_mode",
    "privacy_boundary",
    "end_to_end_dp",
    "formal_dp_status",
    "epsilon_within_target",
    "training_complete",
    "status",
]

missing_columns = [
    column
    for column in required_columns
    if column not in PRIVACY_TRACKING_DF.columns
]

assert not missing_columns, (
    f"Privacy registry is missing columns: {missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 6. Dataset Coverage Validation
# --------------------------------------------------------------------------------------------------

assert len(PRIVACY_TRACKING_DF) == len(DATASET_IDS), (
    "Privacy registry does not contain exactly one record per dataset."
)

assert set(PRIVACY_TRACKING_DF["dataset_id"]) == set(DATASET_IDS), (
    "Privacy registry dataset coverage does not match DATASET_IDS."
)


# --------------------------------------------------------------------------------------------------
# 7. Privacy Budget Validation
# --------------------------------------------------------------------------------------------------

assert (
    PRIVACY_TRACKING_DF["actual_epsilon"]
    <= PRIVACY_TRACKING_DF["target_epsilon"] + 0.05
).all(), (
    "One or more datasets exceed the allowed epsilon tolerance."
)


# --------------------------------------------------------------------------------------------------
# 8. Privacy Parameter Validation
# --------------------------------------------------------------------------------------------------

assert (
    PRIVACY_TRACKING_DF["max_grad_norm"] > 0
).all(), (
    "Invalid max_grad_norm detected."
)

assert (
    PRIVACY_TRACKING_DF["noise_multiplier"] > 0
).all(), (
    "Invalid noise multiplier detected."
)

assert (
    (PRIVACY_TRACKING_DF["sample_rate"] > 0)
    & (PRIVACY_TRACKING_DF["sample_rate"] <= 1)
).all(), (
    "Invalid sampling rate detected."
)

assert (
    PRIVACY_TRACKING_DF["poisson_sampling"]
).all(), (
    "Poisson sampling is not enabled for one or more datasets."
)


# --------------------------------------------------------------------------------------------------
# 9. Privacy Boundary Validation
# --------------------------------------------------------------------------------------------------

assert (
    PRIVACY_TRACKING_DF["privacy_boundary"]
    == "DP-SGD discriminator training"
).all(), (
    "Unexpected privacy boundary detected."
)

assert not PRIVACY_TRACKING_DF["end_to_end_dp"].any(), (
    "The current baseline must not claim end-to-end DP."
)

assert (
    PRIVACY_TRACKING_DF["formal_dp_status"]
    == "DISCRIMINATOR_TRAINING_ONLY"
).all(), (
    "Formal DP boundary status is inconsistent."
)


# --------------------------------------------------------------------------------------------------
# 10. Publication-Readiness Summary
# --------------------------------------------------------------------------------------------------

publication_ready = (
    PRIVACY_TRACKING_DF["epsilon_within_target"]
    & PRIVACY_TRACKING_DF["secure_mode"]
    & PRIVACY_TRACKING_DF["training_complete"]
)

publication_ready_all = bool(publication_ready.all())


# --------------------------------------------------------------------------------------------------
# 11. Persist Privacy Registry
# --------------------------------------------------------------------------------------------------

privacy_registry_path = (
    NB07_PRIVACY_ROOT
    / "dp_ctgan_privacy_registry.csv"
)

privacy_registry_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

PRIVACY_TRACKING_DF.to_csv(
    privacy_registry_path,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 12. Display Registry
# --------------------------------------------------------------------------------------------------

display(PRIVACY_TRACKING_DF)


# --------------------------------------------------------------------------------------------------
# 13. Print Privacy Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY REGISTRY SUMMARY")
print("-" * 100)

print(
    f"Datasets tracked          : {len(PRIVACY_TRACKING_DF)}"
)

print(
    f"Target epsilon            : "
    f"{PRIVACY_TRACKING_DF['target_epsilon'].tolist()}"
)

print(
    f"Achieved epsilon          : "
    f"{PRIVACY_TRACKING_DF['actual_epsilon'].round(6).tolist()}"
)

print(
    f"Target delta              : "
    f"{PRIVACY_TRACKING_DF['target_delta'].tolist()}"
)

print(
    f"Secure mode               : "
    f"{PRIVACY_TRACKING_DF['secure_mode'].tolist()}"
)

print(
    f"Poisson sampling          : "
    f"{PRIVACY_TRACKING_DF['poisson_sampling'].tolist()}"
)

print(
    f"Training complete         : "
    f"{PRIVACY_TRACKING_DF['training_complete'].tolist()}"
)

print(
    f"Privacy boundary          : "
    f"{PRIVACY_TRACKING_DF['privacy_boundary'].iloc[0]}"
)

print(
    f"End-to-end DP claimed     : "
    f"{PRIVACY_TRACKING_DF['end_to_end_dp'].any()}"
)

print(
    f"Publication-ready         : "
    f"{publication_ready_all}"
)

print(
    f"Registry saved            : "
    f"{privacy_registry_path}"
)


# --------------------------------------------------------------------------------------------------
# 14. Final Section Status
# --------------------------------------------------------------------------------------------------

if publication_ready_all:

    print("\n✓ SECTION 13 — PASS")

else:

    print(
        "\n⚠ SECTION 13 — DIAGNOSTIC RUN RECORDED; "
        "PUBLICATION-READY PASS REQUIRES SECURE MODE "
        "AND COMPLETE TRAINING."
    )

13. TRACK PRIVACY PARAMETERS


,dataset_id,algorithm,dp_algorithm,accountant,target_epsilon,actual_epsilon,target_delta,max_grad_norm,clipping,noise_multiplier,...,completed_epochs,batch_size,pac,secure_mode,privacy_boundary,end_to_end_dp,formal_dp_status,epsilon_within_target,training_complete,status
0,adult_income,DP-CTGAN,DP-SGD,rdp,5.0,4.999028,0.00001,1.0,flat,1.214905,...,300,128,1,False,DP-SGD discriminator training,False,DISCRIMINATOR_TRAINING_ONLY,True,True,DIAGNOSTIC_ONLY
1,bank_marketing,DP-CTGAN,DP-SGD,rdp,5.0,4.999538,0.00001,1.0,flat,1.249695,...,300,128,1,False,DP-SGD discriminator training,False,DISCRIMINATOR_TRAINING_ONLY,True,True,DIAGNOSTIC_ONLY
2,diabetes_130us,DP-CTGAN,DP-SGD,rdp,5.0,4.999684,0.00001,1.0,flat,0.953217,...,300,128,1,False,DP-SGD discriminator training,False,DISCRIMINATOR_TRAINING_ONLY,True,True,DIAGNOSTIC_ONLY



----------------------------------------------------------------------------------------------------
PRIVACY REGISTRY SUMMARY
----------------------------------------------------------------------------------------------------
Datasets tracked          : 3
Target epsilon            : [5.0, 5.0, 5.0]
Achieved epsilon          : [4.999028, 4.999538, 4.999684]
Target delta              : [1e-05, 1e-05, 1e-05]
Secure mode               : [False, False, False]
Poisson sampling          : [True, True, True]
Training complete         : [True, True, True]
Privacy boundary          : DP-SGD discriminator training
End-to-end DP claimed     : False
Publication-ready         : False
Registry saved            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/privacy/dp_ctgan_privacy_registry.csv

⚠ SECTION 13 — DIAGNOSTIC RUN RECORDED; PUBLICATION-READY PASS REQUIRES SECURE MODE AND COMPLETE TRAINING.


In [16]:
# ==================================================================================================
# 14. CALCULATE ACTUAL PRIVACY BUDGET
# ==================================================================================================

print("=" * 100)
print("14. CALCULATE ACTUAL PRIVACY BUDGET")
print("=" * 100)

ACTUAL_PRIVACY_BUDGET = []

for dataset_id in DATASET_IDS:

    engine = PRIVACY_ENGINES[dataset_id]
    params = PRIVACY_PARAMETERS[dataset_id]

    actual_epsilon = float(
        engine.get_epsilon(
            delta=params["target_delta"]
        )
    )

    noise_multiplier = float(
        DP_OPTIMIZERS[dataset_id].noise_multiplier
    )

    record = {
        "dataset_id": dataset_id,
        "target_epsilon": params["target_epsilon"],
        "actual_epsilon": actual_epsilon,
        "target_delta": params["target_delta"],
        "noise_multiplier": noise_multiplier,
        "accountant": params["accountant"],
        "max_grad_norm": params["max_grad_norm"],
        "sample_rate": DPCTGAN_TRAINING_OBJECTS[
            dataset_id
        ]["sample_rate"],
        "privacy_budget_satisfied": (
            actual_epsilon
            <= params["target_epsilon"] + 0.05
        ),
    }

    ACTUAL_PRIVACY_BUDGET.append(record)

ACTUAL_PRIVACY_DF = pd.DataFrame(
    ACTUAL_PRIVACY_BUDGET
)

actual_privacy_path = (
    NB07_PRIVACY_ROOT
    / "actual_privacy_budget.csv"
)

ACTUAL_PRIVACY_DF.to_csv(
    actual_privacy_path,
    index=False,
)

display(ACTUAL_PRIVACY_DF)

assert (
    ACTUAL_PRIVACY_DF[
        "privacy_budget_satisfied"
    ]
).all()

print("\n✓ Actual ε calculated from the privacy accountant.")
print("\n✓ SECTION 14 — PASS")

14. CALCULATE ACTUAL PRIVACY BUDGET


,dataset_id,target_epsilon,actual_epsilon,target_delta,noise_multiplier,accountant,max_grad_norm,sample_rate,privacy_budget_satisfied
0,adult_income,5.0,4.999028,0.00001,1.214905,rdp,1.0,0.003731,True
1,bank_marketing,5.0,4.999538,0.00001,1.249695,rdp,1.0,0.004032,True
2,diabetes_130us,5.0,4.999684,0.00001,0.953217,rdp,1.0,0.001795,True



✓ Actual ε calculated from the privacy accountant.

✓ SECTION 14 — PASS


In [17]:
# ==================================================================================================
# 15. GENERATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("15. GENERATE SYNTHETIC DATA")
print("=" * 100)

DPCTGAN_SYNTHETIC_DATA = {}
GENERATION_RESULTS = []

for dataset_id in DATASET_IDS:

    seed_everything(
        DATASET_SEEDS[dataset_id]
    )

    model = DPCTGAN_MODELS[dataset_id]

    n_rows = len(
        TRAINING_DATA[dataset_id]
    )

    start_time = time.time()

    synthetic_df = model.sample(
        n=n_rows
    )

    generation_runtime = (
        time.time() - start_time
    )

    # ----------------------------------------------------------------------------------------------
    # Validation
    # ----------------------------------------------------------------------------------------------

    expected_columns = TRAINING_SCHEMA[
        dataset_id
    ]

    assert isinstance(
        synthetic_df,
        pd.DataFrame
    )

    assert len(synthetic_df) == n_rows

    assert list(
        synthetic_df.columns
    ) == expected_columns

    assert not synthetic_df.columns.duplicated().any()

    assert TARGET_COLUMNS[
        dataset_id
    ] in synthetic_df.columns

    assert PROVENANCE_COLUMN not in synthetic_df.columns

    for identifier in IDENTIFIER_COLUMNS[
        dataset_id
    ]:

        assert identifier not in synthetic_df.columns

    DPCTGAN_SYNTHETIC_DATA[
        dataset_id
    ] = synthetic_df

    GENERATION_RESULTS.append({
        "dataset_id": dataset_id,
        "training_rows": n_rows,
        "synthetic_rows": len(synthetic_df),
        "columns": len(synthetic_df.columns),
        "generation_runtime_seconds": generation_runtime,
        "status": "PASS",
    })

    print(
        f"{dataset_id:<18} | "
        f"{len(synthetic_df):>8,} rows | "
        f"{len(synthetic_df.columns):>3} columns | "
        f"{generation_runtime:.3f} sec | PASS"
    )

GENERATION_RESULTS_DF = pd.DataFrame(
    GENERATION_RESULTS
)

display(GENERATION_RESULTS_DF)

assert (
    GENERATION_RESULTS_DF["status"] == "PASS"
).all()

print("\n✓ SECTION 15 — PASS")

15. GENERATE SYNTHETIC DATA
adult_income       |   34,189 rows |  15 columns | 0.835 sec | PASS
bank_marketing     |   31,647 rows |  17 columns | 0.777 sec | PASS
diabetes_130us     |   71,236 rows |  48 columns | 9.338 sec | PASS


,dataset_id,training_rows,synthetic_rows,columns,generation_runtime_seconds,status
0,adult_income,34189,34189,15,0.834600,PASS
1,bank_marketing,31647,31647,17,0.777305,PASS
2,diabetes_130us,71236,71236,48,9.338231,PASS



✓ SECTION 15 — PASS


In [18]:
# ==================================================================================================
# 16. VALIDATE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("16. VALIDATE SYNTHETIC DATA")
print("=" * 100)

SYNTHETIC_VALIDATION = []

for dataset_id in DATASET_IDS:

    synthetic_df = DPCTGAN_SYNTHETIC_DATA[
        dataset_id
    ]

    training_df = TRAINING_DATA[
        dataset_id
    ]

    target = TARGET_COLUMNS[
        dataset_id
    ]

    identifiers = IDENTIFIER_COLUMNS[
        dataset_id
    ]

    schema_ok = (
        list(synthetic_df.columns)
        == list(training_df.columns)
    )

    size_ok = (
        len(synthetic_df)
        == len(training_df)
    )

    target_ok = target in synthetic_df.columns

    provenance_ok = (
        PROVENANCE_COLUMN
        not in synthetic_df.columns
    )

    identifiers_ok = all(
        identifier not in synthetic_df.columns
        for identifier in identifiers
    )

    duplicate_ok = not synthetic_df.columns.duplicated().any()

    record = {
        "dataset_id": dataset_id,
        "schema_ok": schema_ok,
        "size_ok": size_ok,
        "target_ok": target_ok,
        "provenance_excluded": provenance_ok,
        "identifiers_excluded": identifiers_ok,
        "duplicate_columns_ok": duplicate_ok,
        "status": (
            "PASS"
            if all([
                schema_ok,
                size_ok,
                target_ok,
                provenance_ok,
                identifiers_ok,
                duplicate_ok,
            ])
            else "FAIL"
        ),
    }

    SYNTHETIC_VALIDATION.append(record)

SYNTHETIC_VALIDATION_DF = pd.DataFrame(
    SYNTHETIC_VALIDATION
)

display(SYNTHETIC_VALIDATION_DF)

assert (
    SYNTHETIC_VALIDATION_DF["status"] == "PASS"
).all()

print("\n✓ SECTION 16 — PASS")

16. VALIDATE SYNTHETIC DATA


,dataset_id,schema_ok,size_ok,target_ok,provenance_excluded,identifiers_excluded,duplicate_columns_ok,status
0,adult_income,True,True,True,True,True,True,PASS
1,bank_marketing,True,True,True,True,True,True,PASS
2,diabetes_130us,True,True,True,True,True,True,PASS



✓ SECTION 16 — PASS


In [19]:
# ==================================================================================================
# 17. RECORD RUNTIME
# ==================================================================================================

print("=" * 100)
print("17. RECORD RUNTIME")
print("=" * 100)

RUNTIME_RECORDS = []

for dataset_id in DATASET_IDS:

    train_result = (
        TRAINING_RESULTS_DF[
            TRAINING_RESULTS_DF["dataset_id"]
            == dataset_id
        ]
        .iloc[0]
    )

    generation_result = (
        GENERATION_RESULTS_DF[
            GENERATION_RESULTS_DF["dataset_id"]
            == dataset_id
        ]
        .iloc[0]
    )

    training_runtime = float(
        train_result["runtime_seconds"]
    )

    generation_runtime = float(
        generation_result[
            "generation_runtime_seconds"
        ]
    )

    total_runtime = (
        training_runtime
        + generation_runtime
    )

    assert np.isfinite(training_runtime)
    assert np.isfinite(generation_runtime)
    assert np.isfinite(total_runtime)

    RUNTIME_RECORDS.append({
        "dataset_id": dataset_id,
        "training_runtime_seconds": training_runtime,
        "generation_runtime_seconds": generation_runtime,
        "total_runtime_seconds": total_runtime,
        "status": "PASS",
    })

RUNTIME_DF = pd.DataFrame(
    RUNTIME_RECORDS
)

runtime_path = (
    NB07_HISTORY_ROOT
    / "dp_ctgan_runtime_summary.csv"
)

RUNTIME_DF.to_csv(
    runtime_path,
    index=False,
)

display(RUNTIME_DF)

assert len(RUNTIME_DF) == len(DATASET_IDS)

print("\n✓ SECTION 17 — PASS")

17. RECORD RUNTIME


,dataset_id,training_runtime_seconds,generation_runtime_seconds,total_runtime_seconds,status
0,adult_income,1.922281,0.834600,2.756881,PASS
1,bank_marketing,1.894360,0.777305,2.671664,PASS
2,diabetes_130us,6217.139438,9.338231,6226.477668,PASS



✓ SECTION 17 — PASS


In [21]:
# ==================================================================================================
# 18. SAVE MODEL / CHECKPOINTS
# ==================================================================================================

print("=" * 100)
print("18. SAVE MODEL / CHECKPOINTS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Purpose
# --------------------------------------------------------------------------------------------------
# Save publication-grade DP-CTGAN model and training checkpoint artifacts.
#
# Model artifact:
#   - Generator state
#   - Discriminator state
#   - Transformer
#   - Data sampler
#   - Model configuration
#   - Discrete-column schema
#   - Output dimensions
#   - Dataset seed
#
# Training checkpoint:
#   - Model states
#   - Generator optimizer state
#   - Discriminator optimizer state
#   - Privacy accountant state
#   - Privacy parameters
#   - Training histories
#   - Runtime
#
# Additional integrity controls:
#   - SHA-256 hashes
#   - Non-empty file validation
#   - Reload validation
#   - Required-key validation
#   - Dataset identity validation
#   - Privacy-boundary metadata
#
# IMPORTANT:
# This section does NOT retrain any model.
#
# The current run may have secure_mode=False. Such artifacts are retained and
# explicitly marked as diagnostic-only. A later secure_mode=True final run
# should generate its own final artifacts without overwriting the diagnostic
# artifacts.
# --------------------------------------------------------------------------------------------------


# --------------------------------------------------------------------------------------------------
# 1. Validate Required Objects
# --------------------------------------------------------------------------------------------------

assert "DPCTGAN_MODELS" in globals(), (
    "DPCTGAN_MODELS is not available. "
    "Run the DP-CTGAN training section first."
)

assert "DPCTGAN_TRAINING_OBJECTS" in globals(), (
    "DPCTGAN_TRAINING_OBJECTS is not available."
)

assert "PRIVACY_PARAMETERS" in globals(), (
    "PRIVACY_PARAMETERS is not available."
)

assert "DATASET_IDS" in globals(), (
    "DATASET_IDS is not available."
)

assert "DP_CTGAN_CONFIG" in globals(), (
    "DP_CTGAN_CONFIG is not available."
)

assert "DP_CONFIG" in globals(), (
    "DP_CONFIG is not available."
)


# --------------------------------------------------------------------------------------------------
# 2. Reset Registries
# --------------------------------------------------------------------------------------------------

MODEL_REGISTRY = []
CHECKPOINT_REGISTRY = []


# --------------------------------------------------------------------------------------------------
# 3. SHA-256 Helper
# --------------------------------------------------------------------------------------------------

def sha256_file(path):

    path = Path(path)

    assert path.exists(), (
        f"File does not exist: {path}"
    )

    digest = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. Safe Hook Cleanup Helper
# --------------------------------------------------------------------------------------------------
#
# Hooks are removed before saving the standalone model artifact because the
# standalone inference/generation artifact does not require active DP hooks.
#
# The privacy accountant state is independently preserved in the training
# checkpoint.
#
# Cleanup failures are NOT silently ignored.
# --------------------------------------------------------------------------------------------------

def cleanup_privacy_hooks(training_objects, dataset_id):

    hooks = training_objects.get(
        "privacy_hooks"
    )

    if hooks is None:
        return {
            "hooks_present": False,
            "cleanup_status": "NOT_PRESENT",
            "cleanup_error": None,
        }

    try:

        hooks.cleanup()

        return {
            "hooks_present": True,
            "cleanup_status": "SUCCESS",
            "cleanup_error": None,
        }

    except Exception as exc:

        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"⚠ {dataset_id}: privacy-hook cleanup "
            f"reported an exception: {error_message}"
        )

        return {
            "hooks_present": True,
            "cleanup_status": "ERROR",
            "cleanup_error": error_message,
        }


# --------------------------------------------------------------------------------------------------
# 5. Validate Model Bundle
# --------------------------------------------------------------------------------------------------

def validate_model_bundle(
    bundle,
    dataset_id,
):

    required_keys = [
        "notebook_id",
        "notebook_version",
        "dataset_id",
        "model_type",
        "generator_state_dict",
        "discriminator_state_dict",
        "transformer",
        "data_sampler",
        "model_config",
        "discrete_columns",
        "output_dimensions",
        "seed",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in bundle
    ]

    assert not missing_keys, (
        f"{dataset_id}: model bundle missing keys: "
        f"{missing_keys}"
    )

    assert bundle["dataset_id"] == dataset_id, (
        f"{dataset_id}: model bundle dataset ID mismatch."
    )

    assert bundle["model_type"] == "DP-CTGAN", (
        f"{dataset_id}: unexpected model type."
    )

    assert len(
        bundle["generator_state_dict"]
    ) > 0, (
        f"{dataset_id}: generator state dictionary is empty."
    )

    assert len(
        bundle["discriminator_state_dict"]
    ) > 0, (
        f"{dataset_id}: discriminator state dictionary is empty."
    )

    return True


# --------------------------------------------------------------------------------------------------
# 6. Validate Checkpoint Bundle
# --------------------------------------------------------------------------------------------------

def validate_checkpoint_bundle(
    bundle,
    dataset_id,
):

    required_keys = [
        "notebook_id",
        "notebook_version",
        "dataset_id",
        "model_type",
        "generator_state_dict",
        "discriminator_state_dict",
        "transformer",
        "data_sampler",
        "model_config",
        "discrete_columns",
        "output_dimensions",
        "seed",
        "generator_optimizer_state_dict",
        "discriminator_optimizer_state_dict",
        "privacy_accountant_state",
        "privacy_parameters",
        "loss_history",
        "privacy_history",
        "runtime_seconds",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in bundle
    ]

    assert not missing_keys, (
        f"{dataset_id}: checkpoint missing keys: "
        f"{missing_keys}"
    )

    assert bundle["dataset_id"] == dataset_id, (
        f"{dataset_id}: checkpoint dataset ID mismatch."
    )

    assert bundle["model_type"] == "DP-CTGAN", (
        f"{dataset_id}: unexpected checkpoint model type."
    )

    assert len(
        bundle["generator_state_dict"]
    ) > 0, (
        f"{dataset_id}: checkpoint generator state is empty."
    )

    assert len(
        bundle["discriminator_state_dict"]
    ) > 0, (
        f"{dataset_id}: checkpoint discriminator state is empty."
    )

    assert isinstance(
        bundle["privacy_parameters"],
        dict,
    ), (
        f"{dataset_id}: invalid privacy parameter object."
    )

    assert bundle[
        "privacy_accountant_state"
    ] is not None, (
        f"{dataset_id}: privacy accountant state is missing."
    )

    assert np.isfinite(
        float(bundle["runtime_seconds"])
    ), (
        f"{dataset_id}: invalid runtime value."
    )

    return True


# --------------------------------------------------------------------------------------------------
# 7. Save and Validate Artifacts
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"SAVING ARTIFACTS — {dataset_id}")
    print("-" * 100)

    assert dataset_id in DPCTGAN_MODELS, (
        f"Missing trained model: {dataset_id}"
    )

    assert dataset_id in DPCTGAN_TRAINING_OBJECTS, (
        f"Missing training object: {dataset_id}"
    )

    model = DPCTGAN_MODELS[
        dataset_id
    ]

    training_objects = DPCTGAN_TRAINING_OBJECTS[
        dataset_id
    ]

    privacy_params = PRIVACY_PARAMETERS[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Secure-mode state
    # ----------------------------------------------------------------------------------------------

    secure_mode = bool(
        DP_CONFIG["secure_mode"]
    )

    # ----------------------------------------------------------------------------------------------
    # Training completion state
    # ----------------------------------------------------------------------------------------------

    completed_epochs = training_objects.get(
        "completed_epochs",
        training_objects.get(
            "epochs_completed",
            DP_CTGAN_CONFIG["epochs"],
        ),
    )

    completed_epochs = int(
        completed_epochs
    )

    target_epochs = int(
        DP_CTGAN_CONFIG["epochs"]
    )

    training_complete = (
        completed_epochs >= target_epochs
    )

    # ----------------------------------------------------------------------------------------------
    # Current run classification
    # ----------------------------------------------------------------------------------------------

    if secure_mode and training_complete:

        run_status = "FINAL_CANDIDATE"

    else:

        run_status = "DIAGNOSTIC_ONLY"

    # ----------------------------------------------------------------------------------------------
    # Output directories
    # ----------------------------------------------------------------------------------------------

    model_dir = (
        NB07_MODEL_ROOT
        / dataset_id
    )

    checkpoint_dir = (
        NB07_CHECKPOINT_ROOT
        / dataset_id
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ----------------------------------------------------------------------------------------------
    # Artifact paths
    # ----------------------------------------------------------------------------------------------

    model_path = (
        model_dir
        / "dp_ctgan_model.pt"
    )

    checkpoint_path = (
        checkpoint_dir
        / "dp_ctgan_checkpoint.pt"
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy-hook cleanup
    # ----------------------------------------------------------------------------------------------

    hook_result = cleanup_privacy_hooks(
        training_objects,
        dataset_id,
    )

    # ----------------------------------------------------------------------------------------------
    # Model bundle
    # ----------------------------------------------------------------------------------------------

    model_bundle = {
        "notebook_id": NOTEBOOK_ID,
        "notebook_version": NOTEBOOK_VERSION,
        "dataset_id": dataset_id,
        "model_type": "DP-CTGAN",

        "generator_state_dict": {
            key: value.detach().cpu()
            for key, value
            in model.generator.state_dict().items()
        },

        "discriminator_state_dict": {
            key: value.detach().cpu()
            for key, value
            in model.discriminator.state_dict().items()
        },

        "transformer": model.transformer,

        "data_sampler": model.data_sampler,

        "model_config": DP_CTGAN_CONFIG,

        "discrete_columns": model.discrete_columns,

        "output_dimensions": model.output_dimensions,

        "seed": DATASET_SEEDS[dataset_id],

        # Run metadata
        "secure_mode": secure_mode,
        "run_status": run_status,
        "completed_epochs": completed_epochs,
        "target_epochs": target_epochs,

        # Explicit privacy boundary
        "privacy_boundary": (
            "DP-SGD discriminator training"
        ),

        "end_to_end_dp": False,
    }

    # ----------------------------------------------------------------------------------------------
    # Validate model bundle before serialization
    # ----------------------------------------------------------------------------------------------

    validate_model_bundle(
        model_bundle,
        dataset_id,
    )

    # ----------------------------------------------------------------------------------------------
    # Save model artifact
    # ----------------------------------------------------------------------------------------------

    torch.save(
        model_bundle,
        model_path,
    )

    assert model_path.exists(), (
        f"{dataset_id}: model artifact was not created."
    )

    assert model_path.stat().st_size > 0, (
        f"{dataset_id}: model artifact is empty."
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy accountant state
    # ----------------------------------------------------------------------------------------------

    privacy_engine = training_objects.get(
        "privacy_engine"
    )

    assert privacy_engine is not None, (
        f"{dataset_id}: privacy engine is missing."
    )

    accountant_state = (
        privacy_engine.accountant.state_dict()
    )

    assert accountant_state is not None, (
        f"{dataset_id}: accountant state is missing."
    )

    # ----------------------------------------------------------------------------------------------
    # Optimizer state
    # ----------------------------------------------------------------------------------------------

    generator_optimizer = training_objects.get(
        "generator_optimizer"
    )

    discriminator_optimizer = training_objects.get(
        "discriminator_optimizer"
    )

    assert generator_optimizer is not None, (
        f"{dataset_id}: generator optimizer is missing."
    )

    assert discriminator_optimizer is not None, (
        f"{dataset_id}: discriminator optimizer is missing."
    )

    # Opacus wraps the original discriminator optimizer.
    if hasattr(
        discriminator_optimizer,
        "original_optimizer",
    ):

        discriminator_optimizer_state = (
            discriminator_optimizer
            .original_optimizer
            .state_dict()
        )

    else:

        discriminator_optimizer_state = (
            discriminator_optimizer
            .state_dict()
        )

    # ----------------------------------------------------------------------------------------------
    # Full training checkpoint
    # ----------------------------------------------------------------------------------------------

    checkpoint_bundle = {
        **model_bundle,

        "generator_optimizer_state_dict": (
            generator_optimizer.state_dict()
        ),

        "discriminator_optimizer_state_dict": (
            discriminator_optimizer_state
        ),

        "privacy_accountant_state": (
            accountant_state
        ),

        "privacy_parameters": privacy_params,

        "loss_history": model.loss_history,

        "privacy_history": model.privacy_history,

        "runtime_seconds": float(
            training_objects[
                "runtime_seconds"
            ]
        ),

        # Checkpoint metadata
        "completed_epochs": completed_epochs,

        "target_epochs": target_epochs,

        "secure_mode": secure_mode,

        "run_status": run_status,

        "privacy_boundary": (
            "DP-SGD discriminator training"
        ),

        "end_to_end_dp": False,

        "privacy_hook_cleanup": hook_result,
    }

    # ----------------------------------------------------------------------------------------------
    # Validate checkpoint before serialization
    # ----------------------------------------------------------------------------------------------

    validate_checkpoint_bundle(
        checkpoint_bundle,
        dataset_id,
    )

    # ----------------------------------------------------------------------------------------------
    # Save checkpoint
    # ----------------------------------------------------------------------------------------------

    torch.save(
        checkpoint_bundle,
        checkpoint_path,
    )

    assert checkpoint_path.exists(), (
        f"{dataset_id}: checkpoint was not created."
    )

    assert checkpoint_path.stat().st_size > 0, (
        f"{dataset_id}: checkpoint is empty."
    )

    # ----------------------------------------------------------------------------------------------
    # File hashes
    # ----------------------------------------------------------------------------------------------

    model_hash = sha256_file(
        model_path
    )

    checkpoint_hash = sha256_file(
        checkpoint_path
    )

    # ----------------------------------------------------------------------------------------------
    # File sizes
    # ----------------------------------------------------------------------------------------------

    model_size = model_path.stat().st_size

    checkpoint_size = checkpoint_path.stat().st_size

    assert model_size > 0
    assert checkpoint_size > 0

    # ----------------------------------------------------------------------------------------------
    # Reload model artifact
    # ----------------------------------------------------------------------------------------------

    reloaded_model_bundle = torch.load(
        model_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_model_bundle(
        reloaded_model_bundle,
        dataset_id,
    )

    # ----------------------------------------------------------------------------------------------
    # Reload checkpoint artifact
    # ----------------------------------------------------------------------------------------------

    reloaded_checkpoint_bundle = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    validate_checkpoint_bundle(
        reloaded_checkpoint_bundle,
        dataset_id,
    )

    # ----------------------------------------------------------------------------------------------
    # Confirm reload identity
    # ----------------------------------------------------------------------------------------------

    assert (
        reloaded_model_bundle["dataset_id"]
        == dataset_id
    )

    assert (
        reloaded_checkpoint_bundle["dataset_id"]
        == dataset_id
    )

    assert (
        reloaded_checkpoint_bundle[
            "completed_epochs"
        ]
        == completed_epochs
    )

    assert (
        reloaded_checkpoint_bundle[
            "secure_mode"
        ]
        == secure_mode
    )

    assert (
        reloaded_checkpoint_bundle[
            "privacy_boundary"
        ]
        == "DP-SGD discriminator training"
    )

    assert (
        reloaded_checkpoint_bundle[
            "end_to_end_dp"
        ]
        is False
    )

    # ----------------------------------------------------------------------------------------------
    # Registry entries
    # ----------------------------------------------------------------------------------------------

    MODEL_REGISTRY.append({
        "dataset_id": dataset_id,
        "model_path": str(model_path),
        "model_size_bytes": model_size,
        "model_sha256": model_hash,
        "secure_mode": secure_mode,
        "completed_epochs": completed_epochs,
        "target_epochs": target_epochs,
        "run_status": run_status,
        "reload_validation": "PASS",
        "status": "PASS",
    })

    CHECKPOINT_REGISTRY.append({
        "dataset_id": dataset_id,
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_size_bytes": checkpoint_size,
        "checkpoint_sha256": checkpoint_hash,
        "secure_mode": secure_mode,
        "completed_epochs": completed_epochs,
        "target_epochs": target_epochs,
        "run_status": run_status,
        "reload_validation": "PASS",
        "privacy_accountant_saved": True,
        "status": "PASS",
    })

    print(
        f"{dataset_id:<18} | "
        f"model={model_size:,} bytes | "
        f"checkpoint={checkpoint_size:,} bytes | "
        f"secure_mode={secure_mode} | "
        f"epochs={completed_epochs}/{target_epochs} | "
        f"{run_status}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Create Registry DataFrames
# --------------------------------------------------------------------------------------------------

MODEL_REGISTRY_DF = pd.DataFrame(
    MODEL_REGISTRY
)

CHECKPOINT_REGISTRY_DF = pd.DataFrame(
    CHECKPOINT_REGISTRY
)


# --------------------------------------------------------------------------------------------------
# 9. Persist Registries
# --------------------------------------------------------------------------------------------------

model_registry_path = (
    NB07_MODEL_ROOT
    / "dp_ctgan_model_registry.csv"
)

checkpoint_registry_path = (
    NB07_CHECKPOINT_ROOT
    / "dp_ctgan_checkpoint_registry.csv"
)

MODEL_REGISTRY_DF.to_csv(
    model_registry_path,
    index=False,
)

CHECKPOINT_REGISTRY_DF.to_csv(
    checkpoint_registry_path,
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 10. Registry Validation
# --------------------------------------------------------------------------------------------------

assert len(
    MODEL_REGISTRY_DF
) == len(DATASET_IDS), (
    "Model registry does not contain all datasets."
)

assert len(
    CHECKPOINT_REGISTRY_DF
) == len(DATASET_IDS), (
    "Checkpoint registry does not contain all datasets."
)

assert (
    MODEL_REGISTRY_DF["status"] == "PASS"
).all(), (
    "One or more model artifacts failed validation."
)

assert (
    CHECKPOINT_REGISTRY_DF["status"] == "PASS"
).all(), (
    "One or more checkpoint artifacts failed validation."
)

assert (
    MODEL_REGISTRY_DF["reload_validation"] == "PASS"
).all(), (
    "One or more model artifacts failed reload validation."
)

assert (
    CHECKPOINT_REGISTRY_DF["reload_validation"] == "PASS"
).all(), (
    "One or more checkpoints failed reload validation."
)


# --------------------------------------------------------------------------------------------------
# 11. Display Registries
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("MODEL REGISTRY")
print("-" * 100)

display(
    MODEL_REGISTRY_DF
)

print("\n" + "-" * 100)
print("CHECKPOINT REGISTRY")
print("-" * 100)

display(
    CHECKPOINT_REGISTRY_DF
)


# --------------------------------------------------------------------------------------------------
# 12. Final Artifact Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("SECTION 18 ARTIFACT SUMMARY")
print("-" * 100)

print(
    f"Datasets                  : {len(DATASET_IDS)}"
)

print(
    f"Secure mode               : {DP_CONFIG['secure_mode']}"
)

print(
    f"Model artifacts validated : "
    f"{len(MODEL_REGISTRY_DF)}"
)

print(
    f"Checkpoints validated     : "
    f"{len(CHECKPOINT_REGISTRY_DF)}"
)

print(
    f"Model registry            : "
    f"{model_registry_path}"
)

print(
    f"Checkpoint registry       : "
    f"{checkpoint_registry_path}"
)

print(
    "\nPrivacy boundary          : "
    "DP-SGD discriminator training"
)

print(
    "End-to-end DP claimed     : False"
)


# --------------------------------------------------------------------------------------------------
# 13. Final Section Status
# --------------------------------------------------------------------------------------------------
#
# Artifact saving itself is successful for both diagnostic and final runs.
# Therefore the section can pass artifact-integrity validation even when the
# current run is diagnostic-only.
#
# The run_status field separately distinguishes:
#   FINAL_CANDIDATE
#   DIAGNOSTIC_ONLY
# --------------------------------------------------------------------------------------------------

print("\n✓ SECTION 18 — PASS")

18. SAVE MODEL / CHECKPOINTS

----------------------------------------------------------------------------------------------------
SAVING ARTIFACTS — adult_income
----------------------------------------------------------------------------------------------------
adult_income       | model=4,792,805 bytes | checkpoint=8,350,753 bytes | secure_mode=False | epochs=300/300 | DIAGNOSTIC_ONLY

----------------------------------------------------------------------------------------------------
SAVING ARTIFACTS — bank_marketing
----------------------------------------------------------------------------------------------------
bank_marketing     | model=4,511,717 bytes | checkpoint=7,307,809 bytes | secure_mode=False | epochs=300/300 | DIAGNOSTIC_ONLY

----------------------------------------------------------------------------------------------------
SAVING ARTIFACTS — diabetes_130us
----------------------------------------------------------------------------------------------------
diabetes

,dataset_id,model_path,model_size_bytes,model_sha256,secure_mode,completed_epochs,target_epochs,run_status,reload_validation,status
0,adult_income,/content/drive/MyDrive/SPP_GAN_Research/result...,4792805,5b9ace6318526e29e8da91ad3d259ec3570eee7f74303f...,False,300,300,DIAGNOSTIC_ONLY,PASS,PASS
1,bank_marketing,/content/drive/MyDrive/SPP_GAN_Research/result...,4511717,03a9f686708b3dab0c4168778109606c12b4a06cbd84d3...,False,300,300,DIAGNOSTIC_ONLY,PASS,PASS
2,diabetes_130us,/content/drive/MyDrive/SPP_GAN_Research/result...,64680933,4664cffb9efd88b5b5998d76a7834ef0149fca5ef151fc...,False,300,300,DIAGNOSTIC_ONLY,PASS,PASS



----------------------------------------------------------------------------------------------------
CHECKPOINT REGISTRY
----------------------------------------------------------------------------------------------------


,dataset_id,checkpoint_path,checkpoint_size_bytes,checkpoint_sha256,secure_mode,completed_epochs,target_epochs,run_status,reload_validation,privacy_accountant_saved,status
0,adult_income,/content/drive/MyDrive/SPP_GAN_Research/result...,8350753,7fd961824dfb7ffcafbad7f205eb5fb6ab5e7b26e11124...,False,300,300,DIAGNOSTIC_ONLY,PASS,True,PASS
1,bank_marketing,/content/drive/MyDrive/SPP_GAN_Research/result...,7307809,edd6db50789a94e89cba686f54e6226356ad8506d2aa01...,False,300,300,DIAGNOSTIC_ONLY,PASS,True,PASS
2,diabetes_130us,/content/drive/MyDrive/SPP_GAN_Research/result...,143228321,b1c6b078d8bd401af42e3aec82e6e6c6ee44feae5f990d...,False,300,300,DIAGNOSTIC_ONLY,PASS,True,PASS



----------------------------------------------------------------------------------------------------
SECTION 18 ARTIFACT SUMMARY
----------------------------------------------------------------------------------------------------
Datasets                  : 3
Secure mode               : False
Model artifacts validated : 3
Checkpoints validated     : 3
Model registry            : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/models/dp_ctgan_model_registry.csv
Checkpoint registry       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_07/checkpoints/dp_ctgan_checkpoint_registry.csv

Privacy boundary          : DP-SGD discriminator training
End-to-end DP claimed     : False

✓ SECTION 18 — PASS


In [23]:
# ==================================================================================================
# 19. SAVE PRIVACY METADATA
# ==================================================================================================

print("=" * 100)
print("19. SAVE PRIVACY METADATA")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Validate Required Objects
# --------------------------------------------------------------------------------------------------

assert "TRAINING_RESULTS_DF" in globals(), (
    "TRAINING_RESULTS_DF is not available."
)

assert "ACTUAL_PRIVACY_DF" in globals(), (
    "ACTUAL_PRIVACY_DF is not available. "
    "Run Section 14 first."
)

assert "DP_CONFIG" in globals(), (
    "DP_CONFIG is not available."
)

assert "DP_CTGAN_CONFIG" in globals(), (
    "DP_CTGAN_CONFIG is not available."
)

assert "PRIVACY_PARAMETERS" in globals(), (
    "PRIVACY_PARAMETERS is not available."
)

assert "DATASET_IDS" in globals(), (
    "DATASET_IDS is not available."
)


# --------------------------------------------------------------------------------------------------
# 2. Reset Metadata Container
# --------------------------------------------------------------------------------------------------

PRIVACY_METADATA = {}


# --------------------------------------------------------------------------------------------------
# 3. Validate ACTUAL_PRIVACY_DF
# --------------------------------------------------------------------------------------------------

required_privacy_columns = [
    "dataset_id",
    "target_epsilon",
    "actual_epsilon",
    "target_delta",
    "noise_multiplier",
    "accountant",
    "max_grad_norm",
    "sample_rate",
]

missing_privacy_columns = [
    column
    for column in required_privacy_columns
    if column not in ACTUAL_PRIVACY_DF.columns
]

assert not missing_privacy_columns, (
    "ACTUAL_PRIVACY_DF is missing required columns: "
    f"{missing_privacy_columns}"
)


# --------------------------------------------------------------------------------------------------
# 4. Generate Dataset-Level Privacy Metadata
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    # ----------------------------------------------------------------------------------------------
    # Training result
    # ----------------------------------------------------------------------------------------------

    training_matches = TRAINING_RESULTS_DF[
        TRAINING_RESULTS_DF["dataset_id"] == dataset_id
    ]

    assert len(training_matches) == 1, (
        f"{dataset_id}: expected exactly one training-result record, "
        f"found {len(training_matches)}."
    )

    training_result = (
        training_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy result
    # ----------------------------------------------------------------------------------------------

    privacy_matches = ACTUAL_PRIVACY_DF[
        ACTUAL_PRIVACY_DF["dataset_id"] == dataset_id
    ]

    assert len(privacy_matches) == 1, (
        f"{dataset_id}: expected exactly one privacy record, "
        f"found {len(privacy_matches)}."
    )

    privacy_result = (
        privacy_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Privacy configuration
    # ----------------------------------------------------------------------------------------------

    params = PRIVACY_PARAMETERS[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Retrieve clipping configuration
    #
    # ACTUAL_PRIVACY_DF does not contain a "clipping" column.
    # The authoritative clipping configuration is therefore retrieved
    # from PRIVACY_PARAMETERS.
    # ----------------------------------------------------------------------------------------------

    clipping = params.get(
        "clipping",
        "flat",
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve Poisson configuration
    # ----------------------------------------------------------------------------------------------

    poisson_sampling = bool(
        params.get(
            "poisson_sampling",
            DP_CONFIG.get(
                "poisson_sampling",
                True,
            ),
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Secure-mode configuration
    #
    # IMPORTANT:
    # Do not hard-code this value.
    # ----------------------------------------------------------------------------------------------

    secure_mode = bool(
        DP_CONFIG["secure_mode"]
    )

    # ----------------------------------------------------------------------------------------------
    # Explicit privacy boundary
    # ----------------------------------------------------------------------------------------------

    privacy_boundary = (
        "DP-SGD discriminator training"
    )

    end_to_end_dp = False

    # ----------------------------------------------------------------------------------------------
    # Run classification
    # ----------------------------------------------------------------------------------------------

    completed_epochs = training_result.get(
        "completed_epochs",
        training_result.get(
            "epochs_completed",
            DP_CTGAN_CONFIG["epochs"],
        ),
    )

    completed_epochs = int(
        completed_epochs
    )

    target_epochs = int(
        DP_CTGAN_CONFIG["epochs"]
    )

    training_complete = (
        completed_epochs >= target_epochs
    )

    if secure_mode and training_complete:
        run_status = "FINAL_CANDIDATE"
    else:
        run_status = "DIAGNOSTIC_ONLY"

    # ----------------------------------------------------------------------------------------------
    # Build metadata
    # ----------------------------------------------------------------------------------------------

    metadata = {

        # ------------------------------------------------------------------------------------------
        # Notebook identification
        # ------------------------------------------------------------------------------------------

        "notebook_id": NOTEBOOK_ID,

        "notebook_name": NOTEBOOK_NAME,

        "notebook_version": NOTEBOOK_VERSION,

        # ------------------------------------------------------------------------------------------
        # Dataset
        # ------------------------------------------------------------------------------------------

        "dataset_id": dataset_id,

        # ------------------------------------------------------------------------------------------
        # Model configuration
        # ------------------------------------------------------------------------------------------

        "model": {

            "name": "DP-CTGAN",

            "architecture": "CTGAN",

            "pac": int(
                DP_CTGAN_CONFIG["pac"]
            ),

            "embedding_dim": int(
                DP_CTGAN_CONFIG["embedding_dim"]
            ),

            "generator_dim": [
                int(value)
                for value
                in DP_CTGAN_CONFIG["generator_dim"]
            ],

            "discriminator_dim": [
                int(value)
                for value
                in DP_CTGAN_CONFIG["discriminator_dim"]
            ],

            "epochs": target_epochs,

            "completed_epochs": completed_epochs,

            "batch_size": int(
                DP_CTGAN_CONFIG["batch_size"]
            ),
        },

        # ------------------------------------------------------------------------------------------
        # Privacy configuration
        # ------------------------------------------------------------------------------------------

        "privacy": {

            "algorithm": "DP-SGD",

            "accountant": str(
                privacy_result["accountant"]
            ),

            "target_epsilon": float(
                privacy_result["target_epsilon"]
            ),

            "actual_epsilon": float(
                privacy_result["actual_epsilon"]
            ),

            "target_delta": float(
                privacy_result["target_delta"]
            ),

            "noise_multiplier": float(
                privacy_result["noise_multiplier"]
            ),

            "max_grad_norm": float(
                privacy_result["max_grad_norm"]
            ),

            "clipping": clipping,

            "sample_rate": float(
                privacy_result["sample_rate"]
            ),

            "poisson_sampling": poisson_sampling,

            "secure_mode": secure_mode,

            "privacy_boundary": privacy_boundary,

            "end_to_end_dp": end_to_end_dp,
        },

        # ------------------------------------------------------------------------------------------
        # Experimental policy
        # ------------------------------------------------------------------------------------------

        "policy": {

            "train_only": True,

            "validation_training": False,

            "test_training": False,

            "statistical_guidance": False,

            "spp_gan_components": False,

            "provenance_excluded": True,

            "identifiers_excluded": True,

            "target_retained": True,
        },

        # ------------------------------------------------------------------------------------------
        # Run status
        # ------------------------------------------------------------------------------------------

        "run": {

            "secure_mode": secure_mode,

            "completed_epochs": completed_epochs,

            "target_epochs": target_epochs,

            "training_complete": training_complete,

            "run_status": run_status,
        },

        # ------------------------------------------------------------------------------------------
        # Reproducibility
        # ------------------------------------------------------------------------------------------

        "seed": int(
            DATASET_SEEDS[dataset_id]
        ),

        "runtime_seconds": float(
            training_result["runtime_seconds"]
        ),

        "timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    # ----------------------------------------------------------------------------------------------
    # Store metadata in memory
    # ----------------------------------------------------------------------------------------------

    PRIVACY_METADATA[
        dataset_id
    ] = metadata

    # ----------------------------------------------------------------------------------------------
    # Metadata output path
    # ----------------------------------------------------------------------------------------------

    metadata_path = (
        NB07_PRIVACY_ROOT
        / dataset_id
        / "dp_ctgan_privacy_metadata.json"
    )

    metadata_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ----------------------------------------------------------------------------------------------
    # Save JSON
    # ----------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            metadata,
            file,
            indent=2,
        )

    # ----------------------------------------------------------------------------------------------
    # Reload and validate metadata
    # ----------------------------------------------------------------------------------------------

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as file:

        reloaded_metadata = json.load(
            file
        )

    assert (
        reloaded_metadata["dataset_id"]
        == dataset_id
    )

    assert (
        reloaded_metadata["privacy"]["algorithm"]
        == "DP-SGD"
    )

    assert (
        reloaded_metadata["privacy"]["accountant"]
        == str(privacy_result["accountant"])
    )

    assert (
        reloaded_metadata["privacy"]["secure_mode"]
        == secure_mode
    )

    assert (
        reloaded_metadata["privacy"]["privacy_boundary"]
        == "DP-SGD discriminator training"
    )

    assert (
        reloaded_metadata["privacy"]["end_to_end_dp"]
        is False
    )

    assert (
        reloaded_metadata["run"]["completed_epochs"]
        == completed_epochs
    )

    print(
        f"{dataset_id:<18} | "
        f"ε={privacy_result['actual_epsilon']:.6f} | "
        f"δ={privacy_result['target_delta']:.2e} | "
        f"secure_mode={secure_mode} | "
        f"epochs={completed_epochs}/{target_epochs} | "
        f"{run_status} | PASS"
    )


# --------------------------------------------------------------------------------------------------
# 5. Final Validation
# --------------------------------------------------------------------------------------------------

assert len(
    PRIVACY_METADATA
) == len(DATASET_IDS), (
    "Privacy metadata was not generated for all datasets."
)

for dataset_id in DATASET_IDS:

    assert dataset_id in PRIVACY_METADATA

    metadata = PRIVACY_METADATA[
        dataset_id
    ]

    assert (
        metadata["privacy"]["actual_epsilon"]
        <= metadata["privacy"]["target_epsilon"] + 0.05
    ), (
        f"{dataset_id}: achieved epsilon exceeds "
        "the configured tolerance."
    )

    assert (
        metadata["privacy"]["end_to_end_dp"]
        is False
    ), (
        f"{dataset_id}: end-to-end DP must not be claimed "
        "for the current DP-CTGAN privacy boundary."
    )


# --------------------------------------------------------------------------------------------------
# 6. Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY METADATA SUMMARY")
print("-" * 100)

print(
    f"Datasets                  : {len(PRIVACY_METADATA)}"
)

print(
    f"Secure mode               : "
    f"{DP_CONFIG['secure_mode']}"
)

print(
    "Privacy boundary          : "
    "DP-SGD discriminator training"
)

print(
    "End-to-end DP claimed     : False"
)

print(
    "Metadata files saved      : "
    f"{len(PRIVACY_METADATA)}"
)

print(
    "\n✓ Privacy metadata saved and reload-validated "
    "for all datasets."
)

print("\n✓ SECTION 19 — PASS")

19. SAVE PRIVACY METADATA
adult_income       | ε=4.999028 | δ=1.00e-05 | secure_mode=False | epochs=300/300 | DIAGNOSTIC_ONLY | PASS
bank_marketing     | ε=4.999538 | δ=1.00e-05 | secure_mode=False | epochs=300/300 | DIAGNOSTIC_ONLY | PASS
diabetes_130us     | ε=4.999684 | δ=1.00e-05 | secure_mode=False | epochs=300/300 | DIAGNOSTIC_ONLY | PASS

----------------------------------------------------------------------------------------------------
PRIVACY METADATA SUMMARY
----------------------------------------------------------------------------------------------------
Datasets                  : 3
Secure mode               : False
Privacy boundary          : DP-SGD discriminator training
End-to-end DP claimed     : False
Metadata files saved      : 3

✓ Privacy metadata saved and reload-validated for all datasets.

✓ SECTION 19 — PASS


In [24]:
# ==================================================================================================
# 20. SAVE SYNTHETIC DATA
# ==================================================================================================

print("=" * 100)
print("20. SAVE SYNTHETIC DATA")
print("=" * 100)

SYNTHETIC_REGISTRY = []

for dataset_id in DATASET_IDS:

    synthetic_df = DPCTGAN_SYNTHETIC_DATA[
        dataset_id
    ]

    output_dir = (
        NB07_SYNTHETIC_ROOT
        / dataset_id
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = (
        output_dir
        / "dp_ctgan_synthetic.csv"
    )

    synthetic_df.to_csv(
        output_path,
        index=False,
    )

    assert output_path.exists()
    assert output_path.stat().st_size > 0

    # Reload validation
    reloaded_df = pd.read_csv(
        output_path,
        low_memory=False,
    )

    assert len(reloaded_df) == len(
        synthetic_df
    )

    assert list(
        reloaded_df.columns
    ) == list(
        synthetic_df.columns
    )

    def dataframe_hash(df):

        values = pd.util.hash_pandas_object(
            df,
            index=True,
        ).values

        return hashlib.sha256(
            values.tobytes()
        ).hexdigest()

    source_hash = dataframe_hash(
        synthetic_df
    )

    reloaded_hash = dataframe_hash(
        reloaded_df
    )

    assert source_hash == reloaded_hash

    file_hash = hashlib.sha256(
        output_path.read_bytes()
    ).hexdigest()

    SYNTHETIC_REGISTRY.append({
        "dataset_id": dataset_id,
        "synthetic_path": str(output_path),
        "rows": len(reloaded_df),
        "columns": len(reloaded_df.columns),
        "source_content_hash": source_hash,
        "reloaded_content_hash": reloaded_hash,
        "file_sha256": file_hash,
        "file_size_bytes": output_path.stat().st_size,
        "status": "PASS",
    })

    del reloaded_df
    gc.collect()

SYNTHETIC_REGISTRY_DF = pd.DataFrame(
    SYNTHETIC_REGISTRY
)

synthetic_registry_path = (
    NB07_SYNTHETIC_ROOT
    / "dp_ctgan_synthetic_registry.csv"
)

SYNTHETIC_REGISTRY_DF.to_csv(
    synthetic_registry_path,
    index=False,
)

display(SYNTHETIC_REGISTRY_DF)

assert (
    SYNTHETIC_REGISTRY_DF["status"] == "PASS"
).all()

print("\n✓ SECTION 20 — PASS")

20. SAVE SYNTHETIC DATA


,dataset_id,synthetic_path,rows,columns,source_content_hash,reloaded_content_hash,file_sha256,file_size_bytes,status
0,adult_income,/content/drive/MyDrive/SPP_GAN_Research/result...,34189,15,6889d82c6683dbb36807e8eae0beaf270e695adb9e72c4...,6889d82c6683dbb36807e8eae0beaf270e695adb9e72c4...,ec8a25acf389efaa602ece0c704ef00483542573a3c680...,3648370,PASS
1,bank_marketing,/content/drive/MyDrive/SPP_GAN_Research/result...,31647,17,a97a5150006c4df2e0821131f6f72de773c6294f282e10...,a97a5150006c4df2e0821131f6f72de773c6294f282e10...,5a8ca6be1ca3a48f7256e7b2329b5e0613abc944bc9da2...,2623208,PASS
2,diabetes_130us,/content/drive/MyDrive/SPP_GAN_Research/result...,71236,48,1fa717bab9b16e764721685838ec46c2de96051652f52d...,1fa717bab9b16e764721685838ec46c2de96051652f52d...,3d30d8f20ec28b2e010bd3a8b995ebd634cb2684873a6c...,11503661,PASS



✓ SECTION 20 — PASS


In [26]:
# ==================================================================================================
# 21. SAVE MANIFEST
# ==================================================================================================

print("=" * 100)
print("21. SAVE MANIFEST")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# Purpose
# --------------------------------------------------------------------------------------------------
# Create publication-grade dataset-level and master manifests for the DP-CTGAN
# baseline.
#
# The manifest consolidates:
#   - notebook identity and version
#   - dataset/training schema information
#   - model configuration
#   - privacy configuration and achieved privacy budget
#   - privacy protection boundary
#   - random seed
#   - model artifact registry
#   - checkpoint registry
#   - synthetic-data registry
#   - software dependencies
#   - run status
#
# IMPORTANT:
# "status": "PASS" means the manifest/artifact bookkeeping succeeded.
#
# "run_status" separately identifies whether the underlying experiment is:
#   - DIAGNOSTIC_ONLY
#   - FINAL_CANDIDATE
#
# This prevents a successful manifest from being interpreted as proof that
# the current experiment is the final publication run.
# --------------------------------------------------------------------------------------------------


# --------------------------------------------------------------------------------------------------
# 1. Validate Required Objects
# --------------------------------------------------------------------------------------------------

assert "DATASET_IDS" in globals(), (
    "DATASET_IDS is not available."
)

assert "TRAINING_DATA" in globals(), (
    "TRAINING_DATA is not available."
)

assert "TARGET_COLUMNS" in globals(), (
    "TARGET_COLUMNS is not available."
)

assert "IDENTIFIER_COLUMNS" in globals(), (
    "IDENTIFIER_COLUMNS is not available."
)

assert "PROVENANCE_COLUMN" in globals(), (
    "PROVENANCE_COLUMN is not available."
)

assert "DP_CTGAN_CONFIG" in globals(), (
    "DP_CTGAN_CONFIG is not available."
)

assert "DP_CONFIG" in globals(), (
    "DP_CONFIG is not available."
)

assert "MODEL_REGISTRY_DF" in globals(), (
    "MODEL_REGISTRY_DF is not available. "
    "Run Section 18 first."
)

assert "CHECKPOINT_REGISTRY_DF" in globals(), (
    "CHECKPOINT_REGISTRY_DF is not available. "
    "Run Section 18 first."
)

assert "SYNTHETIC_REGISTRY_DF" in globals(), (
    "SYNTHETIC_REGISTRY_DF is not available. "
    "Run Section 20 first."
)

assert "ACTUAL_PRIVACY_DF" in globals(), (
    "ACTUAL_PRIVACY_DF is not available. "
    "Run Section 14 first."
)

assert "DATASET_SEEDS" in globals(), (
    "DATASET_SEEDS is not available."
)

assert "PRIVACY_METADATA" in globals(), (
    "PRIVACY_METADATA is not available. "
    "Run Section 19 first."
)


# --------------------------------------------------------------------------------------------------
# 2. Reset Manifest Registry
# --------------------------------------------------------------------------------------------------

MANIFEST_RECORDS = []


# --------------------------------------------------------------------------------------------------
# 3. Current Run Configuration
# --------------------------------------------------------------------------------------------------

CURRENT_SECURE_MODE = bool(
    DP_CONFIG["secure_mode"]
)


# --------------------------------------------------------------------------------------------------
# 4. Build Dataset-Level Manifests
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"CREATING MANIFEST — {dataset_id}")
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Retrieve model registry record
    # ----------------------------------------------------------------------------------------------

    model_matches = MODEL_REGISTRY_DF[
        MODEL_REGISTRY_DF["dataset_id"] == dataset_id
    ]

    assert len(model_matches) == 1, (
        f"{dataset_id}: expected exactly one model registry record, "
        f"found {len(model_matches)}."
    )

    model_record = (
        model_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve checkpoint registry record
    # ----------------------------------------------------------------------------------------------

    checkpoint_matches = CHECKPOINT_REGISTRY_DF[
        CHECKPOINT_REGISTRY_DF["dataset_id"] == dataset_id
    ]

    assert len(checkpoint_matches) == 1, (
        f"{dataset_id}: expected exactly one checkpoint registry record, "
        f"found {len(checkpoint_matches)}."
    )

    checkpoint_record = (
        checkpoint_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve synthetic-data registry record
    # ----------------------------------------------------------------------------------------------

    synthetic_matches = SYNTHETIC_REGISTRY_DF[
        SYNTHETIC_REGISTRY_DF["dataset_id"] == dataset_id
    ]

    assert len(synthetic_matches) == 1, (
        f"{dataset_id}: expected exactly one synthetic registry record, "
        f"found {len(synthetic_matches)}."
    )

    synthetic_record = (
        synthetic_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve privacy record
    # ----------------------------------------------------------------------------------------------

    privacy_matches = ACTUAL_PRIVACY_DF[
        ACTUAL_PRIVACY_DF["dataset_id"] == dataset_id
    ]

    assert len(privacy_matches) == 1, (
        f"{dataset_id}: expected exactly one privacy record, "
        f"found {len(privacy_matches)}."
    )

    privacy_record = (
        privacy_matches
        .iloc[0]
        .to_dict()
    )

    # ----------------------------------------------------------------------------------------------
    # Retrieve Section 19 metadata
    # ----------------------------------------------------------------------------------------------

    assert dataset_id in PRIVACY_METADATA, (
        f"{dataset_id}: Section 19 privacy metadata is missing."
    )

    privacy_metadata = PRIVACY_METADATA[
        dataset_id
    ]

    # ----------------------------------------------------------------------------------------------
    # Validate artifact registry records
    # ----------------------------------------------------------------------------------------------

    assert model_record["status"] == "PASS", (
        f"{dataset_id}: model artifact registry is not PASS."
    )

    assert model_record["reload_validation"] == "PASS", (
        f"{dataset_id}: model reload validation failed."
    )

    assert checkpoint_record["status"] == "PASS", (
        f"{dataset_id}: checkpoint artifact registry is not PASS."
    )

    assert checkpoint_record["reload_validation"] == "PASS", (
        f"{dataset_id}: checkpoint reload validation failed."
    )

    assert checkpoint_record["privacy_accountant_saved"], (
        f"{dataset_id}: privacy accountant was not saved."
    )

    assert synthetic_record["status"] == "PASS", (
        f"{dataset_id}: synthetic artifact registry is not PASS."
    )

    assert (
        synthetic_record["source_content_hash"]
        == synthetic_record["reloaded_content_hash"]
    ), (
        f"{dataset_id}: synthetic source/reloaded content hashes differ."
    )

    # ----------------------------------------------------------------------------------------------
    # Validate secure-mode consistency
    # ----------------------------------------------------------------------------------------------

    model_secure_mode = bool(
        model_record["secure_mode"]
    )

    checkpoint_secure_mode = bool(
        checkpoint_record["secure_mode"]
    )

    metadata_secure_mode = bool(
        privacy_metadata["privacy"]["secure_mode"]
    )

    assert model_secure_mode == CURRENT_SECURE_MODE, (
        f"{dataset_id}: model secure-mode state does not match "
        "current DP configuration."
    )

    assert checkpoint_secure_mode == CURRENT_SECURE_MODE, (
        f"{dataset_id}: checkpoint secure-mode state does not match "
        "current DP configuration."
    )

    assert metadata_secure_mode == CURRENT_SECURE_MODE, (
        f"{dataset_id}: privacy metadata secure-mode state does not match "
        "current DP configuration."
    )

    # ----------------------------------------------------------------------------------------------
    # Determine training completion
    # ----------------------------------------------------------------------------------------------

    completed_epochs = int(
        model_record["completed_epochs"]
    )

    target_epochs = int(
        model_record["target_epochs"]
    )

    training_complete = (
        completed_epochs >= target_epochs
    )

    # ----------------------------------------------------------------------------------------------
    # Determine run status
    # ----------------------------------------------------------------------------------------------

    if (
        CURRENT_SECURE_MODE
        and training_complete
    ):

        run_status = "FINAL_CANDIDATE"

    else:

        run_status = "DIAGNOSTIC_ONLY"

    # ----------------------------------------------------------------------------------------------
    # Explicit privacy boundary
    # ----------------------------------------------------------------------------------------------

    privacy_boundary = (
        "DP-SGD discriminator training"
    )

    end_to_end_dp = False

    # ----------------------------------------------------------------------------------------------
    # Construct manifest
    # ----------------------------------------------------------------------------------------------

    manifest = {

        # ==========================================================================================
        # Notebook identity
        # ==========================================================================================

        "notebook_id": NOTEBOOK_ID,

        "notebook_name": NOTEBOOK_NAME,

        "notebook_version": NOTEBOOK_VERSION,

        # ==========================================================================================
        # Dataset identity
        # ==========================================================================================

        "dataset_id": dataset_id,

        "project_root": str(
            PROJECT_ROOT
        ),

        # ==========================================================================================
        # Training data
        # ==========================================================================================

        "training": {

            "rows": len(
                TRAINING_DATA[dataset_id]
            ),

            "columns": len(
                TRAINING_DATA[dataset_id].columns
            ),

            "target": TARGET_COLUMNS[
                dataset_id
            ],

            "identifiers": list(
                IDENTIFIER_COLUMNS[
                    dataset_id
                ]
            ),

            "provenance_column": PROVENANCE_COLUMN,

            "train_only": True,
        },

        # ==========================================================================================
        # Model configuration
        # ==========================================================================================

        "model": DP_CTGAN_CONFIG,

        # ==========================================================================================
        # Privacy information
        # ==========================================================================================

        "privacy": {

            # Accountant values
            "algorithm": str(
                privacy_record["dp_algorithm"]
                if "dp_algorithm" in privacy_record
                else "DP-SGD"
            ),

            "accountant": str(
                privacy_record["accountant"]
            ),

            "target_epsilon": float(
                privacy_record["target_epsilon"]
            ),

            "actual_epsilon": float(
                privacy_record["actual_epsilon"]
            ),

            "target_delta": float(
                privacy_record["target_delta"]
            ),

            "noise_multiplier": float(
                privacy_record["noise_multiplier"]
            ),

            "max_grad_norm": float(
                privacy_record["max_grad_norm"]
            ),

            "clipping": privacy_metadata[
                "privacy"
            ]["clipping"],

            "sample_rate": float(
                privacy_record["sample_rate"]
            ),

            "poisson_sampling": bool(
                privacy_metadata[
                    "privacy"
                ]["poisson_sampling"]
            ),

            # Secure execution
            "secure_mode": CURRENT_SECURE_MODE,

            # Explicit protection boundary
            "privacy_boundary": privacy_boundary,

            "end_to_end_dp": end_to_end_dp,
        },

        # ==========================================================================================
        # Experimental policy
        # ==========================================================================================

        "policy": {

            "train_only": True,

            "validation_training": False,

            "test_training": False,

            "statistical_guidance": False,

            "spp_gan_components": False,

            "provenance_excluded": True,

            "identifiers_excluded": True,

            "target_retained": True,
        },

        # ==========================================================================================
        # Reproducibility
        # ==========================================================================================

        "seed": int(
            DATASET_SEEDS[
                dataset_id
            ]
        ),

        # ==========================================================================================
        # Run state
        # ==========================================================================================

        "run": {

            "secure_mode": CURRENT_SECURE_MODE,

            "completed_epochs": completed_epochs,

            "target_epochs": target_epochs,

            "training_complete": training_complete,

            "run_status": run_status,
        },

        # ==========================================================================================
        # Artifact registries
        # ==========================================================================================

        "artifacts": {

            "model": model_record,

            "checkpoint": checkpoint_record,

            "synthetic": synthetic_record,
        },

        # ==========================================================================================
        # Software dependencies
        # ==========================================================================================

        "dependencies": {

            "sdv_version": sdv.__version__,

            "opacus_version": opacus.__version__,

            "pytorch_version": torch.__version__,
        },

        # ==========================================================================================
        # Manifest status
        # ==========================================================================================

        "status": "PASS",
    }

    # ----------------------------------------------------------------------------------------------
    # Save dataset manifest
    # ----------------------------------------------------------------------------------------------

    manifest_path = (
        NB07_MANIFEST_ROOT
        / dataset_id
        / "dp_ctgan_manifest.json"
    )

    manifest_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        manifest_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            manifest,
            file,
            indent=2,
            default=str,
        )

    # ----------------------------------------------------------------------------------------------
    # Verify manifest exists and is non-empty
    # ----------------------------------------------------------------------------------------------

    assert manifest_path.exists(), (
        f"{dataset_id}: manifest was not created."
    )

    assert manifest_path.stat().st_size > 0, (
        f"{dataset_id}: manifest is empty."
    )

    # ----------------------------------------------------------------------------------------------
    # Reload manifest
    # ----------------------------------------------------------------------------------------------

    with open(
        manifest_path,
        "r",
        encoding="utf-8",
    ) as file:

        reloaded_manifest = json.load(
            file
        )

    # ----------------------------------------------------------------------------------------------
    # Reload validation
    # ----------------------------------------------------------------------------------------------

    assert (
        reloaded_manifest["dataset_id"]
        == dataset_id
    )

    assert (
        reloaded_manifest["notebook_id"]
        == NOTEBOOK_ID
    )

    assert (
        reloaded_manifest["status"]
        == "PASS"
    )

    assert (
        reloaded_manifest["run"]["secure_mode"]
        == CURRENT_SECURE_MODE
    )

    assert (
        reloaded_manifest["privacy"]["secure_mode"]
        == CURRENT_SECURE_MODE
    )

    assert (
        reloaded_manifest["privacy"]["privacy_boundary"]
        == privacy_boundary
    )

    assert (
        reloaded_manifest["privacy"]["end_to_end_dp"]
        is False
    )

    assert (
        reloaded_manifest["run"]["completed_epochs"]
        == completed_epochs
    )

    assert (
        reloaded_manifest["run"]["target_epochs"]
        == target_epochs
    )

    assert (
        reloaded_manifest["artifacts"]["model"]["status"]
        == "PASS"
    )

    assert (
        reloaded_manifest["artifacts"]["checkpoint"]["status"]
        == "PASS"
    )

    assert (
        reloaded_manifest["artifacts"]["synthetic"]["status"]
        == "PASS"
    )

    # ----------------------------------------------------------------------------------------------
    # Record manifest
    # ----------------------------------------------------------------------------------------------

    MANIFEST_RECORDS.append(
        reloaded_manifest
    )

    print(
        f"{dataset_id:<18} | "
        f"epochs={completed_epochs}/{target_epochs} | "
        f"secure_mode={CURRENT_SECURE_MODE} | "
        f"{run_status} | PASS"
    )


# --------------------------------------------------------------------------------------------------
# 5. Validate Dataset Manifest Coverage
# --------------------------------------------------------------------------------------------------

assert len(
    MANIFEST_RECORDS
) == len(DATASET_IDS), (
    "Manifest count does not match DATASET_IDS."
)

assert {
    record["dataset_id"]
    for record in MANIFEST_RECORDS
} == set(DATASET_IDS), (
    "Manifest dataset coverage does not match DATASET_IDS."
)


# --------------------------------------------------------------------------------------------------
# 6. Create Master Manifest
# --------------------------------------------------------------------------------------------------

master_manifest = {

    "notebook_id": NOTEBOOK_ID,

    "notebook_name": NOTEBOOK_NAME,

    "notebook_version": NOTEBOOK_VERSION,

    "project_root": str(
        PROJECT_ROOT
    ),

    "secure_mode": CURRENT_SECURE_MODE,

    "privacy_boundary": (
        "DP-SGD discriminator training"
    ),

    "end_to_end_dp": False,

    "datasets": MANIFEST_RECORDS,

    "dataset_count": len(
        MANIFEST_RECORDS
    ),

    "status": "PASS",
}


# --------------------------------------------------------------------------------------------------
# 7. Save Master Manifest
# --------------------------------------------------------------------------------------------------

master_manifest_path = (
    NB07_MANIFEST_ROOT
    / "dp_ctgan_manifest.json"
)

master_manifest_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    master_manifest_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        master_manifest,
        file,
        indent=2,
        default=str,
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate Master Manifest
# --------------------------------------------------------------------------------------------------

assert master_manifest_path.exists(), (
    "Master manifest was not created."
)

assert master_manifest_path.stat().st_size > 0, (
    "Master manifest is empty."
)

with open(
    master_manifest_path,
    "r",
    encoding="utf-8",
) as file:

    reloaded_master_manifest = json.load(
        file
    )

assert (
    reloaded_master_manifest["notebook_id"]
    == NOTEBOOK_ID
)

assert (
    reloaded_master_manifest["dataset_count"]
    == len(DATASET_IDS)
)

assert (
    len(
        reloaded_master_manifest["datasets"]
    )
    == len(DATASET_IDS)
)

assert (
    reloaded_master_manifest["secure_mode"]
    == CURRENT_SECURE_MODE
)

assert (
    reloaded_master_manifest["privacy_boundary"]
    == "DP-SGD discriminator training"
)

assert (
    reloaded_master_manifest["end_to_end_dp"]
    is False
)

assert (
    reloaded_master_manifest["status"]
    == "PASS"
)


# --------------------------------------------------------------------------------------------------
# 9. Final Summary
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("MANIFEST SUMMARY")
print("-" * 100)

print(
    f"Dataset manifests      : "
    f"{len(MANIFEST_RECORDS)}"
)

print(
    f"Secure mode             : "
    f"{CURRENT_SECURE_MODE}"
)

print(
    "Privacy boundary        : "
    "DP-SGD discriminator training"
)

print(
    "End-to-end DP claimed   : False"
)

print(
    f"Master manifest         : "
    f"{master_manifest_path}"
)

print(
    "\n✓ Dataset manifests saved and reload-validated."
)

print(
    "✓ Master manifest saved and reload-validated."
)

print("\n✓ SECTION 21 — PASS")

21. SAVE MANIFEST

----------------------------------------------------------------------------------------------------
CREATING MANIFEST — adult_income
----------------------------------------------------------------------------------------------------
adult_income       | epochs=300/300 | secure_mode=False | DIAGNOSTIC_ONLY | PASS

----------------------------------------------------------------------------------------------------
CREATING MANIFEST — bank_marketing
----------------------------------------------------------------------------------------------------
bank_marketing     | epochs=300/300 | secure_mode=False | DIAGNOSTIC_ONLY | PASS

----------------------------------------------------------------------------------------------------
CREATING MANIFEST — diabetes_130us
----------------------------------------------------------------------------------------------------
diabetes_130us     | epochs=300/300 | secure_mode=False | DIAGNOSTIC_ONLY | PASS

--------------------------

In [27]:
# ==================================================================================================
# 22. FINAL PRIVACY VERIFICATION
# ==================================================================================================

print("=" * 100)
print("22. FINAL PRIVACY VERIFICATION")
print("=" * 100)

PRIVACY_VERIFICATION = []

for dataset_id in DATASET_IDS:

    metadata_path = (
        NB07_PRIVACY_ROOT
        / dataset_id
        / "dp_ctgan_privacy_metadata.json"
    )

    manifest_path = (
        NB07_MANIFEST_ROOT
        / dataset_id
        / "dp_ctgan_manifest.json"
    )

    assert metadata_path.exists()
    assert manifest_path.exists()

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as f:

        metadata = json.load(f)

    with open(
        manifest_path,
        "r",
        encoding="utf-8",
    ) as f:

        manifest = json.load(f)

    privacy = metadata["privacy"]

    epsilon_ok = (
        float(privacy["actual_epsilon"])
        <= float(
            privacy["target_epsilon"]
        ) + 0.05
    )

    delta_ok = (
        0 < float(
            privacy["target_delta"]
        ) < 1
    )

    noise_ok = (
        float(
            privacy["noise_multiplier"]
        ) > 0
    )

    clipping_ok = (
        float(
            privacy["max_grad_norm"]
        ) > 0
    )

    poisson_ok = (
        privacy["poisson_sampling"]
        is True
    )

    accountant_ok = (
        privacy["accountant"]
        == "rdp"
    )

    pac_ok = (
        manifest["model"]["pac"]
        == 1
    )

    train_only_ok = (
        metadata["policy"]["train_only"]
        is True
        and metadata["policy"][
            "validation_training"
        ] is False
        and metadata["policy"][
            "test_training"
        ] is False
    )

    provenance_ok = (
        metadata["policy"][
            "provenance_excluded"
        ] is True
    )

    identifiers_ok = (
        metadata["policy"][
            "identifiers_excluded"
        ] is True
    )

    status = (
        "PASS"
        if all([
            epsilon_ok,
            delta_ok,
            noise_ok,
            clipping_ok,
            poisson_ok,
            accountant_ok,
            pac_ok,
            train_only_ok,
            provenance_ok,
            identifiers_ok,
        ])
        else "FAIL"
    )

    PRIVACY_VERIFICATION.append({
        "dataset_id": dataset_id,
        "epsilon_ok": epsilon_ok,
        "delta_ok": delta_ok,
        "noise_ok": noise_ok,
        "clipping_ok": clipping_ok,
        "poisson_sampling_ok": poisson_ok,
        "accountant_ok": accountant_ok,
        "pac_ok": pac_ok,
        "train_only_ok": train_only_ok,
        "provenance_excluded": provenance_ok,
        "identifiers_excluded": identifiers_ok,
        "actual_epsilon": privacy[
            "actual_epsilon"
        ],
        "target_epsilon": privacy[
            "target_epsilon"
        ],
        "target_delta": privacy[
            "target_delta"
        ],
        "noise_multiplier": privacy[
            "noise_multiplier"
        ],
        "status": status,
    })

PRIVACY_VERIFICATION_DF = pd.DataFrame(
    PRIVACY_VERIFICATION
)

verification_csv = (
    NB07_VALIDATION_ROOT
    / "dp_ctgan_privacy_verification.csv"
)

verification_json = (
    NB07_VALIDATION_ROOT
    / "dp_ctgan_privacy_verification.json"
)

PRIVACY_VERIFICATION_DF.to_csv(
    verification_csv,
    index=False,
)

with open(
    verification_json,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PRIVACY_VERIFICATION,
        f,
        indent=2,
        default=str,
    )

display(PRIVACY_VERIFICATION_DF)

assert (
    PRIVACY_VERIFICATION_DF["status"]
    == "PASS"
).all()

print("\n✓ FINAL PRIVACY VERIFICATION — PASS")
print("\n✓ SECTION 22 — PASS")

22. FINAL PRIVACY VERIFICATION


,dataset_id,epsilon_ok,delta_ok,noise_ok,clipping_ok,poisson_sampling_ok,accountant_ok,pac_ok,train_only_ok,provenance_excluded,identifiers_excluded,actual_epsilon,target_epsilon,target_delta,noise_multiplier,status
0,adult_income,True,True,True,True,True,True,True,True,True,True,4.999028,5.0,0.00001,1.214905,PASS
1,bank_marketing,True,True,True,True,True,True,True,True,True,True,4.999538,5.0,0.00001,1.249695,PASS
2,diabetes_130us,True,True,True,True,True,True,True,True,True,True,4.999684,5.0,0.00001,0.953217,PASS



✓ FINAL PRIVACY VERIFICATION — PASS

✓ SECTION 22 — PASS


In [29]:
# ==================================================================================================
# 23. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("23. COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Resolve Run Status
# --------------------------------------------------------------------------------------------------

secure_mode = bool(
    DP_CONFIG.get("secure_mode", False)
)

training_complete = True

for dataset_id in DATASET_IDS:

    if dataset_id not in DPCTGAN_TRAINING_OBJECTS:
        training_complete = False
        break

    training_object = DPCTGAN_TRAINING_OBJECTS[
        dataset_id
    ]

    completed_epochs = int(
        training_object.get(
            "completed_epochs",
            DP_CTGAN_CONFIG["epochs"]
        )
    )

    target_epochs = int(
        training_object.get(
            "target_epochs",
            DP_CTGAN_CONFIG["epochs"]
        )
    )

    if completed_epochs < target_epochs:
        training_complete = False
        break


RUN_STATUS = (
    "FINAL_CANDIDATE"
    if secure_mode and training_complete
    else "DIAGNOSTIC_ONLY"
)

PRIVACY_BOUNDARY = (
    "DP-SGD discriminator training"
)

END_TO_END_DP = False

# --------------------------------------------------------------------------------------------------
# 2. Final Integrity Gates
# --------------------------------------------------------------------------------------------------

assert len(DATASET_IDS) == 3

assert len(TRAINING_RESULTS_DF) == 3
assert len(GENERATION_RESULTS_DF) == 3
assert len(SYNTHETIC_VALIDATION_DF) == 3
assert len(PRIVACY_VERIFICATION_DF) == 3
assert len(MODEL_REGISTRY_DF) == 3
assert len(CHECKPOINT_REGISTRY_DF) == 3
assert len(SYNTHETIC_REGISTRY_DF) == 3
assert len(PRIVACY_TRACKING_DF) == 3

assert (
    TRAINING_RESULTS_DF["status"] == "PASS"
).all()

assert (
    GENERATION_RESULTS_DF["status"] == "PASS"
).all()

assert (
    SYNTHETIC_VALIDATION_DF["status"] == "PASS"
).all()

assert (
    PRIVACY_VERIFICATION_DF["status"] == "PASS"
).all()

# --------------------------------------------------------------------------------------------------
# 3. Final Epsilon Gate
# --------------------------------------------------------------------------------------------------

for _, row in ACTUAL_PRIVACY_DF.iterrows():

    actual_epsilon = float(
        row["actual_epsilon"]
    )

    target_epsilon = float(
        row["target_epsilon"]
    )

    assert (
        actual_epsilon
        <= target_epsilon + 0.05
    )

# --------------------------------------------------------------------------------------------------
# 4. Verify Secure-Mode Consistency Across Registries
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    model_row = MODEL_REGISTRY_DF[
        MODEL_REGISTRY_DF["dataset_id"]
        == dataset_id
    ]

    checkpoint_row = CHECKPOINT_REGISTRY_DF[
        CHECKPOINT_REGISTRY_DF["dataset_id"]
        == dataset_id
    ]

    privacy_row = PRIVACY_TRACKING_DF[
        PRIVACY_TRACKING_DF["dataset_id"]
        == dataset_id
    ]

    assert len(model_row) == 1
    assert len(checkpoint_row) == 1
    assert len(privacy_row) == 1

    if "secure_mode" in model_row.columns:
        assert bool(
            model_row.iloc[0]["secure_mode"]
        ) == secure_mode

    if "secure_mode" in checkpoint_row.columns:
        assert bool(
            checkpoint_row.iloc[0]["secure_mode"]
        ) == secure_mode

    if "secure_mode" in privacy_row.columns:
        assert bool(
            privacy_row.iloc[0]["secure_mode"]
        ) == secure_mode

# --------------------------------------------------------------------------------------------------
# 5. Completion Report
# --------------------------------------------------------------------------------------------------

completion_report = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,

    "project_root": str(
        PROJECT_ROOT
    ),

    "notebook_root": str(
        NB07_ROOT
    ),

    # ----------------------------------------------------------------------------------------------
    # Run classification
    # ----------------------------------------------------------------------------------------------

    "status": "PASS",

    "run_status": RUN_STATUS,

    "secure_mode": secure_mode,

    "training_complete": training_complete,

    # ----------------------------------------------------------------------------------------------
    # Dataset registry
    # ----------------------------------------------------------------------------------------------

    "datasets": DATASET_IDS,

    "dataset_summary": [
        {
            "dataset_id": dataset_id,

            "training_rows": len(
                TRAINING_DATA[dataset_id]
            ),

            "training_columns": len(
                TRAINING_DATA[dataset_id].columns
            ),

            "target": TARGET_COLUMNS[
                dataset_id
            ],

            "seed": DATASET_SEEDS[
                dataset_id
            ],
        }
        for dataset_id in DATASET_IDS
    ],

    # ----------------------------------------------------------------------------------------------
    # Model configuration
    # ----------------------------------------------------------------------------------------------

    "model": {
        "name": "DP-CTGAN",

        "architecture": "CTGAN",

        "embedding_dim": DP_CTGAN_CONFIG[
            "embedding_dim"
        ],

        "generator_dim": list(
            DP_CTGAN_CONFIG[
                "generator_dim"
            ]
        ),

        "discriminator_dim": list(
            DP_CTGAN_CONFIG[
                "discriminator_dim"
            ]
        ),

        "epochs": DP_CTGAN_CONFIG[
            "epochs"
        ],

        "batch_size": DP_CTGAN_CONFIG[
            "batch_size"
        ],

        "pac": DP_CTGAN_CONFIG[
            "pac"
        ],
    },

    # ----------------------------------------------------------------------------------------------
    # Privacy configuration
    # ----------------------------------------------------------------------------------------------

    "privacy": {
        "algorithm": "DP-SGD",

        "accountant": DP_CONFIG[
            "privacy_accountant"
        ],

        "target_epsilon": DP_CONFIG[
            "target_epsilon"
        ],

        "max_grad_norm": DP_CONFIG[
            "max_grad_norm"
        ],

        "clipping": DP_CONFIG[
            "clipping"
        ],

        "poisson_sampling": True,

        "secure_mode": secure_mode,

        "privacy_boundary": PRIVACY_BOUNDARY,

        "end_to_end_dp": END_TO_END_DP,

        "datasets_verified": 3,
    },

    # ----------------------------------------------------------------------------------------------
    # Research integrity
    # ----------------------------------------------------------------------------------------------

    "research_integrity": {
        "train_only": True,

        "validation_training": False,

        "test_training": False,

        "notebook_02_preprocessing_reused": True,

        "notebook_06_ctgan_baseline_frozen": True,

        "statistical_guidance": False,

        "spp_gan_components": False,

        "provenance_excluded": True,

        "identifiers_excluded": True,

        "target_retained": True,

        "synthetic_rows_equal_training_rows": True,

        "record_level_dp_pac": 1,

        "privacy_boundary": PRIVACY_BOUNDARY,

        "end_to_end_dp": END_TO_END_DP,
    },

    # ----------------------------------------------------------------------------------------------
    # Artifact counts
    # ----------------------------------------------------------------------------------------------

    "artifact_counts": {
        "models": len(
            MODEL_REGISTRY_DF
        ),

        "checkpoints": len(
            CHECKPOINT_REGISTRY_DF
        ),

        "synthetic_datasets": len(
            SYNTHETIC_REGISTRY_DF
        ),

        "privacy_metadata": 3,

        "privacy_verification": len(
            PRIVACY_VERIFICATION_DF
        ),
    },

    # ----------------------------------------------------------------------------------------------
    # Artifact locations
    # ----------------------------------------------------------------------------------------------

    "artifact_locations": {
        "models": str(
            NB07_MODEL_ROOT
        ),

        "checkpoints": str(
            NB07_CHECKPOINT_ROOT
        ),

        "synthetic": str(
            NB07_SYNTHETIC_ROOT
        ),

        "privacy": str(
            NB07_PRIVACY_ROOT
        ),

        "manifest": str(
            NB07_MANIFEST_ROOT
        ),

        "validation": str(
            NB07_VALIDATION_ROOT
        ),
    },

    # ----------------------------------------------------------------------------------------------
    # Reproducibility / dependencies
    # ----------------------------------------------------------------------------------------------

    "dependencies": {
        "python": sys.version,

        "pytorch": torch.__version__,

        "sdv": sdv.__version__,

        "opacus": opacus.__version__,

        "cuda_available": bool(
            torch.cuda.is_available()
        ),

        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else "CPU"
        ),
    },

    # ----------------------------------------------------------------------------------------------
    # Completion timestamp
    # ----------------------------------------------------------------------------------------------

    "completion_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

# --------------------------------------------------------------------------------------------------
# 6. Persist Completion Report
# --------------------------------------------------------------------------------------------------

completion_path = (
    NB07_VALIDATION_ROOT
    / "dp_ctgan_completion_report.json"
)

with open(
    completion_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        completion_report,
        f,
        indent=2,
        default=str,
    )

assert completion_path.exists()

# --------------------------------------------------------------------------------------------------
# 7. Reload and Validate Completion Report
# --------------------------------------------------------------------------------------------------

with open(
    completion_path,
    "r",
    encoding="utf-8",
) as f:

    reloaded_completion = json.load(f)

assert (
    reloaded_completion["status"]
    == "PASS"
)

assert (
    reloaded_completion["run_status"]
    == RUN_STATUS
)

assert (
    bool(
        reloaded_completion["secure_mode"]
    )
    == secure_mode
)

assert (
    reloaded_completion[
        "privacy"
    ][
        "privacy_boundary"
    ]
    == PRIVACY_BOUNDARY
)

assert (
    reloaded_completion[
        "privacy"
    ][
        "end_to_end_dp"
    ]
    is False
)

assert (
    reloaded_completion[
        "research_integrity"
    ][
        "end_to_end_dp"
    ]
    is False
)

# --------------------------------------------------------------------------------------------------
# 8. Final Display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 07 — DP-CTGAN BASELINE")
print("=" * 100)

for dataset_id in DATASET_IDS:

    privacy_row = (
        ACTUAL_PRIVACY_DF[
            ACTUAL_PRIVACY_DF["dataset_id"]
            == dataset_id
        ]
        .iloc[0]
    )

    print(
        f"{dataset_id:<18} | "
        f"Rows: {len(TRAINING_DATA[dataset_id]):>8,} | "
        f"ε={float(privacy_row['actual_epsilon']):.5f} | "
        f"δ={float(privacy_row['target_delta']):.8f} | "
        f"σ={float(privacy_row['noise_multiplier']):.5f} | "
        f"PASS"
    )

# --------------------------------------------------------------------------------------------------
# 9. Final Integrity Status
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("FINAL INTEGRITY STATUS")
print("-" * 100)

print("✓ Training datasets                 : 3/3")
print("✓ DP-CTGAN models                   : 3/3")
print("✓ Checkpoints                       : 3/3")
print("✓ Synthetic datasets                : 3/3")
print("✓ Synthetic validation              : 3/3")
print("✓ Privacy verification              : 3/3")
print("✓ Privacy budgets verified          : 3/3")
print("✓ Train-only policy                 : PASS")
print("✓ Provenance exclusion              : PASS")
print("✓ Identifier exclusion             : PASS")
print("✓ Poisson sampling                 : PASS")
print("✓ Gradient clipping                : PASS")
print("✓ Gaussian noise                   : PASS")
print("✓ RDP privacy accounting           : PASS")
print("✓ Actual ε recorded                : PASS")
print("✓ Metadata persisted               : PASS")
print("✓ Manifest persisted               : PASS")
print("✓ Completion report persisted      : PASS")

print("\n" + "-" * 100)
print("PRIVACY BOUNDARY")
print("-" * 100)

print(
    f"✓ Privacy mechanism                : DP-SGD"
)

print(
    f"✓ Privacy boundary                 : {PRIVACY_BOUNDARY}"
)

print(
    f"✓ End-to-end DP claimed            : {END_TO_END_DP}"
)

print(
    f"✓ Secure RNG mode                  : {secure_mode}"
)

print(
    f"✓ Training complete                : {training_complete}"
)

print(
    f"✓ Run status                       : {RUN_STATUS}"
)

# --------------------------------------------------------------------------------------------------
# 10. Final Notebook Status
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)

if RUN_STATUS == "FINAL_CANDIDATE":

    print(
        "✓ NOTEBOOK 07 — DP-CTGAN BASELINE : "
        "PASS / FINAL CANDIDATE"
    )

else:

    print(
        "✓ NOTEBOOK 07 — DP-CTGAN BASELINE : "
        "PASS / DIAGNOSTIC ONLY"
    )

print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 11. RAM / CUDA Cleanup
# --------------------------------------------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "\n✓ RAM / CUDA cache cleanup completed."
)

# ==================================================================================================
# END OF SECTION 23
# ==================================================================================================

23. COMPLETION SUMMARY

NOTEBOOK 07 — DP-CTGAN BASELINE
adult_income       | Rows:   34,189 | ε=4.99903 | δ=0.00001000 | σ=1.21490 | PASS
bank_marketing     | Rows:   31,647 | ε=4.99954 | δ=0.00001000 | σ=1.24969 | PASS
diabetes_130us     | Rows:   71,236 | ε=4.99968 | δ=0.00001000 | σ=0.95322 | PASS

----------------------------------------------------------------------------------------------------
FINAL INTEGRITY STATUS
----------------------------------------------------------------------------------------------------
✓ Training datasets                 : 3/3
✓ DP-CTGAN models                   : 3/3
✓ Checkpoints                       : 3/3
✓ Synthetic datasets                : 3/3
✓ Synthetic validation              : 3/3
✓ Privacy verification              : 3/3
✓ Privacy budgets verified          : 3/3
✓ Train-only policy                 : PASS
✓ Provenance exclusion              : PASS
✓ Identifier exclusion             : PASS
✓ Poisson sampling                 : PASS
✓ Gradie